# Run Two-Pass Inference and Evaluate the Paper Framework

This notebook is the final stage of the project. It loads the two frozen model
checkpoints and runs one canonical method:

```text
Graph-evidence prediction
        ↓
Pass I: physical graph hypothesis construction
        ↓
Pass II: graph-context structural finalization
        ↓
Agreement-gated ResNet34 node-region refinement
        ↓
Final graph JSON and paper metrics
```

There are no alternative execution modes or historical comparison branches in
this notebook.

## Before running

Complete the preceding stages or use the supplied fail-safe artifacts:

1. Reconstruct the converted dataset with notebooks `00` and `01`.
2. Use the supplied graph-evidence checkpoint or train it with notebook `03`.
3. Use the supplied plain ResNet34 node checkpoint or train it with notebook `02`.
4. Fill all four paths in the next cell.

## Main outputs

The notebook writes:

- one final prediction JSON per test image;
- per-image and aggregate metric files;
- family-wise results;
- the proposed-framework row for Table 4;
- the family rows for Table 5;
- selected prediction and ground-truth overlays;
- checkpoint SHA-256 hashes and a reproduction manifest.


In [ ]:
import random
import shutil
import time
import traceback
from copy import deepcopy

# ============================================================
# USER SETTINGS — FILL ALL FOUR PATHS BEFORE RUNNING
# ============================================================

# Reconstructed dataset created by notebooks 00 and 01.
# Expected family/split structure: <root>/<family>/<split>/{images,json}/
DATASET_ROOT = ""

# Checkpoint produced by 03_Train_Graph_Evidence_Model.ipynb
GRAPH_MODEL_CHECKPOINT = ""

# Plain ResNet34 U-Net checkpoint produced by
# 02_Train_ResNet34_Node_Segmentation.ipynb
NODE_MODEL_CHECKPOINT = ""

# All predictions, metrics, summaries, and selected visual checks are written here.
OUTPUT_ROOT = ""

# Held-out evaluation corpus
DATASET_FAMILIES = ("fa", "fca", "fcb")
INFERENCE_SPLITS = ("test",)

# Graph-evidence model settings
TARGET_W = 512
TARGET_H = 512
IMAGE_MODE = "grayscale"
NORMALIZE_IMAGE = True
APPLY_OTSU_THRESHOLD = True
GRAPH_BASE_CHANNELS = 16
NORM_TYPE = "auto"

# Plain ResNet34 node-region inference settings.
# The published run evaluates the fully convolutional model at 320 × 320.
NODE_SEG_INPUT_SIZE = 320
NODE_SEG_THRESHOLD = 0.46
NODE_SEG_MIN_AREA = 35
NODE_SEG_CLOSE_KERNEL = 3

# Topology-preserving node refinement
ENABLE_NODE_REFINEMENT = True
NODE_REFINEMENT_IOU_ACCEPT = 0.10
NODE_REFINEMENT_CENTER_TOLERANCE_RATIO = 0.02
NODE_REFINEMENT_KEEP_UNMATCHED_GRAPH_NODES = True
NODE_REFINEMENT_REPAIR_EDGE_IDS = False

# Evaluation definitions
METRIC_TOLERANCE_RATIO = 0.02
NODE_IOU_ACCEPT = 0.25
PATH_BASE_TOLERANCE_PX = 3
NGED_NORMALIZER = "max_pred_gt_nodes_plus_max_pred_gt_edges"

# Output policy
SAVE_ALL_PREDICTION_JSON = True
SAVE_DEBUG_EXAMPLES = True
DEBUG_EXAMPLE_POLICY = "first"  # "first" or "random"
MAX_DEBUG_EXAMPLES = 5
SAVE_DEBUG_OVERLAYS = True
SAVE_DEBUG_ASSEMBLER_JSON = True
CLEAR_OUTPUT_BEFORE_RUN = False

# Runtime
PARALLEL_INFERENCE = True
PARALLEL_BACKEND = "threads"  # "threads" or "single"
MAX_GPU_WORKERS = 2
CPU_WORKERS_FOR_IO = 4
PRINT_TIMING_DIAGNOSTICS = True
PROGRESS_EVERY_N = 10
RANDOM_SEED = 39

# Optional quick test. The normal paper run uses all held-out test images.
QUICK_TEST = False
QUICK_TEST_N = 12

## 1. Graph-evidence model and Pass-I assembler

The following implementation matches the graph-evidence checkpoint produced by
the training notebook. The model consumes 512 × 512 grayscale diagrams and
predicts node evidence, shaft and skeleton evidence, endpoint heatmaps,
directions, progress, offsets, and endpoint tangents.

Pass I converts those dense fields into an initial physical graph hypothesis.


In [ ]:
"""
Graph-evidence model and first-pass physical graph assembler.

This notebook cell contains the model architecture and the deterministic first-pass
decoder used by the paper. It consumes every output produced by the graph-evidence
network:

    node_center, node_extent, shaft, skeleton, head, tail, dir,
    progress, head_offset, tail_offset, head_tangent, tail_tangent

Main idea
---------
1. Detect nodes from node_extent + node_center.
2. Detect coarse head/tail endpoints from heatmaps.
3. Refine endpoints using the learned offset heads. The offsets are interpreted
   exactly as trained in the notebook: normalized displacement from local pixel
   to exact endpoint, multiplied by endpoint_radius.
4. Attach endpoints to nodes using nearest bbox plus the endpoint tangent.
5. Trace tail -> head paths using a deterministic A* over a fused support field.
   The path cost uses shaft, skeleton, direction, and progress monotonicity.
6. Select non-conflicting arrows.
7. Repair crowded ports by creating virtual opposite-role endpoints when shaft evidence remains.
8. Export graph JSON.

Coordinates
-----------
By default, writer JSON is in model space, e.g. 512x512. The script also saves an
optional original-space writer JSON by scaling coordinates back to the source image.
"""

from __future__ import annotations

import argparse
import csv
import heapq
import json
import math
import os
import sys
import traceback
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import networkx as nx
except Exception:  # networkx is useful but not mandatory for JSON export
    nx = None


# Notebook paths are supplied only in the USER SETTINGS cell.
INPUT_DIR = ""
OUTPUT_DIR = ""
CHECKPOINT_PATH = ""

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
NEIGHBORS_8 = [
    (-1, -1), (0, -1), (1, -1),
    (-1,  0),          (1,  0),
    (-1,  1), (0,  1), (1,  1),
]


# -----------------------------------------------------------------------------
# Model definition: matches the current notebook model, including GroupNorm,
# dropout, and all upgraded heads.
# -----------------------------------------------------------------------------

def make_group_norm(num_channels: int, max_groups: int = 8) -> nn.GroupNorm:
    for g in [max_groups, 4, 2, 1]:
        if num_channels % g == 0:
            return nn.GroupNorm(num_groups=g, num_channels=num_channels)
    return nn.GroupNorm(num_groups=1, num_channels=num_channels)


def make_norm_layer(num_channels: int, norm_type: str) -> nn.Module:
    norm_type = norm_type.lower()
    if norm_type == "group":
        return make_group_norm(num_channels)
    if norm_type == "batch":
        return nn.BatchNorm2d(num_channels)
    raise ValueError(f"Unsupported norm_type={norm_type!r}; use 'group' or 'batch'.")


class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, norm_type: str = "group"):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            make_norm_layer(out_ch, norm_type),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            make_norm_layer(out_ch, norm_type),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, norm_type: str = "group"):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch, norm_type=norm_type),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Up(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, norm_type: str = "group"):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv((in_ch // 2) + skip_ch, out_ch, norm_type=norm_type)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        if diff_x != 0 or diff_y != 0:
            x = F.pad(
                x,
                [diff_x // 2, diff_x - diff_x // 2,
                 diff_y // 2, diff_y - diff_y // 2],
            )
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class DiagramUNet(nn.Module):
    def __init__(self, in_channels: int = 1, base_ch: int = 32, norm_type: str = "group"):
        super().__init__()
        self.inc = DoubleConv(in_channels, base_ch, norm_type=norm_type)
        self.down1 = Down(base_ch, base_ch * 2, norm_type=norm_type)
        self.down2 = Down(base_ch * 2, base_ch * 4, norm_type=norm_type)
        self.down3 = Down(base_ch * 4, base_ch * 8, norm_type=norm_type)
        self.down4 = Down(base_ch * 8, base_ch * 16, norm_type=norm_type)

        self.bottleneck_dropout = nn.Dropout2d(0.25)

        self.up1 = Up(base_ch * 16, base_ch * 8, base_ch * 8, norm_type=norm_type)
        self.up2 = Up(base_ch * 8, base_ch * 4, base_ch * 4, norm_type=norm_type)
        self.up3 = Up(base_ch * 4, base_ch * 2, base_ch * 2, norm_type=norm_type)
        self.up4 = Up(base_ch * 2, base_ch, base_ch, norm_type=norm_type)

        self.node_center_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.node_extent_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.shaft_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.skeleton_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.head_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.tail_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.dir_head = nn.Conv2d(base_ch, 2, kernel_size=1)

        self.progress_head = nn.Conv2d(base_ch, 1, kernel_size=1)
        self.head_offset_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        self.tail_offset_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        self.head_tangent_head = nn.Conv2d(base_ch, 2, kernel_size=1)
        self.tail_tangent_head = nn.Conv2d(base_ch, 2, kernel_size=1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x5 = self.bottleneck_dropout(x5)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        return {
            "node_center": self.node_center_head(x),
            "node_extent": self.node_extent_head(x),
            "shaft": self.shaft_head(x),
            "skeleton": self.skeleton_head(x),
            "head": self.head_head(x),
            "tail": self.tail_head(x),
            "dir": self.dir_head(x),
            "progress": self.progress_head(x),
            "head_offset": self.head_offset_head(x),
            "tail_offset": self.tail_offset_head(x),
            "head_tangent": self.head_tangent_head(x),
            "tail_tangent": self.tail_tangent_head(x),
        }


# -----------------------------------------------------------------------------
# Data containers
# -----------------------------------------------------------------------------

@dataclass
class Node:
    node_id: str
    bbox_xyxy: Tuple[int, int, int, int]
    bbox_xywh: Tuple[int, int, int, int]
    center_xy: Tuple[float, float]
    score: float
    area: int


@dataclass
class Endpoint:
    endpoint_id: str
    kind: str  # "head" or "tail"
    coarse_xy: Tuple[float, float]
    refined_xy: Tuple[float, float]
    snapped_xy: Tuple[int, int]
    score: float
    progress_value: float
    tangent_xy: Tuple[float, float]
    offset_xy: Tuple[float, float]
    attached_node_id: Optional[str] = None
    attach_score: Optional[float] = None
    component_id: Optional[int] = None


@dataclass
class Arrow:
    arrow_id: int
    tail_endpoint_id: str
    head_endpoint_id: str
    tail_xy: Tuple[float, float]
    head_xy: Tuple[float, float]
    tail_hint: Dict[str, float]
    head_hint: Dict[str, float]
    path_xy: List[Tuple[int, int]]
    tail_node_id: Optional[str]
    head_node_id: Optional[str]
    path_cost: float
    confidence: float
    support_score: float
    shaft_score: float
    skeleton_score: float
    direction_score: float
    progress_score: float
    endpoint_score: float
    tangent_score: float
    self_loop: bool
    meta: Dict[str, Any] = field(default_factory=dict)


@dataclass
class DecodeResult:
    nodes: List[Node]
    heads: List[Endpoint]
    tails: List[Endpoint]
    candidate_arrows: List[Arrow]
    arrows: List[Arrow]
    writer_style_json: Dict[str, Any]
    writer_style_original_json: Optional[Dict[str, Any]]
    decoder_debug_json: Dict[str, Any]
    debug_maps: Dict[str, np.ndarray]
    nx_graph: Optional[Any] = None


@dataclass
class LADDecoderConfig:
    # Node extraction
    node_extent_thresh: float = 0.20
    node_center_thresh: float = 0.25
    min_node_area: int = 35
    node_merge_iou: float = 0.45
    node_close_kernel: int = 3

    # Endpoint extraction
    head_thresh: float = 0.25
    tail_thresh: float = 0.25
    min_endpoint_area: int = 2
    endpoint_nms_dist: float = 5.0
    endpoint_radius: float = 8.0  # must match training offset normalization radius
    endpoint_patch_radius: int = 2

    # Support construction
    shaft_thresh: float = 0.20
    skeleton_thresh: float = 0.20
    progress_support_thresh: float = 0.02
    support_thresh: float = 0.20
    support_close_kernel: int = 3
    support_dilate_kernel: int = 1
    node_core_shrink_px: int = 5
    endpoint_force_radius: int = 5
    use_image_ink_support: bool = False
    ink_weight: float = 0.20
    shaft_weight: float = 0.45
    skeleton_weight: float = 0.30
    progress_weight: float = 0.15
    endpoint_weight: float = 0.10

    # Attachment
    node_attach_max_dist: float = 28.0
    tangent_probe_dist: float = 14.0

    # A* tracing
    max_pair_dist: float = 1e9
    max_expansions: int = 90000
    goal_radius: int = 4
    nearest_walkable_radius: int = 12
    min_path_len: int = 4
    support_cost_weight: float = 3.00
    direction_cost_weight: float = 1.00
    progress_cost_weight: float = 0.85
    curvature_cost_weight: float = 0.25
    node_core_cost_weight: float = 2.50
    step_cost_weight: float = 0.15
    heuristic_weight: float = 0.08
    progress_margin: float = 0.015

    # Candidate acceptance / selection
    min_arrow_confidence: float = 0.32

    # Endpoint reuse / branch handling
    # allow_endpoint_reuse=True is the broad switch: all endpoint reuse is allowed
    # after duplicate filtering. Useful for debugging, but it can overgenerate.
    allow_endpoint_reuse: bool = False

    # Safer default: allow reuse only when it looks like a real fan-out/fan-in branch.
    # This is needed because crowded node boundaries can collapse two real arrows into
    # one heatmap endpoint. The decoder should not throw away the second arrow merely
    # because the tail/head seed is shared.
    branch_aware_endpoint_reuse: bool = True
    branch_min_opposite_endpoint_dist: float = 16.0
    branch_max_path_iou: float = 0.62
    branch_require_different_node_pair: bool = True

    duplicate_endpoint_dist: float = 8.0
    duplicate_path_iou: float = 0.82
    keep_unattached_arrows: bool = True

    # Port-role repair
    # Some DFA nodes reuse the same tiny boundary port as the head of one edge and
    # the tail of another. In those cases the heatmap may keep only one role. The
    # normal decoder then has shaft evidence but no legal tail/head seed. This
    # repair pass creates low-priority virtual opposite-role endpoints at already
    # accepted node ports and tries to connect them to otherwise unused endpoints.
    enable_port_role_repair: bool = True
    port_repair_score_scale: float = 0.62
    port_repair_confidence_scale: float = 0.82
    port_repair_min_confidence: float = 0.46
    port_repair_progress_cost_weight: float = 0.20
    port_repair_max_path_iou: float = 0.52
    port_repair_min_support_score: float = 0.72
    port_repair_min_shaft_score: float = 0.72
    port_repair_max_candidates: int = 12

    # Adaptive decoding
    # These values are not meant to be hand-tuned per image. When auto_adapt=True,
    # the decoder derives thresholds/distances from the current prediction maps and
    # node geometry. A few guard rails remain because fully unconstrained decoding
    # is just hallucination wearing a lab coat.
    auto_adapt: bool = True
    adaptive_min_prob_floor: float = 0.05
    adaptive_endpoint_quantile: float = 99.75
    adaptive_node_quantile: float = 95.0
    adaptive_support_quantile: float = 88.0

    # Stroke-graph candidate generation
    # This is the core V5 change: generate extra edge candidates from continuous
    # stroke topology/support, not only from predicted tail/head endpoint pairs.
    enable_stroke_graph_candidates: bool = True
    stroke_graph_min_component_area: int = 18
    stroke_graph_min_path_len: int = 12
    stroke_graph_node_contact_factor: float = 0.42
    stroke_graph_node_contact_min: float = 10.0
    stroke_graph_node_contact_max: float = 42.0
    stroke_graph_min_confidence: float = 0.34
    stroke_graph_max_path_iou: float = 0.78
    stroke_graph_max_components: int = 96
    stroke_graph_max_candidates: int = 80
    # In V6 this is False by default. DFA self-loops are not exotic; they are normal edges.
    # We still validate them through path length/excursion instead of blindly accepting tiny node-border strokes.
    stroke_graph_skip_self_loops: bool = False
    stroke_graph_endpoint_score: float = 0.82
    stroke_graph_confidence_scale: float = 0.92

    # V6 ink-bridge candidate generation
    # Endpoint/stroke candidates can fail when the learned shaft support is broken, even
    # though raw ink visibly connects a tail/head pair. This fallback builds a narrow
    # endpoint-to-endpoint corridor and allows A* to use raw ink only inside that corridor.
    enable_ink_bridge_candidates: bool = True
    ink_bridge_min_confidence: float = 0.40
    ink_bridge_confidence_scale: float = 0.88
    ink_bridge_min_support_score: float = 0.42
    ink_bridge_min_width: int = 10
    ink_bridge_max_width: int = 32
    ink_bridge_width_factor: float = 0.10
    ink_bridge_threshold: float = 0.18
    ink_bridge_max_candidates: int = 80
    ink_bridge_progress_cost_weight: float = 0.28
    ink_bridge_support_cost_weight: float = 1.35
    ink_bridge_allow_self_loops: bool = True

    # Runtime
    verbose: bool = False


# -----------------------------------------------------------------------------
# Numeric / image utilities
# -----------------------------------------------------------------------------

def normalize01(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    mn = float(np.nanmin(x))
    mx = float(np.nanmax(x))
    if mx - mn < eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return (1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))).astype(np.float32)


def to_uint8_vis(x: np.ndarray) -> np.ndarray:
    return np.clip(normalize01(x) * 255.0, 0, 255).astype(np.uint8)


def ensure_rgb_uint8(img: np.ndarray) -> np.ndarray:
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    if img.dtype != np.uint8:
        if float(np.nanmax(img)) <= 1.0:
            img = (img * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img = img.clip(0, 255).astype(np.uint8)
    return img


def safe_json(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): safe_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [safe_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if hasattr(obj, "__dataclass_fields__"):
        return safe_json(asdict(obj))
    return obj


def inside(x: int, y: int, w: int, h: int) -> bool:
    return 0 <= x < w and 0 <= y < h


def l2(p: Sequence[float], q: Sequence[float]) -> float:
    return float(math.hypot(float(p[0]) - float(q[0]), float(p[1]) - float(q[1])))


def unit(vx: float, vy: float, eps: float = 1e-8) -> Tuple[float, float, float]:
    mag = math.hypot(float(vx), float(vy))
    if mag < eps:
        return 0.0, 0.0, 0.0
    return float(vx) / mag, float(vy) / mag, mag


def cosine2(ax: float, ay: float, bx: float, by: float, eps: float = 1e-8) -> float:
    an = math.hypot(ax, ay)
    bn = math.hypot(bx, by)
    if an < eps or bn < eps:
        return 0.0
    return float((ax * bx + ay * by) / (an * bn))


def sample_map_bilinear(arr: np.ndarray, xy: Sequence[float]) -> float:
    h, w = arr.shape[:2]
    x = float(np.clip(xy[0], 0, w - 1))
    y = float(np.clip(xy[1], 0, h - 1))
    x0 = int(math.floor(x)); y0 = int(math.floor(y))
    x1 = min(w - 1, x0 + 1); y1 = min(h - 1, y0 + 1)
    wx = x - x0; wy = y - y0
    v00 = float(arr[y0, x0]); v10 = float(arr[y0, x1])
    v01 = float(arr[y1, x0]); v11 = float(arr[y1, x1])
    return (1 - wx) * (1 - wy) * v00 + wx * (1 - wy) * v10 + (1 - wx) * wy * v01 + wx * wy * v11


def sample_pair_bilinear(xmap: np.ndarray, ymap: np.ndarray, xy: Sequence[float]) -> Tuple[float, float]:
    return sample_map_bilinear(xmap, xy), sample_map_bilinear(ymap, xy)


def step_length(dx: int, dy: int) -> float:
    return math.sqrt(2.0) if dx != 0 and dy != 0 else 1.0


def bbox_iou(a: Sequence[int], b: Sequence[int]) -> float:
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1 + 1), max(0.0, iy2 - iy1 + 1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1 + 1) * max(0.0, ay2 - ay1 + 1)
    area_b = max(0.0, bx2 - bx1 + 1) * max(0.0, by2 - by1 + 1)
    return float(inter / max(area_a + area_b - inter, 1e-6))


def point_to_bbox_distance(pt: Sequence[float], box_xyxy: Sequence[int]) -> float:
    x, y = float(pt[0]), float(pt[1])
    x1, y1, x2, y2 = map(float, box_xyxy)
    dx = max(x1 - x, 0.0, x - x2)
    dy = max(y1 - y, 0.0, y - y2)
    return float(math.hypot(dx, dy))


def point_inside_bbox(pt: Sequence[float], box_xyxy: Sequence[int]) -> bool:
    x, y = float(pt[0]), float(pt[1])
    x1, y1, x2, y2 = map(float, box_xyxy)
    return x1 <= x <= x2 and y1 <= y <= y2


def draw_disk(mask: np.ndarray, xy: Sequence[float], radius: int, value: int = 1) -> None:
    h, w = mask.shape[:2]
    x = int(round(float(xy[0]))); y = int(round(float(xy[1])))
    if inside(x, y, w, h):
        cv2.circle(mask, (x, y), int(radius), int(value), -1)


def image_ink_support(decoder_img: np.ndarray) -> np.ndarray:
    """Auto-detect whether foreground ink is dark-on-light or light-on-dark."""
    rgb = ensure_rgb_uint8(decoder_img)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    # If mean is bright, assume black ink on white background. If dark, assume white ink on black.
    if float(gray.mean()) > 127.0:
        ink = 1.0 - gray.astype(np.float32) / 255.0
    else:
        ink = gray.astype(np.float32) / 255.0
    return normalize01(ink)


# -----------------------------------------------------------------------------
# Checkpoint loading and model output conversion
# -----------------------------------------------------------------------------

def extract_state_dict(ckpt: Any) -> Dict[str, torch.Tensor]:
    if isinstance(ckpt, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                return ckpt[key]
        return ckpt
    raise ValueError("Unsupported checkpoint object; expected a dict or a dict containing model_state_dict.")


def strip_module_prefix(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}


def infer_arch_from_state(state: Dict[str, torch.Tensor]) -> Dict[str, Any]:
    clean = strip_module_prefix(state)
    base_ch = None
    in_channels = None
    norm_type = "group"

    for k, v in clean.items():
        if k.endswith("inc.block.0.weight") and hasattr(v, "shape"):
            base_ch = int(v.shape[0])
            in_channels = int(v.shape[1])
            break

    # BatchNorm checkpoints have running stats. GroupNorm checkpoints do not.
    if any("running_mean" in k or "running_var" in k for k in clean.keys()):
        norm_type = "batch"

    return {"base_ch": base_ch, "in_channels": in_channels, "norm_type": norm_type}


def load_checkpoint_model(
    checkpoint_path: Path,
    device: torch.device,
    image_mode: str,
    base_ch: int = 0,
    norm_type: str = "auto",
    strict: bool = True,
) -> Tuple[DiagramUNet, Dict[str, Any]]:
    ckpt = torch.load(str(checkpoint_path), map_location=device)
    state = strip_module_prefix(extract_state_dict(ckpt))
    inferred = infer_arch_from_state(state)

    ckpt_base_ch = inferred.get("base_ch")
    ckpt_in_channels = inferred.get("in_channels")
    ckpt_norm_type = inferred.get("norm_type") or "group"

    if base_ch <= 0:
        if ckpt_base_ch is None:
            raise ValueError("Could not infer base_ch from checkpoint. Pass --base-ch explicitly.")
        base_ch = ckpt_base_ch

    requested_in_channels = 1 if image_mode.lower() == "grayscale" else 3
    in_channels = ckpt_in_channels if ckpt_in_channels is not None else requested_in_channels
    if in_channels != requested_in_channels:
        raise ValueError(
            f"Checkpoint expects in_channels={in_channels}, but --image-mode={image_mode!r} gives "
            f"{requested_in_channels}. Use the same image mode used during training."
        )

    if norm_type == "auto":
        norm_type = ckpt_norm_type

    model = DiagramUNet(in_channels=in_channels, base_ch=base_ch, norm_type=norm_type).to(device)
    load_info: Dict[str, Any] = {
        "checkpoint_path": str(checkpoint_path),
        "base_ch": base_ch,
        "in_channels": in_channels,
        "norm_type": norm_type,
        "strict": strict,
        "inferred": inferred,
        "checkpoint_top_level_keys": list(ckpt.keys()) if isinstance(ckpt, dict) else [],
    }

    if strict:
        model.load_state_dict(state, strict=True)
        load_info["missing_keys"] = []
        load_info["unexpected_keys"] = []
    else:
        missing, unexpected = model.load_state_dict(state, strict=False)
        load_info["missing_keys"] = list(missing)
        load_info["unexpected_keys"] = list(unexpected)

    model.eval()
    return model, load_info


@torch.no_grad()
def outputs_to_pred_maps(outputs: Dict[str, torch.Tensor]) -> Dict[str, np.ndarray]:
    # Semantic / probability heads
    pred: Dict[str, np.ndarray] = {
        "node_center": torch.sigmoid(outputs["node_center"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "node_extent": torch.sigmoid(outputs["node_extent"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "shaft": torch.sigmoid(outputs["shaft"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "skeleton": torch.sigmoid(outputs["skeleton"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "head": torch.sigmoid(outputs["head"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "tail": torch.sigmoid(outputs["tail"])[0, 0].detach().cpu().numpy().astype(np.float32),
        "progress": torch.sigmoid(outputs["progress"])[0, 0].detach().cpu().numpy().astype(np.float32),
    }

    # Raw regression/vector heads
    pred["dir_x"] = outputs["dir"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["dir_y"] = outputs["dir"][0, 1].detach().cpu().numpy().astype(np.float32)

    pred["head_offset_x"] = outputs["head_offset"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["head_offset_y"] = outputs["head_offset"][0, 1].detach().cpu().numpy().astype(np.float32)
    pred["tail_offset_x"] = outputs["tail_offset"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["tail_offset_y"] = outputs["tail_offset"][0, 1].detach().cpu().numpy().astype(np.float32)

    pred["head_tangent_x"] = outputs["head_tangent"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["head_tangent_y"] = outputs["head_tangent"][0, 1].detach().cpu().numpy().astype(np.float32)
    pred["tail_tangent_x"] = outputs["tail_tangent"][0, 0].detach().cpu().numpy().astype(np.float32)
    pred["tail_tangent_y"] = outputs["tail_tangent"][0, 1].detach().cpu().numpy().astype(np.float32)

    pred["dir_mag"] = np.sqrt(pred["dir_x"] ** 2 + pred["dir_y"] ** 2).astype(np.float32)
    pred["head_offset_mag"] = np.sqrt(pred["head_offset_x"] ** 2 + pred["head_offset_y"] ** 2).astype(np.float32)
    pred["tail_offset_mag"] = np.sqrt(pred["tail_offset_x"] ** 2 + pred["tail_offset_y"] ** 2).astype(np.float32)
    return pred


# -----------------------------------------------------------------------------
# Decoder internals
# -----------------------------------------------------------------------------

def connected_components_from_prob(prob: np.ndarray, thresh: float, close_kernel: int = 0) -> Tuple[int, np.ndarray, np.ndarray, np.ndarray]:
    mask = (prob >= thresh).astype(np.uint8)
    if close_kernel and close_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    return cv2.connectedComponentsWithStats(mask, connectivity=8)


def weighted_centroid_from_mask(weight_map: np.ndarray, comp_mask: np.ndarray, fallback_xy: Tuple[float, float]) -> Tuple[float, float]:
    ys, xs = np.where(comp_mask > 0)
    if len(xs) == 0:
        return fallback_xy
    weights = weight_map[ys, xs].astype(np.float64)
    s = float(weights.sum())
    if s <= 1e-8:
        return float(xs.mean()), float(ys.mean())
    return float((xs * weights).sum() / s), float((ys * weights).sum() / s)


def nms_nodes(nodes: List[Node], iou_thresh: float) -> List[Node]:
    kept: List[Node] = []
    for n in sorted(nodes, key=lambda x: x.score, reverse=True):
        if any(bbox_iou(n.bbox_xyxy, k.bbox_xyxy) >= iou_thresh for k in kept):
            continue
        kept.append(n)
    kept.sort(key=lambda n: (n.center_xy[1], n.center_xy[0]))
    for i, n in enumerate(kept):
        n.node_id = f"N{i}"
    return kept


def extract_nodes(pred: Dict[str, np.ndarray], cfg: LADDecoderConfig) -> List[Node]:
    extent = pred["node_extent"]
    center = pred["node_center"]
    h, w = extent.shape

    num, labels, stats, cents = connected_components_from_prob(
        extent, cfg.node_extent_thresh, close_kernel=cfg.node_close_kernel
    )

    nodes: List[Node] = []
    for cid in range(1, num):
        x, y, bw, bh, area = stats[cid].tolist()
        if int(area) < cfg.min_node_area:
            continue
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + bw - 1), int(y + bh - 1)
        comp_mask = (labels == cid).astype(np.uint8)
        cx, cy = weighted_centroid_from_mask(center, comp_mask, (float(cents[cid][0]), float(cents[cid][1])))
        score = float(max(center[labels == cid].max(initial=0), extent[labels == cid].mean()))
        nodes.append(
            Node(
                node_id=f"N{len(nodes)}",
                bbox_xyxy=(x1, y1, x2, y2),
                bbox_xywh=(x1, y1, int(bw), int(bh)),
                center_xy=(cx, cy),
                score=score,
                area=int(area),
            )
        )

    # Fallback: if extents are weak, use center blobs as tiny boxes.
    if not nodes:
        num_c, lab_c, stats_c, cents_c = connected_components_from_prob(center, cfg.node_center_thresh, close_kernel=3)
        for cid in range(1, num_c):
            x, y, bw, bh, area = stats_c[cid].tolist()
            if int(area) < max(2, cfg.min_node_area // 8):
                continue
            cx, cy = float(cents_c[cid][0]), float(cents_c[cid][1])
            pad = 12
            x1 = max(0, int(cx - pad)); y1 = max(0, int(cy - pad))
            x2 = min(w - 1, int(cx + pad)); y2 = min(h - 1, int(cy + pad))
            nodes.append(
                Node(
                    node_id=f"N{len(nodes)}",
                    bbox_xyxy=(x1, y1, x2, y2),
                    bbox_xywh=(x1, y1, x2 - x1 + 1, y2 - y1 + 1),
                    center_xy=(cx, cy),
                    score=float(center[lab_c == cid].max(initial=0)),
                    area=int(area),
                )
            )

    return nms_nodes(nodes, cfg.node_merge_iou)


def endpoint_patch_average(
    map_x: np.ndarray,
    map_y: np.ndarray,
    heat: np.ndarray,
    xy: Sequence[float],
    patch_radius: int,
) -> Tuple[float, float]:
    h, w = heat.shape
    x0 = int(round(float(xy[0]))); y0 = int(round(float(xy[1])))
    xs0 = max(0, x0 - patch_radius); xs1 = min(w - 1, x0 + patch_radius)
    ys0 = max(0, y0 - patch_radius); ys1 = min(h - 1, y0 + patch_radius)
    patch_w = heat[ys0:ys1 + 1, xs0:xs1 + 1].astype(np.float64)
    s = float(patch_w.sum())
    if s <= 1e-8:
        return sample_pair_bilinear(map_x, map_y, xy)
    mx = float((map_x[ys0:ys1 + 1, xs0:xs1 + 1] * patch_w).sum() / s)
    my = float((map_y[ys0:ys1 + 1, xs0:xs1 + 1] * patch_w).sum() / s)
    return mx, my


def nms_endpoints(endpoints: List[Endpoint], min_dist: float) -> List[Endpoint]:
    kept: List[Endpoint] = []
    for ep in sorted(endpoints, key=lambda e: e.score, reverse=True):
        if any(l2(ep.refined_xy, k.refined_xy) <= min_dist for k in kept):
            continue
        kept.append(ep)
    kept.sort(key=lambda e: (e.refined_xy[1], e.refined_xy[0]))
    for i, ep in enumerate(kept):
        ep.endpoint_id = f"{ep.kind[0].upper()}{i}"
    return kept


def extract_endpoints(pred: Dict[str, np.ndarray], kind: str, cfg: LADDecoderConfig) -> List[Endpoint]:
    assert kind in {"head", "tail"}
    heat = pred[kind]
    thresh = cfg.head_thresh if kind == "head" else cfg.tail_thresh
    num, labels, stats, cents = connected_components_from_prob(heat, thresh, close_kernel=0)
    h, w = heat.shape

    if kind == "head":
        off_x, off_y = pred["head_offset_x"], pred["head_offset_y"]
        tan_x, tan_y = pred["head_tangent_x"], pred["head_tangent_y"]
    else:
        off_x, off_y = pred["tail_offset_x"], pred["tail_offset_y"]
        tan_x, tan_y = pred["tail_tangent_x"], pred["tail_tangent_y"]

    endpoints: List[Endpoint] = []
    for cid in range(1, num):
        x, y, bw, bh, area = stats[cid].tolist()
        if int(area) < cfg.min_endpoint_area:
            continue
        comp = labels == cid
        ys, xs = np.where(comp)
        if len(xs) == 0:
            continue

        values = heat[ys, xs]
        imax = int(np.argmax(values))
        max_x = float(xs[imax]); max_y = float(ys[imax])
        wcx, wcy = weighted_centroid_from_mask(heat, comp.astype(np.uint8), (max_x, max_y))
        # Use weighted centroid as coarse point, but keep it pulled toward the max.
        coarse = (0.65 * wcx + 0.35 * max_x, 0.65 * wcy + 0.35 * max_y)

        ox_norm, oy_norm = endpoint_patch_average(off_x, off_y, heat, coarse, cfg.endpoint_patch_radius)
        ox = float(ox_norm) * cfg.endpoint_radius
        oy = float(oy_norm) * cfg.endpoint_radius
        refined_x = float(np.clip(coarse[0] + ox, 0, w - 1))
        refined_y = float(np.clip(coarse[1] + oy, 0, h - 1))

        tx, ty = endpoint_patch_average(tan_x, tan_y, heat, coarse, cfg.endpoint_patch_radius)
        tx, ty, _ = unit(tx, ty)

        progress_val = sample_map_bilinear(pred["progress"], (refined_x, refined_y))
        score = float(values.max())
        endpoints.append(
            Endpoint(
                endpoint_id=f"{kind[0].upper()}{len(endpoints)}",
                kind=kind,
                coarse_xy=(float(coarse[0]), float(coarse[1])),
                refined_xy=(refined_x, refined_y),
                snapped_xy=(int(round(refined_x)), int(round(refined_y))),
                score=score,
                progress_value=float(progress_val),
                tangent_xy=(float(tx), float(ty)),
                offset_xy=(float(ox), float(oy)),
                component_id=int(cid),
            )
        )

    return nms_endpoints(endpoints, cfg.endpoint_nms_dist)




def adaptive_prob_threshold(
    arr: np.ndarray,
    floor: float,
    quantile: float,
    cap: float = 0.92,
) -> float:
    """Map-wise robust threshold.

    This intentionally returns conservative values. For sparse heatmaps, high
    quantiles can collapse to background, so the training-time floor remains a
    lower bound. For very confident maps, the cap prevents the decoder from using
    only the single brightest pixel and fragmenting a real stroke.
    """
    x = np.asarray(arr, dtype=np.float32)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float(floor)
    qv = float(np.percentile(x, quantile))
    mu = float(np.mean(x))
    sd = float(np.std(x))
    stat = mu + 2.5 * sd
    # Use the stronger of percentile/statistical threshold, but do not exceed cap.
    t = max(float(floor), min(max(qv, stat), float(cap)))
    return float(np.clip(t, floor, cap))


def median_node_diag(nodes: List[Node]) -> float:
    if not nodes:
        return 48.0
    vals = []
    for n in nodes:
        x1, y1, x2, y2 = n.bbox_xyxy
        vals.append(math.hypot(max(1, x2 - x1 + 1), max(1, y2 - y1 + 1)))
    return float(np.median(vals)) if vals else 48.0


def adapt_config_pre_nodes(pred: Dict[str, np.ndarray], cfg: LADDecoderConfig) -> LADDecoderConfig:
    """Adapt map thresholds before node/endpoint extraction."""
    if not cfg.auto_adapt:
        return cfg
    out = replace(cfg)
    out.node_extent_thresh = adaptive_prob_threshold(
        pred["node_extent"], cfg.node_extent_thresh, cfg.adaptive_node_quantile, cap=0.88
    )
    out.node_center_thresh = adaptive_prob_threshold(
        pred["node_center"], cfg.node_center_thresh, 99.2, cap=0.92
    )
    out.head_thresh = adaptive_prob_threshold(
        pred["head"], cfg.head_thresh, cfg.adaptive_endpoint_quantile, cap=0.90
    )
    out.tail_thresh = adaptive_prob_threshold(
        pred["tail"], cfg.tail_thresh, cfg.adaptive_endpoint_quantile, cap=0.90
    )
    out.shaft_thresh = adaptive_prob_threshold(
        pred["shaft"], cfg.shaft_thresh, cfg.adaptive_support_quantile, cap=0.75
    )
    out.skeleton_thresh = adaptive_prob_threshold(
        pred["skeleton"], cfg.skeleton_thresh, cfg.adaptive_support_quantile, cap=0.75
    )
    return out


def adapt_config_post_nodes(pred: Dict[str, np.ndarray], cfg: LADDecoderConfig, nodes: List[Node]) -> LADDecoderConfig:
    """Adapt geometry-sensitive distances after node extraction."""
    if not cfg.auto_adapt:
        return cfg
    out = replace(cfg)
    med = median_node_diag(nodes)
    # These scale with actual diagram geometry instead of one 512x512-wide constant.
    out.endpoint_nms_dist = float(np.clip(0.075 * med, 4.0, 10.0))
    out.node_attach_max_dist = float(np.clip(0.42 * med, 16.0, 45.0))
    out.branch_min_opposite_endpoint_dist = float(np.clip(0.22 * med, 10.0, 28.0))
    out.duplicate_endpoint_dist = float(np.clip(0.12 * med, 6.0, 14.0))
    out.node_core_shrink_px = int(np.clip(round(0.08 * med), 4, 10))
    out.endpoint_force_radius = int(np.clip(round(0.08 * med), 4, 8))
    out.nearest_walkable_radius = int(np.clip(round(0.22 * med), 8, 18))
    # Support threshold adapts to the fused map but keeps a low floor, because
    # handwritten arrow strokes may be thin and broken.
    fused = (
        out.shaft_weight * pred["shaft"].astype(np.float32)
        + out.skeleton_weight * pred["skeleton"].astype(np.float32)
        + out.progress_weight * (pred["progress"] >= out.progress_support_thresh).astype(np.float32)
        + out.endpoint_weight * np.maximum(pred["head"], pred["tail"]).astype(np.float32)
    ) / max(out.shaft_weight + out.skeleton_weight + out.progress_weight + out.endpoint_weight, 1e-6)
    dyn_support = adaptive_prob_threshold(fused, cfg.support_thresh, cfg.adaptive_support_quantile, cap=0.55)
    out.support_thresh = float(np.clip(min(dyn_support, 0.30), 0.12, 0.30))
    return out

def build_node_core_mask(nodes: List[Node], shape_hw: Tuple[int, int], shrink_px: int) -> np.ndarray:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    for n in nodes:
        x1, y1, x2, y2 = n.bbox_xyxy
        sx1 = int(np.clip(x1 + shrink_px, 0, w - 1))
        sy1 = int(np.clip(y1 + shrink_px, 0, h - 1))
        sx2 = int(np.clip(x2 - shrink_px, 0, w - 1))
        sy2 = int(np.clip(y2 - shrink_px, 0, h - 1))
        if sx2 < sx1 or sy2 < sy1:
            sx1, sy1, sx2, sy2 = x1, y1, x2, y2
        cv2.rectangle(mask, (sx1, sy1), (sx2, sy2), 1, -1)
    return mask


def build_support_field(
    pred: Dict[str, np.ndarray],
    decoder_img: np.ndarray,
    nodes: List[Node],
    heads: List[Endpoint],
    tails: List[Endpoint],
    cfg: LADDecoderConfig,
) -> Dict[str, np.ndarray]:
    shaft = pred["shaft"].astype(np.float32)
    skeleton = pred["skeleton"].astype(np.float32)
    progress_mask_like = (pred["progress"] >= cfg.progress_support_thresh).astype(np.float32)
    endpoints_union = np.maximum(pred["head"], pred["tail"]).astype(np.float32)

    support = (
        cfg.shaft_weight * shaft +
        cfg.skeleton_weight * skeleton +
        cfg.progress_weight * progress_mask_like +
        cfg.endpoint_weight * endpoints_union
    )
    total_weight = cfg.shaft_weight + cfg.skeleton_weight + cfg.progress_weight + cfg.endpoint_weight

    # Always compute raw ink. V5 only computed this when use_image_ink_support=True,
    # but V6 uses it as a local fallback for broken learned-support paths. It is not
    # mixed into the main support unless requested, so normal cases remain model-driven.
    ink = image_ink_support(decoder_img).astype(np.float32)
    if cfg.use_image_ink_support:
        support += cfg.ink_weight * ink
        total_weight += cfg.ink_weight

    support = np.clip(support / max(total_weight, 1e-6), 0.0, 1.0).astype(np.float32)

    walkable = (support >= cfg.support_thresh).astype(np.uint8)
    walkable = np.maximum(walkable, (shaft >= cfg.shaft_thresh).astype(np.uint8))
    walkable = np.maximum(walkable, (skeleton >= cfg.skeleton_thresh).astype(np.uint8))

    if cfg.support_close_kernel and cfg.support_close_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg.support_close_kernel, cfg.support_close_kernel))
        walkable = cv2.morphologyEx(walkable, cv2.MORPH_CLOSE, k)
    if cfg.support_dilate_kernel and cfg.support_dilate_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg.support_dilate_kernel, cfg.support_dilate_kernel))
        walkable = cv2.dilate(walkable, k, iterations=1)

    node_core = build_node_core_mask(nodes, shaft.shape, cfg.node_core_shrink_px)
    # Do not completely remove node interiors from walkable. Instead, A* gets a penalty.
    # For endpoints, force small disks walkable so snapped starts/goals do not disappear.
    endpoint_force = np.zeros_like(walkable, dtype=np.uint8)
    for ep in list(heads) + list(tails):
        draw_disk(endpoint_force, ep.refined_xy, cfg.endpoint_force_radius, value=1)
    walkable = np.maximum(walkable, endpoint_force)

    return {
        "support": support,
        "walkable": walkable.astype(np.uint8),
        "node_core": node_core.astype(np.uint8),
        "ink": ink.astype(np.float32),
        "endpoint_force": endpoint_force.astype(np.uint8),
    }


def attach_endpoint_to_node(ep: Endpoint, nodes: List[Node], cfg: LADDecoderConfig) -> Tuple[Optional[str], Optional[float]]:
    if not nodes:
        return None, None

    px, py = ep.refined_xy
    tx, ty = ep.tangent_xy
    # Tail tangent points away from source node, so source probe is behind.
    # Head tangent points into destination endpoint, so destination probe is ahead.
    if ep.kind == "tail":
        probe = (px - tx * cfg.tangent_probe_dist, py - ty * cfg.tangent_probe_dist)
    else:
        probe = (px + tx * cfg.tangent_probe_dist, py + ty * cfg.tangent_probe_dist)

    best_id = None
    best_score = float("inf")
    for n in nodes:
        d_point = point_to_bbox_distance(ep.refined_xy, n.bbox_xyxy)
        d_probe = point_to_bbox_distance(probe, n.bbox_xyxy)
        inside_bonus = -4.0 if point_inside_bbox(ep.refined_xy, n.bbox_xyxy) else 0.0
        score = 0.65 * d_point + 0.35 * d_probe + inside_bonus
        if score < best_score:
            best_score = float(score)
            best_id = n.node_id

    if best_score <= cfg.node_attach_max_dist:
        return best_id, best_score
    return None, best_score


def attach_all_endpoints(endpoints: List[Endpoint], nodes: List[Node], cfg: LADDecoderConfig) -> None:
    for ep in endpoints:
        nid, score = attach_endpoint_to_node(ep, nodes, cfg)
        ep.attached_node_id = nid
        ep.attach_score = score


def nearest_walkable(seed_xy: Sequence[float], walkable: np.ndarray, max_radius: int) -> Optional[Tuple[int, int]]:
    h, w = walkable.shape
    sx = int(round(float(seed_xy[0]))); sy = int(round(float(seed_xy[1])))
    if inside(sx, sy, w, h) and walkable[sy, sx] > 0:
        return sx, sy
    best = None
    best_d2 = float("inf")
    for r in range(1, max_radius + 1):
        x0 = max(0, sx - r); x1 = min(w - 1, sx + r)
        y0 = max(0, sy - r); y1 = min(h - 1, sy + r)
        ys, xs = np.where(walkable[y0:y1 + 1, x0:x1 + 1] > 0)
        if len(xs) == 0:
            continue
        xs = xs + x0; ys = ys + y0
        d2 = (xs - sx) ** 2 + (ys - sy) ** 2
        idx = int(np.argmin(d2))
        if float(d2[idx]) < best_d2:
            best_d2 = float(d2[idx])
            best = (int(xs[idx]), int(ys[idx]))
        if best is not None:
            return best
    return None


def reconstruct_path(came_from: Dict[Tuple[int, int], Tuple[int, int]], end_state: Tuple[int, int]) -> List[Tuple[int, int]]:
    path = [end_state]
    cur = end_state
    while cur in came_from:
        cur = came_from[cur]
        path.append(cur)
    path.reverse()
    return path


def astar_trace(
    tail: Endpoint,
    head: Endpoint,
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    cfg: LADDecoderConfig,
) -> Optional[Dict[str, Any]]:
    support = support_maps["support"]
    walkable = support_maps["walkable"]
    node_core = support_maps["node_core"]
    h, w = support.shape

    start = nearest_walkable(tail.refined_xy, walkable, cfg.nearest_walkable_radius)
    goal = nearest_walkable(head.refined_xy, walkable, cfg.nearest_walkable_radius)
    if start is None or goal is None:
        return None

    dir_x = pred["dir_x"]
    dir_y = pred["dir_y"]
    progress = pred["progress"]

    pq: List[Tuple[float, float, Tuple[int, int], Optional[Tuple[int, int]]]] = []
    heapq.heappush(pq, (0.0, 0.0, start, None))
    came_from: Dict[Tuple[int, int], Tuple[int, int]] = {}
    best_g: Dict[Tuple[int, int], float] = {start: 0.0}
    expansions = 0
    gx, gy = goal

    while pq:
        _, g, cur, prev = heapq.heappop(pq)
        cx, cy = cur
        if (cx - gx) ** 2 + (cy - gy) ** 2 <= cfg.goal_radius ** 2:
            path = reconstruct_path(came_from, cur)
            if cur != goal:
                path.append(goal)
            return {
                "path": path,
                "cost": float(g),
                "start_snap": start,
                "goal_snap": goal,
                "expansions": expansions,
            }

        expansions += 1
        if expansions > cfg.max_expansions:
            return None

        cur_prog = float(progress[cy, cx])
        for ddx, ddy in NEIGHBORS_8:
            nx_, ny_ = cx + ddx, cy + ddy
            if not inside(nx_, ny_, w, h):
                continue
            if walkable[ny_, nx_] == 0:
                continue

            slen = step_length(ddx, ddy)
            svx, svy = ddx / slen, ddy / slen

            support_val = float(support[ny_, nx_])
            support_cost = cfg.support_cost_weight * (1.0 - support_val)

            local_dir_score = max(0.0, cosine2(svx, svy, float(dir_x[ny_, nx_]), float(dir_y[ny_, nx_])))
            direction_cost = cfg.direction_cost_weight * (1.0 - local_dir_score)

            next_prog = float(progress[ny_, nx_])
            backward = max(0.0, cur_prog - next_prog + cfg.progress_margin)
            progress_cost = cfg.progress_cost_weight * backward

            curvature_cost = 0.0
            if prev is not None:
                px, py = prev
                pvx, pvy = cx - px, cy - py
                pn = math.hypot(pvx, pvy)
                if pn > 1e-8:
                    pvx /= pn; pvy /= pn
                    turn_score = max(0.0, cosine2(pvx, pvy, svx, svy))
                    curvature_cost = cfg.curvature_cost_weight * (1.0 - turn_score)

            node_cost = cfg.node_core_cost_weight * float(node_core[ny_, nx_] > 0)
            edge_cost = (
                cfg.step_cost_weight * slen + support_cost + direction_cost +
                progress_cost + curvature_cost + node_cost
            )
            ng = g + edge_cost
            nxt = (nx_, ny_)
            if ng < best_g.get(nxt, float("inf")):
                best_g[nxt] = ng
                came_from[nxt] = cur
                heuristic = cfg.heuristic_weight * math.hypot(gx - nx_, gy - ny_)
                heapq.heappush(pq, (ng + heuristic, ng, nxt, cur))

    return None


def path_iou(path_a: Sequence[Tuple[int, int]], path_b: Sequence[Tuple[int, int]]) -> float:
    sa = set((int(x), int(y)) for x, y in path_a)
    sb = set((int(x), int(y)) for x, y in path_b)
    if not sa and not sb:
        return 1.0
    return float(len(sa & sb) / max(1, len(sa | sb)))


def score_path(
    path: List[Tuple[int, int]],
    tail: Endpoint,
    head: Endpoint,
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
) -> Dict[str, float]:
    if len(path) < 2:
        return {
            "support_score": 0.0, "shaft_score": 0.0, "skeleton_score": 0.0,
            "direction_score": 0.0, "progress_score": 0.0, "tangent_score": 0.0,
        }
    xs = np.array([p[0] for p in path], dtype=np.int32)
    ys = np.array([p[1] for p in path], dtype=np.int32)
    support_vals = support_maps["support"][ys, xs]
    shaft_vals = pred["shaft"][ys, xs]
    skel_vals = pred["skeleton"][ys, xs]
    prog_vals = pred["progress"][ys, xs]

    dir_scores = []
    good_progress = []
    for i in range(len(path) - 1):
        x1, y1 = path[i]
        x2, y2 = path[i + 1]
        dx = x2 - x1; dy = y2 - y1
        n = math.hypot(dx, dy)
        if n < 1e-8:
            continue
        sx, sy = dx / n, dy / n
        dir_scores.append(max(0.0, cosine2(sx, sy, float(pred["dir_x"][y2, x2]), float(pred["dir_y"][y2, x2]))))
        good_progress.append(1.0 if float(pred["progress"][y2, x2]) + 0.025 >= float(pred["progress"][y1, x1]) else 0.0)

    # Endpoint tangent score: first step should align with tail tangent; last step with head tangent.
    x0, y0 = path[0]
    x1, y1 = path[min(len(path) - 1, 2)]
    dx0, dy0 = x1 - x0, y1 - y0
    xn0, yn0 = path[max(0, len(path) - 3)]
    xn1, yn1 = path[-1]
    dxn, dyn = xn1 - xn0, yn1 - yn0
    tail_tan_score = max(0.0, cosine2(dx0, dy0, *tail.tangent_xy))
    head_tan_score = max(0.0, cosine2(dxn, dyn, *head.tangent_xy))
    tangent_score = 0.5 * tail_tan_score + 0.5 * head_tan_score

    progress_monotonic = float(np.mean(good_progress)) if good_progress else 0.0
    progress_span = float(np.clip(prog_vals[-1] - prog_vals[0], 0.0, 1.0)) if len(prog_vals) else 0.0
    progress_score = 0.75 * progress_monotonic + 0.25 * progress_span

    return {
        "support_score": float(np.mean(support_vals)) if len(support_vals) else 0.0,
        "shaft_score": float(np.mean(shaft_vals)) if len(shaft_vals) else 0.0,
        "skeleton_score": float(np.mean(skel_vals)) if len(skel_vals) else 0.0,
        "direction_score": float(np.mean(dir_scores)) if dir_scores else 0.0,
        "progress_score": float(progress_score),
        "tangent_score": float(tangent_score),
    }


def make_arrow_candidate(
    tail: Endpoint,
    head: Endpoint,
    trace: Dict[str, Any],
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
) -> Arrow:
    path = trace["path"]
    scores = score_path(path, tail, head, pred, support_maps)
    endpoint_score = 0.5 * float(tail.score) + 0.5 * float(head.score)
    attach_bonus = 1.0 if tail.attached_node_id is not None and head.attached_node_id is not None else 0.55
    confidence = (
        0.30 * scores["support_score"] +
        0.15 * scores["direction_score"] +
        0.15 * scores["progress_score"] +
        0.10 * scores["tangent_score"] +
        0.15 * endpoint_score +
        0.10 * scores["shaft_score"] +
        0.05 * scores["skeleton_score"]
    ) * attach_bonus
    return Arrow(
        arrow_id=-1,
        tail_endpoint_id=tail.endpoint_id,
        head_endpoint_id=head.endpoint_id,
        tail_xy=tail.refined_xy,
        head_xy=head.refined_xy,
        tail_hint={"x": float(tail.refined_xy[0]), "y": float(tail.refined_xy[1])},
        head_hint={"x": float(head.refined_xy[0]), "y": float(head.refined_xy[1])},
        path_xy=[(int(x), int(y)) for x, y in path],
        tail_node_id=tail.attached_node_id,
        head_node_id=head.attached_node_id,
        path_cost=float(trace["cost"]),
        confidence=float(confidence),
        support_score=scores["support_score"],
        shaft_score=scores["shaft_score"],
        skeleton_score=scores["skeleton_score"],
        direction_score=scores["direction_score"],
        progress_score=scores["progress_score"],
        endpoint_score=float(endpoint_score),
        tangent_score=scores["tangent_score"],
        self_loop=bool(tail.attached_node_id is not None and tail.attached_node_id == head.attached_node_id),
        meta={
            "tail_progress": float(tail.progress_value),
            "head_progress": float(head.progress_value),
            "start_snap": trace.get("start_snap"),
            "goal_snap": trace.get("goal_snap"),
            "expansions": trace.get("expansions"),
            "tail_attach_score": tail.attach_score,
            "head_attach_score": head.attach_score,
        },
    )


def generate_arrow_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    tails: List[Endpoint],
    heads: List[Endpoint],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    candidates: List[Arrow] = []
    for tail in tails:
        for head in heads:
            if l2(tail.refined_xy, head.refined_xy) > cfg.max_pair_dist:
                continue
            trace = astar_trace(tail, head, pred, support_maps, cfg)
            if trace is None:
                continue
            if len(trace["path"]) < cfg.min_path_len:
                continue
            arrow = make_arrow_candidate(tail, head, trace, pred, support_maps)
            if (not cfg.keep_unattached_arrows) and (arrow.tail_node_id is None or arrow.head_node_id is None):
                continue
            candidates.append(arrow)
    return candidates


def _same_node_pair(a: Arrow, b: Arrow) -> bool:
    return a.tail_node_id == b.tail_node_id and a.head_node_id == b.head_node_id


def _branch_endpoint_reuse_ok(cand: Arrow, selected: List[Arrow], cfg: LADDecoderConfig) -> bool:
    """Permit endpoint reuse only when it behaves like a genuine branch.

    Crowded DFA/flowchart nodes often produce one tail/head heatmap peak for two
    nearby physical arrows. A strict one-endpoint-one-arrow rule then deletes a
    correct edge even when the shaft/path evidence exists. This check keeps the
    duplicate guard but allows fan-out/fan-in when the opposite endpoint and traced
    path clearly diverge.
    """
    shared = [
        kept for kept in selected
        if cand.tail_endpoint_id == kept.tail_endpoint_id or cand.head_endpoint_id == kept.head_endpoint_id
    ]
    if not shared:
        return True

    for kept in shared:
        shared_tail = cand.tail_endpoint_id == kept.tail_endpoint_id
        shared_head = cand.head_endpoint_id == kept.head_endpoint_id

        if shared_tail and shared_head:
            return False

        if cfg.branch_require_different_node_pair and _same_node_pair(cand, kept):
            return False

        if shared_tail and l2(cand.head_xy, kept.head_xy) < cfg.branch_min_opposite_endpoint_dist:
            return False

        if shared_head and l2(cand.tail_xy, kept.tail_xy) < cfg.branch_min_opposite_endpoint_dist:
            return False

        if path_iou(cand.path_xy, kept.path_xy) > cfg.branch_max_path_iou:
            return False

    return True


def select_arrows(candidates: List[Arrow], cfg: LADDecoderConfig) -> List[Arrow]:
    selected: List[Arrow] = []
    used_tails = set()
    used_heads = set()
    for cand in sorted(candidates, key=lambda a: a.confidence, reverse=True):
        if cand.confidence < cfg.min_arrow_confidence:
            continue

        if not cfg.allow_endpoint_reuse:
            endpoint_already_used = cand.tail_endpoint_id in used_tails or cand.head_endpoint_id in used_heads
            if endpoint_already_used:
                if not cfg.branch_aware_endpoint_reuse:
                    continue
                if not _branch_endpoint_reuse_ok(cand, selected, cfg):
                    continue

        duplicate = False
        for kept in selected:
            same_ep = l2(cand.tail_xy, kept.tail_xy) <= cfg.duplicate_endpoint_dist and l2(cand.head_xy, kept.head_xy) <= cfg.duplicate_endpoint_dist
            same_path = path_iou(cand.path_xy, kept.path_xy) >= cfg.duplicate_path_iou
            if same_ep or (same_path and cand.tail_node_id == kept.tail_node_id and cand.head_node_id == kept.head_node_id):
                duplicate = True
                break
        if duplicate:
            continue
        selected.append(cand)
        used_tails.add(cand.tail_endpoint_id)
        used_heads.add(cand.head_endpoint_id)
    for idx, a in enumerate(selected, start=1):
        a.arrow_id = idx
    return selected


def _clone_endpoint_as_virtual(ep: Endpoint, new_kind: str, new_id: str, cfg: LADDecoderConfig) -> Endpoint:
    """Create a low-priority opposite-role endpoint at the same physical port."""
    tx, ty = ep.tangent_xy
    return Endpoint(
        endpoint_id=new_id,
        kind=new_kind,
        coarse_xy=ep.coarse_xy,
        refined_xy=ep.refined_xy,
        snapped_xy=ep.snapped_xy,
        score=float(ep.score) * cfg.port_repair_score_scale,
        progress_value=ep.progress_value,
        tangent_xy=(-float(tx), -float(ty)),
        offset_xy=ep.offset_xy,
        attached_node_id=ep.attached_node_id,
        attach_score=ep.attach_score,
        component_id=ep.component_id,
    )


def _max_iou_against_selected(path_xy: Sequence[Tuple[int, int]], selected: Sequence[Arrow]) -> float:
    if not selected:
        return 0.0
    return max(path_iou(path_xy, a.path_xy) for a in selected)


def generate_port_role_repair_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    heads: List[Endpoint],
    tails: List[Endpoint],
    selected: List[Arrow],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    """Second-pass candidate generation for missing role at crowded node ports.

    Normal decoding requires a tail heatmap peak and a head heatmap peak. In dense
    handwritten DFA nodes, a port can be both the head of one arrow and the tail of
    another. The heatmap may preserve only the stronger role. This pass uses the
    already selected edges to identify such ports and creates virtual opposite-role
    endpoints only at those selected ports. It then tries to connect them to unused
    endpoints.
    """
    if not cfg.enable_port_role_repair or not selected:
        return []

    selected_head_ids = {a.head_endpoint_id for a in selected}
    selected_tail_ids = {a.tail_endpoint_id for a in selected}
    unused_heads = [h for h in heads if h.endpoint_id not in selected_head_ids and h.attached_node_id is not None]
    unused_tails = [t for t in tails if t.endpoint_id not in selected_tail_ids and t.attached_node_id is not None]

    head_by_id = {h.endpoint_id: h for h in heads}
    tail_by_id = {t.endpoint_id: t for t in tails}

    repair_cfg = replace(cfg, progress_cost_weight=cfg.port_repair_progress_cost_weight)
    repair_candidates: List[Arrow] = []

    # Case A: accepted head also behaves as a missing outgoing tail.
    for kept in selected:
        src_head = head_by_id.get(kept.head_endpoint_id)
        if src_head is None or src_head.attached_node_id is None:
            continue
        vt = _clone_endpoint_as_virtual(src_head, "tail", f"VT_from_{src_head.endpoint_id}", cfg)
        for head in unused_heads:
            if vt.attached_node_id == head.attached_node_id:
                continue
            if l2(vt.refined_xy, head.refined_xy) > cfg.max_pair_dist:
                continue
            trace = astar_trace(vt, head, pred, support_maps, repair_cfg)
            if trace is None or len(trace["path"]) < cfg.min_path_len:
                continue
            arrow = make_arrow_candidate(vt, head, trace, pred, support_maps)
            arrow.confidence *= cfg.port_repair_confidence_scale
            arrow.meta["port_role_repair"] = True
            arrow.meta["virtual_tail_from_head"] = src_head.endpoint_id
            arrow.meta["selected_arrow_that_exposed_port"] = kept.arrow_id
            arrow.meta["max_iou_with_selected"] = _max_iou_against_selected(arrow.path_xy, selected)
            if arrow.confidence < cfg.port_repair_min_confidence:
                continue
            if arrow.support_score < cfg.port_repair_min_support_score or arrow.shaft_score < cfg.port_repair_min_shaft_score:
                continue
            if arrow.meta["max_iou_with_selected"] > cfg.port_repair_max_path_iou:
                continue
            repair_candidates.append(arrow)

    # Case B: accepted tail also behaves as a missing incoming head.
    for kept in selected:
        dst_tail = tail_by_id.get(kept.tail_endpoint_id)
        if dst_tail is None or dst_tail.attached_node_id is None:
            continue
        vh = _clone_endpoint_as_virtual(dst_tail, "head", f"VH_from_{dst_tail.endpoint_id}", cfg)
        for tail in unused_tails:
            if tail.attached_node_id == vh.attached_node_id:
                continue
            if l2(tail.refined_xy, vh.refined_xy) > cfg.max_pair_dist:
                continue
            trace = astar_trace(tail, vh, pred, support_maps, repair_cfg)
            if trace is None or len(trace["path"]) < cfg.min_path_len:
                continue
            arrow = make_arrow_candidate(tail, vh, trace, pred, support_maps)
            arrow.confidence *= cfg.port_repair_confidence_scale
            arrow.meta["port_role_repair"] = True
            arrow.meta["virtual_head_from_tail"] = dst_tail.endpoint_id
            arrow.meta["selected_arrow_that_exposed_port"] = kept.arrow_id
            arrow.meta["max_iou_with_selected"] = _max_iou_against_selected(arrow.path_xy, selected)
            if arrow.confidence < cfg.port_repair_min_confidence:
                continue
            if arrow.support_score < cfg.port_repair_min_support_score or arrow.shaft_score < cfg.port_repair_min_shaft_score:
                continue
            if arrow.meta["max_iou_with_selected"] > cfg.port_repair_max_path_iou:
                continue
            repair_candidates.append(arrow)

    repair_candidates = sorted(repair_candidates, key=lambda a: a.confidence, reverse=True)[:cfg.port_repair_max_candidates]
    return repair_candidates



# -----------------------------------------------------------------------------
# V5 stroke-graph fallback: topology-first candidate generation
# -----------------------------------------------------------------------------

def zhang_suen_thinning(binary: np.ndarray, max_iter: int = 128) -> np.ndarray:
    """Small dependency-free thinning fallback.

    Input/output are uint8 {0,1}. It is slower than skimage/cv2.ximgproc but fine
    for 512x512 inference maps.
    """
    img = (binary > 0).astype(np.uint8).copy()
    if img.size == 0 or int(img.sum()) == 0:
        return img
    h, w = img.shape
    for _ in range(max_iter):
        changed = False
        to_remove: List[Tuple[int, int]] = []
        for y in range(1, h - 1):
            for x in range(1, w - 1):
                if img[y, x] == 0:
                    continue
                p2 = img[y - 1, x];     p3 = img[y - 1, x + 1]
                p4 = img[y, x + 1];     p5 = img[y + 1, x + 1]
                p6 = img[y + 1, x];     p7 = img[y + 1, x - 1]
                p8 = img[y, x - 1];     p9 = img[y - 1, x - 1]
                nb = int(p2+p3+p4+p5+p6+p7+p8+p9)
                if nb < 2 or nb > 6:
                    continue
                seq = [p2,p3,p4,p5,p6,p7,p8,p9,p2]
                trans = sum(1 for i in range(8) if seq[i] == 0 and seq[i+1] == 1)
                if trans != 1:
                    continue
                if p2 * p4 * p6 != 0:
                    continue
                if p4 * p6 * p8 != 0:
                    continue
                to_remove.append((x, y))
        if to_remove:
            changed = True
            for x, y in to_remove:
                img[y, x] = 0

        to_remove = []
        for y in range(1, h - 1):
            for x in range(1, w - 1):
                if img[y, x] == 0:
                    continue
                p2 = img[y - 1, x];     p3 = img[y - 1, x + 1]
                p4 = img[y, x + 1];     p5 = img[y + 1, x + 1]
                p6 = img[y + 1, x];     p7 = img[y + 1, x - 1]
                p8 = img[y, x - 1];     p9 = img[y - 1, x - 1]
                nb = int(p2+p3+p4+p5+p6+p7+p8+p9)
                if nb < 2 or nb > 6:
                    continue
                seq = [p2,p3,p4,p5,p6,p7,p8,p9,p2]
                trans = sum(1 for i in range(8) if seq[i] == 0 and seq[i+1] == 1)
                if trans != 1:
                    continue
                if p2 * p4 * p8 != 0:
                    continue
                if p2 * p6 * p8 != 0:
                    continue
                to_remove.append((x, y))
        if to_remove:
            changed = True
            for x, y in to_remove:
                img[y, x] = 0
        if not changed:
            break
    return img.astype(np.uint8)


def skeletonize_mask(binary: np.ndarray) -> np.ndarray:
    binary = (binary > 0).astype(np.uint8)
    try:
        from skimage.morphology import skeletonize as _sk_skeletonize  # type: ignore
        return _sk_skeletonize(binary > 0).astype(np.uint8)
    except Exception:
        return zhang_suen_thinning(binary)


def component_endpoint_pixels(skel: np.ndarray, max_points: int = 10) -> List[Tuple[int, int]]:
    ys, xs = np.where(skel > 0)
    pts = [(int(x), int(y)) for x, y in zip(xs, ys)]
    if not pts:
        return []
    h, w = skel.shape
    endpoints: List[Tuple[int, int]] = []
    for x, y in pts:
        deg = 0
        for dx, dy in NEIGHBORS_8:
            nx_, ny_ = x + dx, y + dy
            if inside(nx_, ny_, w, h) and skel[ny_, nx_] > 0:
                deg += 1
        if deg <= 1:
            endpoints.append((x, y))
    if len(endpoints) >= 2:
        # Keep spatially diverse endpoints by greedily taking far-apart points.
        chosen: List[Tuple[int, int]] = []
        for pnt in sorted(endpoints, key=lambda p: (p[1], p[0])):
            if all(l2(pnt, c) > 4.0 for c in chosen):
                chosen.append(pnt)
            if len(chosen) >= max_points:
                break
        return chosen

    # Closed curves or over-thick components: use approximate diameter points.
    if len(pts) == 1:
        return pts
    sample = pts if len(pts) <= 500 else pts[::max(1, len(pts)//500)]
    a = sample[0]
    b = max(sample, key=lambda q: l2(a, q))
    c = max(sample, key=lambda q: l2(b, q))
    return [b, c]


def nearest_point_in_mask(seed: Sequence[float], mask: np.ndarray, max_radius: int = 12) -> Optional[Tuple[int, int]]:
    h, w = mask.shape
    sx = int(round(float(seed[0]))); sy = int(round(float(seed[1])))
    if inside(sx, sy, w, h) and mask[sy, sx] > 0:
        return (sx, sy)
    best = None
    best_d = float("inf")
    for r in range(1, max_radius + 1):
        x0 = max(0, sx - r); x1 = min(w - 1, sx + r)
        y0 = max(0, sy - r); y1 = min(h - 1, sy + r)
        ys, xs = np.where(mask[y0:y1+1, x0:x1+1] > 0)
        if len(xs) == 0:
            continue
        for xx, yy in zip(xs + x0, ys + y0):
            d = math.hypot(float(xx - sx), float(yy - sy))
            if d < best_d:
                best_d = d
                best = (int(xx), int(yy))
        if best is not None:
            return best
    return None


def shortest_path_on_component(
    mask: np.ndarray,
    support: np.ndarray,
    start_xy: Sequence[float],
    goal_xy: Sequence[float],
    max_expansions: int = 60000,
) -> Optional[List[Tuple[int, int]]]:
    """A* constrained to a single stroke component mask."""
    h, w = mask.shape
    start = nearest_point_in_mask(start_xy, mask, max_radius=14)
    goal = nearest_point_in_mask(goal_xy, mask, max_radius=14)
    if start is None or goal is None:
        return None
    if start == goal:
        return [start]
    gx, gy = goal
    pq: List[Tuple[float, float, Tuple[int, int], Optional[Tuple[int, int]]]] = []
    heapq.heappush(pq, (0.0, 0.0, start, None))
    came_from: Dict[Tuple[int, int], Optional[Tuple[int, int]]] = {}
    best_g: Dict[Tuple[int, int], float] = {start: 0.0}
    expansions = 0
    while pq:
        _, g, cur, prev = heapq.heappop(pq)
        if cur in came_from:
            continue
        came_from[cur] = prev
        cx, cy = cur
        if math.hypot(cx - gx, cy - gy) <= 1.5:
            path = [(cx, cy)]
            pcur = cur
            while came_from[pcur] is not None:
                pcur = came_from[pcur]  # type: ignore[index]
                path.append(pcur)
            path.reverse()
            if path[-1] != goal:
                path.append(goal)
            return path
        expansions += 1
        if expansions > max_expansions:
            return None
        for dx, dy in NEIGHBORS_8:
            nx_, ny_ = cx + dx, cy + dy
            if not inside(nx_, ny_, w, h) or mask[ny_, nx_] == 0:
                continue
            sl = step_length(dx, dy)
            # Stay inside high-support pixels but allow thin weak links in the same component.
            cost = sl + 1.25 * (1.0 - float(support[ny_, nx_]))
            nxt = (nx_, ny_)
            ng = g + cost
            if ng < best_g.get(nxt, float("inf")):
                best_g[nxt] = ng
                heuristic = 0.08 * math.hypot(gx - nx_, gy - ny_)
                heapq.heappush(pq, (ng + heuristic, ng, nxt, cur))
    return None


def nearest_node_to_point(pt: Sequence[float], nodes: List[Node], max_dist: float) -> Tuple[Optional[Node], float]:
    best: Optional[Node] = None
    best_d = float("inf")
    for n in nodes:
        d = point_to_bbox_distance(pt, n.bbox_xyxy)
        if d < best_d:
            best_d = d
            best = n
    if best is not None and best_d <= max_dist:
        return best, best_d
    return None, best_d


def orient_stroke_path(
    path: List[Tuple[int, int]],
    pred: Dict[str, np.ndarray],
) -> Tuple[List[Tuple[int, int]], float, float, float]:
    """Return path ordered tail->head using progress + direction-field votes."""
    if len(path) < 2:
        return path, 0.0, 0.0, 0.0
    def orient_score(pp: List[Tuple[int, int]]) -> Tuple[float, float, float]:
        dir_scores: List[float] = []
        prog_good: List[float] = []
        for i in range(len(pp) - 1):
            x1, y1 = pp[i]
            x2, y2 = pp[i + 1]
            dx, dy = x2 - x1, y2 - y1
            n = math.hypot(dx, dy)
            if n < 1e-8:
                continue
            sx, sy = dx / n, dy / n
            dir_scores.append(max(0.0, cosine2(sx, sy, float(pred["dir_x"][y2, x2]), float(pred["dir_y"][y2, x2]))))
            prog_good.append(1.0 if float(pred["progress"][y2, x2]) + 0.025 >= float(pred["progress"][y1, x1]) else 0.0)
        p0 = sample_map_bilinear(pred["progress"], pp[0])
        p1 = sample_map_bilinear(pred["progress"], pp[-1])
        prog_span = float(np.clip(p1 - p0, -1.0, 1.0))
        dir_mean = float(np.mean(dir_scores)) if dir_scores else 0.0
        prog_mean = float(np.mean(prog_good)) if prog_good else 0.0
        score = 0.45 * dir_mean + 0.35 * prog_mean + 0.20 * max(0.0, prog_span)
        return score, dir_mean, prog_span
    f_score, f_dir, f_prog = orient_score(path)
    rpath = list(reversed(path))
    r_score, r_dir, r_prog = orient_score(rpath)
    if r_score > f_score:
        return rpath, r_score, r_dir, r_prog
    return path, f_score, f_dir, f_prog


def virtual_endpoint_from_stroke(
    endpoint_id: str,
    kind: str,
    xy: Sequence[float],
    path: List[Tuple[int, int]],
    node: Optional[Node],
    score: float,
    pred: Dict[str, np.ndarray],
) -> Endpoint:
    if kind == "tail":
        if len(path) >= 3:
            dx = path[min(2, len(path)-1)][0] - path[0][0]
            dy = path[min(2, len(path)-1)][1] - path[0][1]
        else:
            dx, dy = 1.0, 0.0
    else:
        if len(path) >= 3:
            dx = path[-1][0] - path[max(0, len(path)-3)][0]
            dy = path[-1][1] - path[max(0, len(path)-3)][1]
        else:
            dx, dy = 1.0, 0.0
    tx, ty, _ = unit(dx, dy)
    x, y = float(xy[0]), float(xy[1])
    return Endpoint(
        endpoint_id=endpoint_id,
        kind=kind,
        coarse_xy=(x, y),
        refined_xy=(x, y),
        snapped_xy=(int(round(x)), int(round(y))),
        score=float(score),
        progress_value=float(sample_map_bilinear(pred["progress"], (x, y))),
        tangent_xy=(tx, ty),
        offset_xy=(0.0, 0.0),
        attached_node_id=node.node_id if node is not None else None,
        attach_score=point_to_bbox_distance((x, y), node.bbox_xyxy) if node is not None else None,
        component_id=None,
    )


def generate_stroke_graph_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    nodes: List[Node],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    """Generate edge candidates from stroke topology, not endpoint roles.

    This is designed for exactly the failure case where a visible arrow shaft exists
    but the endpoint heatmap gives the wrong role at a crowded node port. The stroke
    itself becomes the candidate; endpoints/progress/direction only score/orient it.
    """
    if not cfg.enable_stroke_graph_candidates or not nodes:
        return []

    support = support_maps["support"].astype(np.float32)
    walkable = support_maps["walkable"].astype(np.uint8).copy()
    node_core = support_maps["node_core"].astype(np.uint8)
    h, w = walkable.shape

    # Remove node interiors so components represent strokes between/around nodes.
    stroke_mask = walkable.copy()
    stroke_mask[node_core > 0] = 0

    # Slightly clean the topology while preserving thin hand-drawn links.
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    stroke_mask = cv2.morphologyEx(stroke_mask, cv2.MORPH_OPEN, k)
    stroke_mask = cv2.morphologyEx(stroke_mask, cv2.MORPH_CLOSE, k)

    num, labels, stats, _ = cv2.connectedComponentsWithStats(stroke_mask.astype(np.uint8), connectivity=8)
    med = median_node_diag(nodes)
    contact_max = float(np.clip(cfg.stroke_graph_node_contact_factor * med,
                                cfg.stroke_graph_node_contact_min,
                                cfg.stroke_graph_node_contact_max))
    min_area = max(int(cfg.stroke_graph_min_component_area), int(0.12 * med))

    comp_ids = list(range(1, num))
    comp_ids = sorted(comp_ids, key=lambda cid: int(stats[cid, cv2.CC_STAT_AREA]), reverse=True)[:cfg.stroke_graph_max_components]

    candidates: List[Arrow] = []
    seen_node_pairs: set = set()
    for cid in comp_ids:
        area = int(stats[cid, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        comp = (labels == cid).astype(np.uint8)
        skel = skeletonize_mask(comp)
        if int(skel.sum()) < cfg.stroke_graph_min_path_len:
            # Fall back to the component itself when thinning is too aggressive.
            skel = comp

        endpoints = component_endpoint_pixels(skel, max_points=12)
        if len(endpoints) < 2:
            continue

        # Attach component endpoints to nearby nodes.
        contacts: List[Tuple[Tuple[int, int], Node, float]] = []
        for ep_xy in endpoints:
            node, dist = nearest_node_to_point(ep_xy, nodes, contact_max)
            if node is not None:
                if all(not (node.node_id == old_node.node_id and l2(ep_xy, old_xy) < 6.0) for old_xy, old_node, _ in contacts):
                    contacts.append((ep_xy, node, dist))
        if len(contacts) < 2:
            continue

        # Try pairs of contacts on the same stroke component.
        for i in range(len(contacts)):
            for j in range(i + 1, len(contacts)):
                p0, n0, d0 = contacts[i]
                p1, n1, d1 = contacts[j]
                same_node_contact = n0.node_id == n1.node_id
                if cfg.stroke_graph_skip_self_loops and same_node_contact:
                    continue

                raw_path = shortest_path_on_component(skel, support, p0, p1, max_expansions=min(cfg.max_expansions, 60000))
                if raw_path is None or len(raw_path) < cfg.stroke_graph_min_path_len:
                    raw_path = shortest_path_on_component(comp, support, p0, p1, max_expansions=min(cfg.max_expansions, 60000))
                if raw_path is None or len(raw_path) < cfg.stroke_graph_min_path_len:
                    continue

                if same_node_contact:
                    # A real self-loop must leave the node region. Otherwise tiny border strokes
                    # and node outline fragments become fake loops.
                    bbox = n0.bbox_xyxy
                    max_excursion = max(point_to_bbox_distance((float(x), float(y)), bbox) for x, y in raw_path)
                    if len(raw_path) < max(cfg.stroke_graph_min_path_len, int(0.75 * med)):
                        continue
                    if max_excursion < max(8.0, 0.16 * med):
                        continue

                ordered_path, orient_score, orient_dir, orient_prog = orient_stroke_path(raw_path, pred)
                tail_node = n0 if ordered_path[0] == raw_path[0] else n1
                head_node = n1 if ordered_path[-1] == raw_path[-1] else n0

                pair_key = (cid, tail_node.node_id, head_node.node_id)
                if pair_key in seen_node_pairs:
                    continue
                seen_node_pairs.add(pair_key)

                tail_score = max(cfg.stroke_graph_endpoint_score, sample_map_bilinear(pred["tail"], ordered_path[0]))
                head_score = max(cfg.stroke_graph_endpoint_score, sample_map_bilinear(pred["head"], ordered_path[-1]))
                tail = virtual_endpoint_from_stroke(
                    f"SG{cid}_T_{tail_node.node_id}_{head_node.node_id}", "tail", ordered_path[0], ordered_path, tail_node, tail_score, pred
                )
                head = virtual_endpoint_from_stroke(
                    f"SG{cid}_H_{tail_node.node_id}_{head_node.node_id}", "head", ordered_path[-1], ordered_path, head_node, head_score, pred
                )
                arrow = make_arrow_candidate(
                    tail,
                    head,
                    {"path": ordered_path, "cost": float(len(ordered_path)), "start_snap": ordered_path[0], "goal_snap": ordered_path[-1], "expansions": None},
                    pred,
                    support_maps,
                )
                # Stroke topology is a fallback, not a licence to hallucinate. Scale it
                # slightly below direct endpoint candidates, then keep only strong paths.
                arrow.confidence *= cfg.stroke_graph_confidence_scale
                arrow.meta.update({
                    "source": "stroke_graph",
                    "stroke_component_id": int(cid),
                    "stroke_component_area": int(area),
                    "stroke_contact_dist_tail": float(d0 if tail_node.node_id == n0.node_id else d1),
                    "stroke_contact_dist_head": float(d1 if head_node.node_id == n1.node_id else d0),
                    "stroke_orient_score": float(orient_score),
                    "stroke_orient_dir_score": float(orient_dir),
                    "stroke_orient_progress_span": float(orient_prog),
                })
                if arrow.confidence < cfg.stroke_graph_min_confidence:
                    continue
                # Avoid exact duplicates inside stroke candidates.
                if any(path_iou(arrow.path_xy, old.path_xy) > cfg.stroke_graph_max_path_iou and _same_node_pair(arrow, old) for old in candidates):
                    continue
                candidates.append(arrow)
                if len(candidates) >= cfg.stroke_graph_max_candidates:
                    return sorted(candidates, key=lambda a: a.confidence, reverse=True)

    return sorted(candidates, key=lambda a: a.confidence, reverse=True)



# -----------------------------------------------------------------------------
# V6 ink-bridge fallback: connect strong endpoint pairs through raw ink corridors
# -----------------------------------------------------------------------------

def line_corridor_mask(shape_hw: Tuple[int, int], p0: Sequence[float], p1: Sequence[float], width: int) -> np.ndarray:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    a = (int(round(float(p0[0]))), int(round(float(p0[1]))))
    b = (int(round(float(p1[0]))), int(round(float(p1[1]))))
    width = max(3, int(width))
    cv2.line(mask, a, b, 1, thickness=width, lineType=cv2.LINE_AA)
    draw_disk(mask, a, max(3, width // 2), value=1)
    draw_disk(mask, b, max(3, width // 2), value=1)
    return mask.astype(np.uint8)


def build_ink_bridge_support_maps(
    tail: Endpoint,
    head: Endpoint,
    support_maps: Dict[str, np.ndarray],
    cfg: LADDecoderConfig,
) -> Optional[Dict[str, np.ndarray]]:
    ink = support_maps.get("ink")
    if ink is None or float(np.max(ink)) <= 1e-6:
        return None
    base_support = support_maps["support"].astype(np.float32)
    base_walkable = support_maps["walkable"].astype(np.uint8)
    node_core = support_maps["node_core"].astype(np.uint8)
    endpoint_force = support_maps.get("endpoint_force", np.zeros_like(base_walkable, dtype=np.uint8)).astype(np.uint8)
    h, w = base_support.shape

    dist = l2(tail.refined_xy, head.refined_xy)
    width = int(np.clip(cfg.ink_bridge_width_factor * dist, cfg.ink_bridge_min_width, cfg.ink_bridge_max_width))
    corridor = line_corridor_mask((h, w), tail.refined_xy, head.refined_xy, width)

    vals = ink[corridor > 0]
    if vals.size == 0:
        return None
    # Local threshold: dark/white stroke ink becomes high after image_ink_support().
    local_thr = float(np.clip(max(cfg.ink_bridge_threshold, 0.55 * float(np.quantile(vals, 0.82))), 0.12, 0.48))
    ink_walk = ((ink >= local_thr) & (corridor > 0)).astype(np.uint8)

    # Bridge small handwriting gaps inside the corridor only. This is the key safety valve:
    # raw ink helps the decoder, but it cannot freely wander over the whole page.
    ksz = int(np.clip(round(width / 3), 3, 11))
    if ksz % 2 == 0:
        ksz += 1
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksz, ksz))
    ink_walk = cv2.morphologyEx(ink_walk, cv2.MORPH_CLOSE, k)
    ink_walk = cv2.dilate(ink_walk, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=1)
    ink_walk = (ink_walk & corridor).astype(np.uint8)

    bridge_walkable = ((base_walkable > 0) & (corridor > 0)).astype(np.uint8)
    bridge_walkable = np.maximum(bridge_walkable, ink_walk)
    bridge_walkable = np.maximum(bridge_walkable, endpoint_force)

    bridge_support = np.maximum(base_support, 0.86 * ink * corridor.astype(np.float32)).astype(np.float32)
    bridge_support = np.clip(bridge_support, 0.0, 1.0)

    return {
        "support": bridge_support,
        "walkable": bridge_walkable.astype(np.uint8),
        "node_core": node_core,
        "ink": ink.astype(np.float32),
        "endpoint_force": endpoint_force,
        "bridge_corridor": corridor,
    }


def generate_ink_bridge_candidates(
    pred: Dict[str, np.ndarray],
    support_maps: Dict[str, np.ndarray],
    tails: List[Endpoint],
    heads: List[Endpoint],
    existing_candidates: List[Arrow],
    cfg: LADDecoderConfig,
) -> List[Arrow]:
    """Recover endpoint pairs that have strong raw-ink evidence but broken learned support.

    This specifically targets cases such as a right-side DFA edge where tail/head
    heatmaps are present, but the learned shaft/skeleton support is discontinuous.
    We do not globally add raw ink to the decoder. We create a narrow corridor per
    endpoint pair and allow raw ink only inside that corridor.
    """
    if not cfg.enable_ink_bridge_candidates:
        return []

    existing_ep_pairs = {(a.tail_endpoint_id, a.head_endpoint_id) for a in existing_candidates}
    existing_node_pairs = {(a.tail_node_id, a.head_node_id) for a in existing_candidates}
    repair_cfg = replace(
        cfg,
        progress_cost_weight=cfg.ink_bridge_progress_cost_weight,
        support_cost_weight=cfg.ink_bridge_support_cost_weight,
        nearest_walkable_radius=max(cfg.nearest_walkable_radius, cfg.ink_bridge_max_width),
    )

    out: List[Arrow] = []
    for tail in tails:
        if tail.attached_node_id is None:
            continue
        for head in heads:
            if head.attached_node_id is None:
                continue
            if (tail.endpoint_id, head.endpoint_id) in existing_ep_pairs:
                continue
            same_node = tail.attached_node_id == head.attached_node_id
            if same_node and not cfg.ink_bridge_allow_self_loops:
                continue

            # Avoid trying every boring pair when the same node pair is already well represented,
            # except for self-loops and multi-edge cases where another geometric route may exist.
            if (tail.attached_node_id, head.attached_node_id) in existing_node_pairs and not same_node:
                continue

            # If the learned progress is strongly contradictory, raw ink alone is not enough.
            if not same_node and (head.progress_value + 0.08 < tail.progress_value):
                continue

            bridge_maps = build_ink_bridge_support_maps(tail, head, support_maps, cfg)
            if bridge_maps is None:
                continue
            trace = astar_trace(tail, head, pred, bridge_maps, repair_cfg)
            if trace is None or len(trace["path"]) < cfg.min_path_len:
                continue

            arrow = make_arrow_candidate(tail, head, trace, pred, bridge_maps)
            arrow.confidence *= cfg.ink_bridge_confidence_scale
            arrow.meta["source"] = "ink_bridge"
            arrow.meta["bridge_pair_already_present"] = (tail.attached_node_id, head.attached_node_id) in existing_node_pairs
            arrow.meta["bridge_corridor_width"] = int(np.clip(cfg.ink_bridge_width_factor * l2(tail.refined_xy, head.refined_xy), cfg.ink_bridge_min_width, cfg.ink_bridge_max_width))

            if arrow.confidence < cfg.ink_bridge_min_confidence:
                continue
            if arrow.support_score < cfg.ink_bridge_min_support_score:
                continue
            # Same-node candidates must actually leave the node area.
            if same_node:
                # We do not have the node bbox here, so use a conservative path length gate.
                if len(arrow.path_xy) < max(cfg.stroke_graph_min_path_len, 32):
                    continue
            out.append(arrow)
            if len(out) >= cfg.ink_bridge_max_candidates:
                return sorted(out, key=lambda a: a.confidence, reverse=True)
    return sorted(out, key=lambda a: a.confidence, reverse=True)

def build_writer_json(image_name: str, width: int, height: int, nodes: List[Node], arrows: List[Arrow]) -> Dict[str, Any]:
    return {
        "image": image_name,
        "size": {"w": int(width), "h": int(height)},
        "nodes": [
            {
                "id": n.node_id,
                "bbox": [int(v) for v in n.bbox_xywh],
                "center": {"x": float(n.center_xy[0]), "y": float(n.center_xy[1])},
                "score": float(n.score),
            }
            for n in nodes
        ],
        "arrows": [
            {
                "id": int(a.arrow_id),
                "tail_hint": {"x": float(a.tail_hint["x"]), "y": float(a.tail_hint["y"])},
                "head_hint": {"x": float(a.head_hint["x"]), "y": float(a.head_hint["y"])},
                "path": [{"x": float(x), "y": float(y)} for x, y in a.path_xy],
                "tail_node_id": a.tail_node_id,
                "head_node_id": a.head_node_id,
                "confidence": float(a.confidence),
            }
            for a in arrows
        ],
    }


def rescale_writer_json(writer: Dict[str, Any], orig_w: int, orig_h: int) -> Dict[str, Any]:
    w0 = float(writer["size"]["w"]); h0 = float(writer["size"]["h"])
    sx = float(orig_w) / max(w0, 1.0); sy = float(orig_h) / max(h0, 1.0)
    out = json.loads(json.dumps(writer))
    out["size"] = {"w": int(orig_w), "h": int(orig_h)}
    out["coordinate_space"] = "original_image"
    for n in out.get("nodes", []):
        x, y, bw, bh = n["bbox"]
        n["bbox"] = [float(x) * sx, float(y) * sy, float(bw) * sx, float(bh) * sy]
        if "center" in n:
            n["center"]["x"] *= sx
            n["center"]["y"] *= sy
    for a in out.get("arrows", []):
        for key in ["tail_hint", "head_hint"]:
            a[key]["x"] *= sx
            a[key]["y"] *= sy
        for p in a.get("path", []):
            p["x"] *= sx
            p["y"] *= sy
    return out


def build_debug_json(
    image_name: str,
    nodes: List[Node],
    heads: List[Endpoint],
    tails: List[Endpoint],
    candidates: List[Arrow],
    arrows: List[Arrow],
    cfg: LADDecoderConfig,
) -> Dict[str, Any]:
    return {
        "image": image_name,
        "decoder": "AdaptiveStrokeGraphDecoderV6",
        "config": asdict(cfg),
        "nodes": [asdict(n) for n in nodes],
        "heads": [asdict(e) for e in heads],
        "tails": [asdict(e) for e in tails],
        "candidate_arrows": [asdict(a) for a in candidates],
        "selected_arrows": [asdict(a) for a in arrows],
        "summary": {
            "num_nodes": len(nodes),
            "num_heads": len(heads),
            "num_tails": len(tails),
            "num_candidates": len(candidates),
            "num_endpoint_candidates": sum(1 for a in candidates if a.meta.get("source") not in {"stroke_graph"} and not a.meta.get("port_role_repair", False)),
            "num_port_repair_candidates": sum(1 for a in candidates if a.meta.get("port_role_repair", False)),
            "num_stroke_graph_candidates": sum(1 for a in candidates if a.meta.get("source") == "stroke_graph"),
            "num_ink_bridge_candidates": sum(1 for a in candidates if a.meta.get("source") == "ink_bridge"),
            "num_arrows": len(arrows),
            "num_stroke_graph_arrows_selected": sum(1 for a in arrows if a.meta.get("source") == "stroke_graph"),
            "num_ink_bridge_arrows_selected": sum(1 for a in arrows if a.meta.get("source") == "ink_bridge"),
        },
    }


def build_networkx_graph(nodes: List[Node], arrows: List[Arrow]) -> Optional[Any]:
    if nx is None:
        return None
    g = nx.MultiDiGraph()
    for n in nodes:
        g.add_node(n.node_id, bbox_xyxy=n.bbox_xyxy, bbox_xywh=n.bbox_xywh, center_xy=n.center_xy, score=n.score)
    for a in arrows:
        src = a.tail_node_id if a.tail_node_id is not None else f"EP::{a.tail_endpoint_id}"
        dst = a.head_node_id if a.head_node_id is not None else f"EP::{a.head_endpoint_id}"
        if src not in g:
            g.add_node(src, kind="endpoint_stub")
        if dst not in g:
            g.add_node(dst, kind="endpoint_stub")
        g.add_edge(src, dst, key=f"A{a.arrow_id}", **asdict(a))
    return g


def decode_graph_learning_aligned(
    pred: Dict[str, np.ndarray],
    decoder_img: np.ndarray,
    image_name: str,
    cfg: LADDecoderConfig,
    orig_size: Optional[Tuple[int, int]] = None,
) -> DecodeResult:
    h, w = pred["shaft"].shape

    # V5 adaptation happens inside the decoder, image by image. The original cfg
    # is still kept in the run config; the debug JSON records the adapted cfg that
    # was actually used for this image.
    cfg = adapt_config_pre_nodes(pred, cfg)
    nodes = extract_nodes(pred, cfg)
    cfg = adapt_config_post_nodes(pred, cfg, nodes)

    heads = extract_endpoints(pred, "head", cfg)
    tails = extract_endpoints(pred, "tail", cfg)
    attach_all_endpoints(heads, nodes, cfg)
    attach_all_endpoints(tails, nodes, cfg)

    support_maps = build_support_field(pred, decoder_img, nodes, heads, tails, cfg)

    endpoint_candidates = generate_arrow_candidates(pred, support_maps, tails, heads, cfg)
    first_pass_arrows = select_arrows(endpoint_candidates, cfg)

    repair_candidates = generate_port_role_repair_candidates(
        pred=pred,
        support_maps=support_maps,
        heads=heads,
        tails=tails,
        selected=first_pass_arrows,
        cfg=cfg,
    )

    # V5: add topology-first candidates from the visible/learned stroke graph.
    # These can recover edges even when no clean role-specific tail/head seed exists.
    stroke_candidates = generate_stroke_graph_candidates(
        pred=pred,
        support_maps=support_maps,
        nodes=nodes,
        cfg=cfg,
    )

    # V6: endpoint-to-endpoint raw-ink bridge fallback. This is deliberately after
    # normal/repair/stroke candidates so it only fills holes, not dominate the decoder.
    ink_bridge_candidates = generate_ink_bridge_candidates(
        pred=pred,
        support_maps=support_maps,
        tails=tails,
        heads=heads,
        existing_candidates=endpoint_candidates + repair_candidates + stroke_candidates,
        cfg=cfg,
    )

    candidates = endpoint_candidates + repair_candidates + stroke_candidates + ink_bridge_candidates
    arrows = select_arrows(candidates, cfg)

    writer = build_writer_json(image_name, w, h, nodes, arrows)
    writer["coordinate_space"] = "model"
    writer_orig = None
    if orig_size is not None:
        orig_w, orig_h = orig_size
        writer_orig = rescale_writer_json(writer, orig_w, orig_h)

    debug_json = build_debug_json(image_name, nodes, heads, tails, candidates, arrows, cfg)
    debug_maps = {
        "support": support_maps["support"],
        "walkable": support_maps["walkable"],
        "node_core": support_maps["node_core"],
        "ink": support_maps["ink"],
        "endpoint_force": support_maps["endpoint_force"],
    }
    graph = build_networkx_graph(nodes, arrows)

    if cfg.verbose:
        print(f"  decoded nodes={len(nodes)}, heads={len(heads)}, tails={len(tails)}, candidates={len(candidates)}, arrows={len(arrows)}")

    return DecodeResult(
        nodes=nodes,
        heads=heads,
        tails=tails,
        candidate_arrows=candidates,
        arrows=arrows,
        writer_style_json=writer,
        writer_style_original_json=writer_orig,
        decoder_debug_json=debug_json,
        debug_maps=debug_maps,
        nx_graph=graph,
    )


# -----------------------------------------------------------------------------
# File IO / visualization
# -----------------------------------------------------------------------------

def save_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_json(obj), f, indent=2)


def save_gray(path: Path, arr: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), to_uint8_vis(arr))


def save_heat(path: Path, arr: np.ndarray, cmap: int = cv2.COLORMAP_TURBO) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    vis = to_uint8_vis(arr)
    color = cv2.applyColorMap(vis, cmap)
    cv2.imwrite(str(path), color)


def save_rgb(path: Path, img: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    rgb = ensure_rgb_uint8(img)
    cv2.imwrite(str(path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))


def draw_cross(img: np.ndarray, x: int, y: int, color: Tuple[int, int, int], size: int = 5, thickness: int = 1) -> None:
    cv2.line(img, (x - size, y - size), (x + size, y + size), color, thickness)
    cv2.line(img, (x - size, y + size), (x + size, y - size), color, thickness)


def draw_plus(img: np.ndarray, x: int, y: int, color: Tuple[int, int, int], size: int = 5, thickness: int = 1) -> None:
    cv2.line(img, (x - size, y), (x + size, y), color, thickness)
    cv2.line(img, (x, y - size), (x, y + size), color, thickness)


def overlay_nodes(img: np.ndarray, nodes: List[Node]) -> np.ndarray:
    vis = ensure_rgb_uint8(img).copy()
    for n in nodes:
        x1, y1, x2, y2 = n.bbox_xyxy
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cx, cy = map(int, map(round, n.center_xy))
        cv2.circle(vis, (cx, cy), 3, (255, 255, 0), -1)
        cv2.putText(vis, n.node_id, (x1, max(0, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1, cv2.LINE_AA)
    return vis


def overlay_endpoints(img: np.ndarray, heads: List[Endpoint], tails: List[Endpoint]) -> np.ndarray:
    vis = ensure_rgb_uint8(img).copy()
    for ep in tails:
        x, y = map(int, map(round, ep.refined_xy))
        draw_plus(vis, x, y, (0, 255, 0), 6, 2)
        if ep.attached_node_id:
            cv2.putText(vis, f"{ep.endpoint_id}->{ep.attached_node_id}", (x + 5, y), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 0), 1, cv2.LINE_AA)
    for ep in heads:
        x, y = map(int, map(round, ep.refined_xy))
        draw_cross(vis, x, y, (255, 0, 0), 6, 2)
        if ep.attached_node_id:
            cv2.putText(vis, f"{ep.endpoint_id}->{ep.attached_node_id}", (x + 5, y), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 0, 0), 1, cv2.LINE_AA)
    return vis


def overlay_final_graph(img: np.ndarray, result: DecodeResult) -> np.ndarray:
    vis = overlay_nodes(img, result.nodes)
    for a in result.arrows:
        if len(a.path_xy) >= 2:
            pts = np.array(a.path_xy, dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(vis, [pts], isClosed=False, color=(0, 255, 0), thickness=2)
            mid = pts[len(pts) // 2, 0]
            label = f"A{a.arrow_id}: {a.tail_node_id}->{a.head_node_id} {a.confidence:.2f}"
            cv2.putText(vis, label, (int(mid[0]), int(mid[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 255, 0), 1, cv2.LINE_AA)
        tx, ty = map(int, map(round, a.tail_xy))
        hx, hy = map(int, map(round, a.head_xy))
        cv2.circle(vis, (tx, ty), 4, (0, 255, 255), -1)
        cv2.circle(vis, (hx, hy), 4, (255, 255, 0), -1)
    return vis


def save_prediction_maps(output_dir: Path, stem: str, pred: Dict[str, np.ndarray]) -> None:
    # Probability heads
    for key in ["node_center", "node_extent", "shaft", "skeleton", "head", "tail", "progress"]:
        save_gray(output_dir / f"{stem}_pred_{key}.png", pred[key])
    # Vector/regression heads
    for key in [
        "dir_x", "dir_y", "dir_mag",
        "head_offset_x", "head_offset_y", "head_offset_mag",
        "tail_offset_x", "tail_offset_y", "tail_offset_mag",
        "head_tangent_x", "head_tangent_y",
        "tail_tangent_x", "tail_tangent_y",
    ]:
        save_heat(output_dir / f"{stem}_pred_{key}.png", pred[key], cmap=cv2.COLORMAP_TURBO)


def save_decode_artifacts(output_dir: Path, stem: str, decoder_img: np.ndarray, result: DecodeResult, save_maps: bool = True) -> None:
    save_json(output_dir / f"{stem}_writer_model.json", result.writer_style_json)
    if result.writer_style_original_json is not None:
        save_json(output_dir / f"{stem}_writer_original.json", result.writer_style_original_json)
    save_json(output_dir / f"{stem}_debug.json", result.decoder_debug_json)

    if save_maps:
        save_heat(output_dir / f"{stem}_stage_support.png", result.debug_maps["support"], cmap=cv2.COLORMAP_MAGMA)
        save_gray(output_dir / f"{stem}_stage_walkable.png", result.debug_maps["walkable"])
        save_gray(output_dir / f"{stem}_stage_node_core.png", result.debug_maps["node_core"])
        save_gray(output_dir / f"{stem}_stage_endpoint_force.png", result.debug_maps["endpoint_force"])
        if float(result.debug_maps["ink"].max()) > 0:
            save_gray(output_dir / f"{stem}_stage_ink_support.png", result.debug_maps["ink"])

    save_rgb(output_dir / f"{stem}_viz_input.png", ensure_rgb_uint8(decoder_img))
    save_rgb(output_dir / f"{stem}_viz_nodes.png", overlay_nodes(decoder_img, result.nodes))
    save_rgb(output_dir / f"{stem}_viz_endpoints_refined.png", overlay_endpoints(decoder_img, result.heads, result.tails))
    save_rgb(output_dir / f"{stem}_viz_final_graph.png", overlay_final_graph(decoder_img, result))


def append_csv_row(csv_path: Path, fieldnames: List[str], row: Dict[str, Any]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not exists:
            writer.writeheader()
        writer.writerow(row)



@dataclass
class InferenceSample:
    sample_id: str
    image_path: Path
    json_path: Optional[Path] = None
    family: str = ""
    split: str = ""


def _parse_csv_list(value: Optional[str]) -> Optional[List[str]]:
    if value is None:
        return None
    value = str(value).strip()
    if not value or value.lower() in {"all", "*"}:
        return None
    return [x.strip() for x in value.split(",") if x.strip()]


def find_json_for_image(image_path: Path, json_dir: Optional[Path] = None) -> Optional[Path]:
    candidates: List[Path] = []
    if json_dir is not None:
        candidates.append(json_dir / f"{image_path.stem}.json")
    # New converted dataset convention: sibling json folder next to images.
    if image_path.parent.name == "images":
        candidates.append(image_path.parent.parent / "json" / f"{image_path.stem}.json")
    # Generic fallback: same directory.
    candidates.append(image_path.with_suffix(".json"))
    for p in candidates:
        if p.exists():
            return p
    return None


def discover_converted_dataset_samples(
    dataset_root: Path,
    families: Optional[Sequence[str]] = None,
    splits: Optional[Sequence[str]] = None,
    require_gt: bool = False,
) -> List[InferenceSample]:
    """Discover samples from <root>/<family>/<split>/{images,json}.

    The converted dataset also contains node_instance_masks, but this inference and
    graph-evaluation path intentionally ignores that folder. The JSON files are the
    annotation source used by the old ANN_DIR logic.
    """
    if not dataset_root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    fam_filter = set(families) if families else None
    split_filter = set(splits) if splits else None
    samples: List[InferenceSample] = []

    for fam_dir in sorted([p for p in dataset_root.iterdir() if p.is_dir()]):
        family = fam_dir.name
        if fam_filter is not None and family not in fam_filter:
            continue
        for split_dir in sorted([p for p in fam_dir.iterdir() if p.is_dir()]):
            split = split_dir.name
            if split_filter is not None and split not in split_filter:
                continue
            image_dir = split_dir / "images"
            json_dir = split_dir / "json"
            if not image_dir.exists():
                continue
            for image_path in find_images(image_dir):
                json_path = find_json_for_image(image_path, json_dir=json_dir)
                if require_gt and json_path is None:
                    print(f"⚠️ Missing GT JSON for {family}/{split}/{image_path.name}")
                    continue
                samples.append(
                    InferenceSample(
                        sample_id=image_path.stem,
                        image_path=image_path,
                        json_path=json_path,
                        family=family,
                        split=split,
                    )
                )

    return samples


def discover_flat_samples(
    input_dir: Path,
    gt_json_dir: Optional[Path] = None,
    require_gt: bool = False,
) -> List[InferenceSample]:
    images = find_images(input_dir)
    samples: List[InferenceSample] = []
    for image_path in images:
        json_path = find_json_for_image(image_path, gt_json_dir)
        if require_gt and json_path is None:
            print(f"⚠️ Missing GT JSON for {image_path.name}")
            continue
        samples.append(InferenceSample(sample_id=image_path.stem, image_path=image_path, json_path=json_path))
    return samples


def sample_output_dir(base_output_dir: Path, sample: InferenceSample, preserve_structure: bool = True) -> Path:
    if preserve_structure and sample.family and sample.split:
        return base_output_dir / sample.family / sample.split
    return base_output_dir


# -----------------------------------------------------------------------------
# Graph/JSON evaluation metrics
# -----------------------------------------------------------------------------

def _bbox_xyxy_from_node(node: Dict[str, Any]) -> Tuple[float, float, float, float]:
    b = node.get("bbox", [0, 0, 0, 0])
    if len(b) < 4:
        return 0.0, 0.0, 0.0, 0.0
    x, y, w, h = map(float, b[:4])
    return x, y, x + max(0.0, w) - 1.0, y + max(0.0, h) - 1.0


def _bbox_center_xy(node: Dict[str, Any]) -> Tuple[float, float]:
    if "center" in node and isinstance(node["center"], dict):
        return float(node["center"].get("x", 0.0)), float(node["center"].get("y", 0.0))
    x1, y1, x2, y2 = _bbox_xyxy_from_node(node)
    return 0.5 * (x1 + x2), 0.5 * (y1 + y2)


def _bbox_iou_float(a: Sequence[float], b: Sequence[float]) -> float:
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1 + 1.0), max(0.0, iy2 - iy1 + 1.0)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1 + 1.0) * max(0.0, ay2 - ay1 + 1.0)
    area_b = max(0.0, bx2 - bx1 + 1.0) * max(0.0, by2 - by1 + 1.0)
    return float(inter / max(area_a + area_b - inter, 1e-8))


def _safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if float(den) != 0.0 else 0.0


def _prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    p = _safe_div(tp, tp + fp)
    r = _safe_div(tp, tp + fn)
    f = _safe_div(2.0 * p * r, p + r)
    return p, r, f


def match_nodes_by_iou(
    pred_nodes: Sequence[Dict[str, Any]],
    gt_nodes: Sequence[Dict[str, Any]],
    iou_thresh: float,
) -> Tuple[Dict[str, str], List[Tuple[str, str, float]]]:
    candidates: List[Tuple[float, str, str]] = []
    for pn in pred_nodes:
        pid = str(pn.get("id"))
        pb = _bbox_xyxy_from_node(pn)
        for gn in gt_nodes:
            gid = str(gn.get("id"))
            giou = _bbox_iou_float(pb, _bbox_xyxy_from_node(gn))
            if giou >= iou_thresh:
                candidates.append((giou, pid, gid))

    pred_used, gt_used = set(), set()
    pred_to_gt: Dict[str, str] = {}
    matches: List[Tuple[str, str, float]] = []
    for giou, pid, gid in sorted(candidates, reverse=True):
        if pid in pred_used or gid in gt_used:
            continue
        pred_used.add(pid)
        gt_used.add(gid)
        pred_to_gt[pid] = gid
        matches.append((pid, gid, float(giou)))
    return pred_to_gt, matches


def _edge_pair(edge: Dict[str, Any]) -> Tuple[Optional[str], Optional[str]]:
    t = edge.get("tail_node_id")
    h = edge.get("head_node_id")
    return (str(t) if t is not None else None, str(h) if h is not None else None)


def _map_edge_pair(edge: Dict[str, Any], pred_to_gt: Dict[str, str]) -> Tuple[Optional[str], Optional[str]]:
    t, h = _edge_pair(edge)
    return (pred_to_gt.get(t) if t is not None else None, pred_to_gt.get(h) if h is not None else None)


def _counter_from_pairs(pairs: Sequence[Tuple[Optional[str], Optional[str]]]) -> Dict[Tuple[Optional[str], Optional[str]], int]:
    out: Dict[Tuple[Optional[str], Optional[str]], int] = {}
    for p in pairs:
        out[p] = out.get(p, 0) + 1
    return out


def _undir(pair: Tuple[Optional[str], Optional[str]]) -> Tuple[str, str]:
    a = "__NONE__" if pair[0] is None else str(pair[0])
    b = "__NONE__" if pair[1] is None else str(pair[1])
    return tuple(sorted([a, b]))


def _counter_intersection_count(a: Dict[Any, int], b: Dict[Any, int]) -> int:
    return int(sum(min(v, b.get(k, 0)) for k, v in a.items()))


def _point_from_hint(edge: Dict[str, Any], key: str) -> Optional[Tuple[float, float]]:
    p = edge.get(key)
    if isinstance(p, dict) and "x" in p and "y" in p:
        return float(p["x"]), float(p["y"])
    return None


def _path_points(edge: Dict[str, Any]) -> List[Tuple[float, float]]:
    pts = []
    for p in edge.get("path", []) or []:
        if isinstance(p, dict) and "x" in p and "y" in p:
            pts.append((float(p["x"]), float(p["y"])))
    if len(pts) >= 2:
        return pts
    tail = _point_from_hint(edge, "tail_hint")
    head = _point_from_hint(edge, "head_hint")
    return [tail, head] if tail is not None and head is not None else []


def _rasterize_polyline(points: Sequence[Tuple[float, float]], w: int, h: int, tol_px: int) -> np.ndarray:
    mask = np.zeros((max(1, int(h)), max(1, int(w))), dtype=np.uint8)
    if len(points) == 0:
        return mask
    arr = np.array([[int(round(x)), int(round(y))] for x, y in points], dtype=np.int32)
    arr[:, 0] = np.clip(arr[:, 0], 0, mask.shape[1] - 1)
    arr[:, 1] = np.clip(arr[:, 1], 0, mask.shape[0] - 1)
    if len(arr) == 1:
        cv2.circle(mask, tuple(arr[0]), max(1, int(tol_px)), 1, -1)
    else:
        cv2.polylines(mask, [arr.reshape(-1, 1, 2)], False, 1, thickness=max(1, int(tol_px)))
    return mask


def _mask_prf_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> Dict[str, float]:
    p = pred_mask.astype(bool)
    g = gt_mask.astype(bool)
    tp = int(np.logical_and(p, g).sum())
    fp = int(np.logical_and(p, np.logical_not(g)).sum())
    fn = int(np.logical_and(np.logical_not(p), g).sum())
    prec, rec, f1 = _prf(tp, fp, fn)
    iou = _safe_div(tp, tp + fp + fn)
    return {"precision": prec, "recall": rec, "f1": f1, "iou": iou}


def _rescale_writer_to_size(writer: Dict[str, Any], target_w: int, target_h: int) -> Dict[str, Any]:
    cur = writer.get("size", {}) or {}
    cur_w = float(cur.get("w", target_w))
    cur_h = float(cur.get("h", target_h))
    if abs(cur_w - target_w) < 1e-6 and abs(cur_h - target_h) < 1e-6:
        return writer
    sx = float(target_w) / max(cur_w, 1.0)
    sy = float(target_h) / max(cur_h, 1.0)
    out = json.loads(json.dumps(writer))
    out["size"] = {"w": int(target_w), "h": int(target_h)}
    for n in out.get("nodes", []):
        if "bbox" in n and len(n["bbox"]) >= 4:
            n["bbox"] = [float(n["bbox"][0]) * sx, float(n["bbox"][1]) * sy, float(n["bbox"][2]) * sx, float(n["bbox"][3]) * sy]
        if "center" in n:
            n["center"]["x"] *= sx
            n["center"]["y"] *= sy
    for e in out.get("arrows", []):
        for key in ["tail_hint", "head_hint"]:
            if isinstance(e.get(key), dict):
                e[key]["x"] *= sx
                e[key]["y"] *= sy
        for p in e.get("path", []) or []:
            p["x"] *= sx
            p["y"] *= sy
    return out


def _pair_matched_edges_for_geometry(
    pred_edges: Sequence[Dict[str, Any]],
    gt_edges: Sequence[Dict[str, Any]],
    pred_pairs: Sequence[Tuple[Optional[str], Optional[str]]],
    gt_pairs: Sequence[Tuple[Optional[str], Optional[str]]],
) -> List[Tuple[Dict[str, Any], Dict[str, Any]]]:
    gt_by_pair: Dict[Tuple[Optional[str], Optional[str]], List[int]] = {}
    for i, pair in enumerate(gt_pairs):
        gt_by_pair.setdefault(pair, []).append(i)

    matched: List[Tuple[Dict[str, Any], Dict[str, Any]]] = []
    used_gt = set()
    for pred_edge, pair in zip(pred_edges, pred_pairs):
        candidate_ids = [i for i in gt_by_pair.get(pair, []) if i not in used_gt]
        if not candidate_ids:
            continue
        pt = _point_from_hint(pred_edge, "tail_hint")
        ph = _point_from_hint(pred_edge, "head_hint")
        best_i, best_d = None, float("inf")
        for gi in candidate_ids:
            gt_edge = gt_edges[gi]
            gt = _point_from_hint(gt_edge, "tail_hint")
            gh = _point_from_hint(gt_edge, "head_hint")
            d = 0.0
            c = 0
            if pt is not None and gt is not None:
                d += l2(pt, gt); c += 1
            if ph is not None and gh is not None:
                d += l2(ph, gh); c += 1
            d = d / max(1, c)
            if d < best_d:
                best_d, best_i = d, gi
        if best_i is not None:
            used_gt.add(best_i)
            matched.append((pred_edge, gt_edges[best_i]))
    return matched


def evaluate_graph_prediction(
    pred_writer: Dict[str, Any],
    gt_writer: Dict[str, Any],
    node_iou_thresh: float = 0.50,
    path_tolerance_px: int = 3,
) -> Dict[str, Any]:
    gt_size = gt_writer.get("size", {}) or {}
    gt_w = int(gt_size.get("w", pred_writer.get("size", {}).get("w", 1)))
    gt_h = int(gt_size.get("h", pred_writer.get("size", {}).get("h", 1)))
    pred_writer = _rescale_writer_to_size(pred_writer, gt_w, gt_h)

    pred_nodes = pred_writer.get("nodes", []) or []
    gt_nodes = gt_writer.get("nodes", []) or []
    pred_edges = pred_writer.get("arrows", []) or []
    gt_edges = gt_writer.get("arrows", []) or []

    pred_to_gt, node_matches = match_nodes_by_iou(pred_nodes, gt_nodes, node_iou_thresh)
    node_tp = len(node_matches)
    node_fp = len(pred_nodes) - node_tp
    node_fn = len(gt_nodes) - node_tp
    node_precision, node_recall, node_f1 = _prf(node_tp, node_fp, node_fn)

    matched_ious = [m[2] for m in node_matches]
    center_errors: List[float] = []
    gt_by_id = {str(n.get("id")): n for n in gt_nodes}
    pred_by_id = {str(n.get("id")): n for n in pred_nodes}
    for pid, gid, _ in node_matches:
        pc = _bbox_center_xy(pred_by_id[pid])
        gc = _bbox_center_xy(gt_by_id[gid])
        center_errors.append(l2(pc, gc))

    gt_pairs = [_edge_pair(e) for e in gt_edges]
    pred_pairs_mapped = [_map_edge_pair(e, pred_to_gt) for e in pred_edges]
    gt_counter = _counter_from_pairs(gt_pairs)
    pred_counter = _counter_from_pairs(pred_pairs_mapped)
    edge_tp = _counter_intersection_count(pred_counter, gt_counter)
    edge_fp = len(pred_edges) - edge_tp
    edge_fn = len(gt_edges) - edge_tp
    edge_precision, edge_recall, edge_f1 = _prf(edge_tp, edge_fp, edge_fn)

    gt_undir_counter = _counter_from_pairs([_undir(p) for p in gt_pairs])
    pred_undir_counter = _counter_from_pairs([_undir(p) for p in pred_pairs_mapped])
    undir_tp = _counter_intersection_count(pred_undir_counter, gt_undir_counter)
    undir_fp = len(pred_edges) - undir_tp
    undir_fn = len(gt_edges) - undir_tp
    undir_precision, undir_recall, undir_f1 = _prf(undir_tp, undir_fp, undir_fn)
    direction_proxy = _safe_div(edge_tp, undir_tp)

    gt_loop_counter = {k: v for k, v in gt_counter.items() if k[0] is not None and k[0] == k[1]}
    pred_loop_counter = {k: v for k, v in pred_counter.items() if k[0] is not None and k[0] == k[1]}
    gt_loop_total = int(sum(gt_loop_counter.values()))
    matched_loop = _counter_intersection_count(pred_loop_counter, gt_loop_counter)
    loop_recall = _safe_div(matched_loop, gt_loop_total)

    out_degree: Dict[Optional[str], int] = {}
    for t, h in gt_pairs:
        if t is not None:
            out_degree[t] = out_degree.get(t, 0) + 1
    branch_sources = {nid for nid, deg in out_degree.items() if deg >= 2}
    branch_gt_pairs = [p for p in gt_pairs if p[0] in branch_sources]
    branch_counter = _counter_from_pairs(branch_gt_pairs)
    branch_total = int(sum(branch_counter.values()))
    branch_matched = _counter_intersection_count(pred_counter, branch_counter)
    decision_branch_recall = _safe_div(branch_matched, branch_total)

    matched_edge_pairs = _pair_matched_edges_for_geometry(pred_edges, gt_edges, pred_pairs_mapped, gt_pairs)
    path_ious: List[float] = []
    path_f1s: List[float] = []
    tail_errors: List[float] = []
    head_errors: List[float] = []
    for pred_edge, gt_edge in matched_edge_pairs:
        pred_path = _path_points(pred_edge)
        gt_path = _path_points(gt_edge)
        if pred_path and gt_path:
            pm = _rasterize_polyline(pred_path, gt_w, gt_h, path_tolerance_px)
            gm = _rasterize_polyline(gt_path, gt_w, gt_h, path_tolerance_px)
            mr = _mask_prf_iou(pm, gm)
            path_ious.append(mr["iou"])
            path_f1s.append(mr["f1"])
        pt = _point_from_hint(pred_edge, "tail_hint")
        gt = _point_from_hint(gt_edge, "tail_hint")
        ph = _point_from_hint(pred_edge, "head_hint")
        gh = _point_from_hint(gt_edge, "head_hint")
        if pt is not None and gt is not None:
            tail_errors.append(l2(pt, gt))
        if ph is not None and gh is not None:
            head_errors.append(l2(ph, gh))

    edit_cost = node_fp + node_fn + edge_fp + edge_fn
    approx_nged = _safe_div(edit_cost, max(1, len(gt_nodes) + len(gt_edges)))

    return {
        "gt_nodes": len(gt_nodes),
        "pred_nodes": len(pred_nodes),
        "node_tp": node_tp,
        "node_fp": node_fp,
        "node_fn": node_fn,
        "node_precision": node_precision,
        "node_recall": node_recall,
        "node_f1": node_f1,
        "node_mean_iou": float(np.mean(matched_ious)) if matched_ious else 0.0,
        "node_center_mae_px": float(np.mean(center_errors)) if center_errors else 0.0,
        "gt_edges": len(gt_edges),
        "pred_edges": len(pred_edges),
        "edge_tp": edge_tp,
        "edge_fp": edge_fp,
        "edge_fn": edge_fn,
        "edge_precision": edge_precision,
        "edge_recall": edge_recall,
        "edge_f1": edge_f1,
        "undirected_edge_precision": undir_precision,
        "undirected_edge_recall": undir_recall,
        "undirected_edge_f1": undir_f1,
        "direction_proxy": direction_proxy,
        "gt_self_loops": gt_loop_total,
        "matched_self_loops": matched_loop,
        "loop_back_recall": loop_recall,
        "gt_branch_edges": branch_total,
        "matched_branch_edges": branch_matched,
        "decision_branch_recall": decision_branch_recall,
        "matched_edge_geometry_count": len(matched_edge_pairs),
        "path_mean_iou": float(np.mean(path_ious)) if path_ious else 0.0,
        "path_mean_f1": float(np.mean(path_f1s)) if path_f1s else 0.0,
        "tail_endpoint_mae_px": float(np.mean(tail_errors)) if tail_errors else 0.0,
        "head_endpoint_mae_px": float(np.mean(head_errors)) if head_errors else 0.0,
        "approx_nged": approx_nged,
        "structural_edit_cost": edit_cost,
    }


def aggregate_metric_rows(rows: Sequence[Dict[str, Any]], group_keys: Sequence[str] = ()) -> List[Dict[str, Any]]:
    groups: Dict[Tuple[Any, ...], List[Dict[str, Any]]] = {}
    for r in rows:
        key = tuple(r.get(k, "") for k in group_keys)
        groups.setdefault(key, []).append(r)

    out: List[Dict[str, Any]] = []
    for key, items in sorted(groups.items(), key=lambda kv: kv[0]):
        s = lambda name: sum(float(x.get(name, 0.0)) for x in items)
        node_tp, node_fp, node_fn = int(s("node_tp")), int(s("node_fp")), int(s("node_fn"))
        edge_tp, edge_fp, edge_fn = int(s("edge_tp")), int(s("edge_fp")), int(s("edge_fn"))
        node_p, node_r, node_f = _prf(node_tp, node_fp, node_fn)
        edge_p, edge_r, edge_f = _prf(edge_tp, edge_fp, edge_fn)
        undir_tp = sum(float(x.get("undirected_edge_recall", 0.0)) * float(x.get("gt_edges", 0.0)) for x in items)
        # For direction, the most stable aggregate is matched directed edges divided by matched undirected edges.
        # Reconstruct matched undirected count from direction proxy where available.
        undir_match = 0.0
        for x in items:
            dp = float(x.get("direction_proxy", 0.0))
            if dp > 0:
                undir_match += float(x.get("edge_tp", 0.0)) / dp
        row = {"num_images": len(items)}
        for k, v in zip(group_keys, key):
            row[k] = v
        row.update({
            "gt_nodes": int(s("gt_nodes")),
            "pred_nodes": int(s("pred_nodes")),
            "node_precision": node_p,
            "node_recall": node_r,
            "node_f1": node_f,
            "node_mean_iou": float(np.mean([float(x.get("node_mean_iou", 0.0)) for x in items])),
            "node_center_mae_px": float(np.mean([float(x.get("node_center_mae_px", 0.0)) for x in items])),
            "gt_edges": int(s("gt_edges")),
            "pred_edges": int(s("pred_edges")),
            "edge_precision": edge_p,
            "edge_recall": edge_r,
            "edge_f1": edge_f,
            "direction_proxy": _safe_div(edge_tp, undir_match),
            "gt_self_loops": int(s("gt_self_loops")),
            "matched_self_loops": int(s("matched_self_loops")),
            "loop_back_recall": _safe_div(s("matched_self_loops"), s("gt_self_loops")),
            "gt_branch_edges": int(s("gt_branch_edges")),
            "matched_branch_edges": int(s("matched_branch_edges")),
            "decision_branch_recall": _safe_div(s("matched_branch_edges"), s("gt_branch_edges")),
            "path_mean_iou": float(np.mean([float(x.get("path_mean_iou", 0.0)) for x in items])),
            "path_mean_f1": float(np.mean([float(x.get("path_mean_f1", 0.0)) for x in items])),
            "tail_endpoint_mae_px": float(np.mean([float(x.get("tail_endpoint_mae_px", 0.0)) for x in items])),
            "head_endpoint_mae_px": float(np.mean([float(x.get("head_endpoint_mae_px", 0.0)) for x in items])),
            "approx_nged": _safe_div(s("structural_edit_cost"), max(1.0, s("gt_nodes") + s("gt_edges"))),
            "structural_edit_cost": int(s("structural_edit_cost")),
        })
        out.append(row)
    return out


def write_csv_rows(csv_path: Path, rows: Sequence[Dict[str, Any]]) -> None:
    if not rows:
        return
    fieldnames: List[str] = []
    for r in rows:
        for k in r.keys():
            if k not in fieldnames:
                fieldnames.append(k)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)


# -----------------------------------------------------------------------------
# Image loading / batch runner
# -----------------------------------------------------------------------------

def find_images(input_dir: Path) -> List[Path]:
    return sorted([p for p in input_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])


def load_image_for_model(
    image_path: Path,
    target_w: int,
    target_h: int,
    image_mode: str,
    normalize_image: bool,
    use_otsu: bool,
) -> Tuple[np.ndarray, np.ndarray, torch.Tensor]:
    original_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if original_bgr is None:
        raise ValueError(f"Failed to read image: {image_path}")

    if image_mode.lower() == "grayscale":
        gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
        if use_otsu:
            _, resized = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        arr = resized.astype(np.float32)
        if normalize_image:
            arr /= 255.0
        tensor = torch.from_numpy(arr[None, None, :, :].astype(np.float32))
        decoder_img = resized
    elif image_mode.lower() == "rgb":
        rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
        arr = resized.astype(np.float32)
        if normalize_image:
            arr /= 255.0
        tensor = torch.from_numpy(np.transpose(arr, (2, 0, 1))[None].astype(np.float32))
        decoder_img = resized
    else:
        raise ValueError(f"Unsupported image_mode={image_mode!r}")

    return original_bgr, decoder_img, tensor


def build_decoder_cfg(args: argparse.Namespace) -> LADDecoderConfig:
    return LADDecoderConfig(
        node_extent_thresh=args.node_extent_thresh,
        node_center_thresh=args.node_center_thresh,
        min_node_area=args.min_node_area,
        head_thresh=args.head_thresh,
        tail_thresh=args.tail_thresh,
        shaft_thresh=args.shaft_thresh,
        skeleton_thresh=args.skeleton_thresh,
        support_thresh=args.support_thresh,
        endpoint_radius=args.endpoint_radius,
        node_attach_max_dist=args.node_attach_max_dist,
        max_pair_dist=args.max_pair_dist,
        max_expansions=args.max_expansions,
        min_arrow_confidence=args.min_arrow_confidence,
        allow_endpoint_reuse=args.allow_endpoint_reuse,
        branch_aware_endpoint_reuse=args.branch_aware_endpoint_reuse,
        branch_min_opposite_endpoint_dist=args.branch_min_opposite_endpoint_dist,
        branch_max_path_iou=args.branch_max_path_iou,
        branch_require_different_node_pair=args.branch_require_different_node_pair,
        keep_unattached_arrows=args.keep_unattached_arrows,
        use_image_ink_support=args.use_image_ink_support,
        enable_port_role_repair=args.enable_port_role_repair,
        port_repair_score_scale=args.port_repair_score_scale,
        port_repair_confidence_scale=args.port_repair_confidence_scale,
        port_repair_min_confidence=args.port_repair_min_confidence,
        port_repair_progress_cost_weight=args.port_repair_progress_cost_weight,
        port_repair_max_path_iou=args.port_repair_max_path_iou,
        port_repair_min_support_score=args.port_repair_min_support_score,
        port_repair_min_shaft_score=args.port_repair_min_shaft_score,
        port_repair_max_candidates=args.port_repair_max_candidates,
        auto_adapt=args.auto_adapt,
        enable_stroke_graph_candidates=args.enable_stroke_graph_candidates,
        stroke_graph_min_component_area=args.stroke_graph_min_component_area,
        stroke_graph_min_path_len=args.stroke_graph_min_path_len,
        stroke_graph_min_confidence=args.stroke_graph_min_confidence,
        stroke_graph_max_path_iou=args.stroke_graph_max_path_iou,
        stroke_graph_max_components=args.stroke_graph_max_components,
        stroke_graph_max_candidates=args.stroke_graph_max_candidates,
        stroke_graph_skip_self_loops=args.stroke_graph_skip_self_loops,
        enable_ink_bridge_candidates=args.enable_ink_bridge_candidates,
        ink_bridge_min_confidence=args.ink_bridge_min_confidence,
        ink_bridge_confidence_scale=args.ink_bridge_confidence_scale,
        ink_bridge_min_support_score=args.ink_bridge_min_support_score,
        ink_bridge_min_width=args.ink_bridge_min_width,
        ink_bridge_max_width=args.ink_bridge_max_width,
        ink_bridge_width_factor=args.ink_bridge_width_factor,
        ink_bridge_threshold=args.ink_bridge_threshold,
        ink_bridge_max_candidates=args.ink_bridge_max_candidates,
        ink_bridge_progress_cost_weight=args.ink_bridge_progress_cost_weight,
        ink_bridge_support_cost_weight=args.ink_bridge_support_cost_weight,
        ink_bridge_allow_self_loops=args.ink_bridge_allow_self_loops,
        verbose=args.verbose,
    )


@torch.no_grad()
def run_inference(args: argparse.Namespace) -> None:
    dataset_root = Path(args.dataset_root) if args.dataset_root else None
    input_dir = Path(args.input_dir)
    output_dir = Path(args.output_dir)
    checkpoint_path = Path(args.checkpoint)
    output_dir.mkdir(parents=True, exist_ok=True)

    if dataset_root is not None and not dataset_root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")
    if dataset_root is None and not input_dir.exists():
        raise FileNotFoundError(f"Input dir not found: {input_dir}")
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    families = _parse_csv_list(args.families)
    splits = _parse_csv_list(args.splits)
    require_gt = bool(args.evaluate)

    if dataset_root is not None:
        samples = discover_converted_dataset_samples(
            dataset_root=dataset_root,
            families=families,
            splits=splits,
            require_gt=require_gt,
        )
    else:
        gt_json_dir = Path(args.gt_json_dir) if args.gt_json_dir else None
        samples = discover_flat_samples(input_dir=input_dir, gt_json_dir=gt_json_dir, require_gt=require_gt)

    if not samples:
        print("No samples found.")
        return

    device = torch.device(args.device if args.device else ("cuda" if torch.cuda.is_available() else "cpu"))
    print(f"Loading checkpoint: {checkpoint_path}")
    print(f"Device: {device}")
    print(f"Found {len(samples)} samples")
    if dataset_root is not None:
        print(f"Dataset root: {dataset_root}")
        print(f"Families: {families or 'all'} | Splits: {splits or 'all'}")

    model, load_info = load_checkpoint_model(
        checkpoint_path=checkpoint_path,
        device=device,
        image_mode=args.image_mode,
        base_ch=args.base_ch,
        norm_type=args.norm_type,
        strict=not args.allow_partial_checkpoint,
    )

    cfg = build_decoder_cfg(args)
    save_json(output_dir / "run_config.json", vars(args))
    save_json(output_dir / "checkpoint_load_info.json", load_info)
    save_json(output_dir / "decoder_config.json", asdict(cfg))

    success_csv = output_dir / "success_log.csv"
    failure_csv = output_dir / "failure_log.csv"
    metric_rows: List[Dict[str, Any]] = []
    n_ok = 0
    n_fail = 0

    for idx, sample in enumerate(samples, start=1):
        image_path = sample.image_path
        stem = image_path.stem
        out_dir = sample_output_dir(output_dir, sample, preserve_structure=args.preserve_dataset_structure)
        out_dir.mkdir(parents=True, exist_ok=True)
        rel_label = f"{sample.family}/{sample.split}/{image_path.name}" if sample.family else image_path.name
        print(f"[{idx}/{len(samples)}] {rel_label}")
        try:
            original_bgr, decoder_img, image_tensor = load_image_for_model(
                image_path=image_path,
                target_w=args.target_w,
                target_h=args.target_h,
                image_mode=args.image_mode,
                normalize_image=args.normalize_image,
                use_otsu=args.use_otsu,
            )
            image_tensor = image_tensor.to(device)
            outputs = model(image_tensor)
            required = {
                "node_center", "node_extent", "shaft", "skeleton", "head", "tail", "dir",
                "progress", "head_offset", "tail_offset", "head_tangent", "tail_tangent",
            }
            missing = sorted(required - set(outputs.keys()))
            if missing:
                raise RuntimeError(f"Model did not return required heads: {missing}")

            pred = outputs_to_pred_maps(outputs)
            if args.save_pred_maps:
                save_prediction_maps(out_dir, stem, pred)

            orig_h, orig_w = original_bgr.shape[:2]
            result = decode_graph_learning_aligned(
                pred=pred,
                decoder_img=decoder_img,
                image_name=image_path.name,
                cfg=cfg,
                orig_size=(orig_w, orig_h) if args.save_original_space_json else None,
            )

            save_decode_artifacts(out_dir, stem, decoder_img, result, save_maps=args.save_stage_maps)
            save_json(out_dir / f"{stem}_meta.json", {
                "image_name": image_path.name,
                "image_path": str(image_path),
                "json_path": str(sample.json_path) if sample.json_path is not None else "",
                "family": sample.family,
                "split": sample.split,
                "orig_w": int(orig_w),
                "orig_h": int(orig_h),
                "model_w": int(args.target_w),
                "model_h": int(args.target_h),
                "image_mode": args.image_mode,
            })

            metric_row: Dict[str, Any] = {}
            if args.evaluate and sample.json_path is not None:
                with open(sample.json_path, "r", encoding="utf-8") as f:
                    gt_writer = json.load(f)
                pred_writer_for_eval = result.writer_style_original_json or result.writer_style_json
                metric_row = evaluate_graph_prediction(
                    pred_writer=pred_writer_for_eval,
                    gt_writer=gt_writer,
                    node_iou_thresh=args.eval_node_iou_thresh,
                    path_tolerance_px=args.eval_path_tolerance_px,
                )
                metric_row.update({
                    "image_name": image_path.name,
                    "sample_id": sample.sample_id,
                    "family": sample.family,
                    "split": sample.split,
                    "json_path": str(sample.json_path),
                })
                metric_rows.append(metric_row)
                save_json(out_dir / f"{stem}_metrics.json", metric_row)

            success_row = {
                "image_name": image_path.name,
                "sample_id": sample.sample_id,
                "family": sample.family,
                "split": sample.split,
                "num_nodes": len(result.nodes),
                "num_heads": len(result.heads),
                "num_tails": len(result.tails),
                "num_candidates": len(result.candidate_arrows),
                "num_arrows": len(result.arrows),
                "writer_model_json": str((out_dir / f"{stem}_writer_model.json").resolve()),
                "writer_original_json": str((out_dir / f"{stem}_writer_original.json").resolve()) if result.writer_style_original_json is not None else "",
                "debug_json": str((out_dir / f"{stem}_debug.json").resolve()),
                "final_viz": str((out_dir / f"{stem}_viz_final_graph.png").resolve()),
            }
            if metric_row:
                for k in ["node_f1", "edge_f1", "direction_proxy", "loop_back_recall", "decision_branch_recall", "path_mean_f1", "approx_nged"]:
                    success_row[k] = metric_row.get(k, "")
            append_csv_row(
                success_csv,
                list(success_row.keys()),
                success_row,
            )
            n_ok += 1
        except Exception as e:
            tb = traceback.format_exc(limit=60)
            print(f"  FAILED: {rel_label} -> {e}")
            append_csv_row(
                failure_csv,
                ["image_name", "sample_id", "family", "split", "error", "traceback"],
                {
                    "image_name": image_path.name,
                    "sample_id": sample.sample_id,
                    "family": sample.family,
                    "split": sample.split,
                    "error": str(e),
                    "traceback": tb,
                },
            )
            with open(out_dir / f"{stem}_traceback.txt", "w", encoding="utf-8") as f:
                f.write(tb)
            n_fail += 1

    if metric_rows:
        write_csv_rows(output_dir / "metrics_per_image.csv", metric_rows)
        overall = aggregate_metric_rows(metric_rows, group_keys=[])[0]
        by_family = aggregate_metric_rows(metric_rows, group_keys=["family"])
        by_split = aggregate_metric_rows(metric_rows, group_keys=["split"])
        by_family_split = aggregate_metric_rows(metric_rows, group_keys=["family", "split"])
        write_csv_rows(output_dir / "metrics_by_family.csv", by_family)
        write_csv_rows(output_dir / "metrics_by_split.csv", by_split)
        write_csv_rows(output_dir / "metrics_by_family_split.csv", by_family_split)
        save_json(output_dir / "metrics_summary.json", {
            "overall": overall,
            "by_family": by_family,
            "by_split": by_split,
            "by_family_split": by_family_split,
            "metric_notes": {
                "node_f1": "Node detection after greedy bbox-IoU matching.",
                "edge_f1": "Directed graph edge F1 after mapping predicted nodes to matched GT node ids. Multi-edges are counted as multisets.",
                "direction_proxy": "Directed matches divided by undirected matches; measures whether connected node pairs have the correct orientation.",
                "loop_back_recall": "Recall over GT self-loop edges.",
                "decision_branch_recall": "Recall over GT outgoing edges from nodes with out-degree >= 2.",
                "path_mean_f1/path_mean_iou": "Raster overlap of matched arrow paths with tolerance thickness.",
                "approx_nged": "Normalized structural edit proxy: (node FP+FN+edge FP+FN)/(GT nodes+GT edges).",
            },
        })
        print("\nEvaluation summary:")
        print(json.dumps(overall, indent=2))

    print("\nDone.")
    print(f"Output dir: {output_dir}")
    print(f"Success={n_ok}, Failed={n_fail}")


# -----------------------------------------------------------------------------
# CLI
# -----------------------------------------------------------------------------

def parse_args(argv=None) -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Learning-Aligned inference for upgraded DiagramUNet graph extraction")

    p.add_argument("--input-dir", type=str, default=INPUT_DIR,
                   help="Flat image directory fallback. Ignored when --dataset-root is supplied.")
    p.add_argument("--dataset-root", type=str, default=None,
                   help="Converted dataset root with <family>/<split>/images and <family>/<split>/json folders.")
    p.add_argument("--families", type=str, default="all",
                   help="Comma-separated family filter, e.g. fa,fca,fcb. Use 'all' for every family.")
    p.add_argument("--splits", type=str, default="all",
                   help="Comma-separated split filter, e.g. train,val,test. Use 'all' for every split.")
    p.add_argument("--gt-json-dir", type=str, default=None,
                   help="Optional flat GT JSON directory when using --input-dir instead of --dataset-root.")
    p.add_argument("--evaluate", action="store_true", default=False,
                   help="Compare exported graph JSON against GT JSON and write metrics CSV/JSON summaries.")
    p.add_argument("--eval-node-iou-thresh", type=float, default=0.50,
                   help="IoU threshold for matching predicted nodes to GT nodes.")
    p.add_argument("--eval-path-tolerance-px", type=int, default=3,
                   help="Polyline thickness/tolerance for path overlap metrics in GT coordinate space.")
    p.add_argument("--preserve-dataset-structure", action="store_true", default=True,
                   help="Save outputs under <output>/<family>/<split>/ for converted datasets.")
    p.add_argument("--flat-output", dest="preserve_dataset_structure", action="store_false",
                   help="Put every image output directly in --output-dir.")
    p.add_argument("--checkpoint", type=str, default=CHECKPOINT_PATH)
    p.add_argument("--output-dir", type=str, default=OUTPUT_DIR)

    p.add_argument("--image-mode", type=str, default="grayscale", choices=["grayscale", "rgb"])
    p.add_argument("--normalize-image", action="store_true", default=True)
    p.add_argument("--no-normalize-image", dest="normalize_image", action="store_false")
    p.add_argument("--use-otsu", action="store_true", default=False)
    p.add_argument("--target-w", type=int, default=512)
    p.add_argument("--target-h", type=int, default=512)
    p.add_argument("--base-ch", type=int, default=0, help="0 = infer from checkpoint")
    p.add_argument("--norm-type", type=str, default="auto", choices=["auto", "group", "batch"])
    p.add_argument("--device", type=str, default=None)
    p.add_argument("--allow-partial-checkpoint", action="store_true", default=False,
                   help="Use strict=False while loading. Not recommended for paper runs.")

    # Decoder thresholds
    p.add_argument("--node-center-thresh", type=float, default=0.25)
    p.add_argument("--node-extent-thresh", type=float, default=0.20)
    p.add_argument("--min-node-area", type=int, default=35)
    p.add_argument("--head-thresh", type=float, default=0.25)
    p.add_argument("--tail-thresh", type=float, default=0.25)
    p.add_argument("--shaft-thresh", type=float, default=0.20)
    p.add_argument("--skeleton-thresh", type=float, default=0.20)
    p.add_argument("--support-thresh", type=float, default=0.20)
    p.add_argument("--endpoint-radius", type=float, default=8.0,
                   help="Must match ENDPOINT_RADIUS used during training because offsets are normalized by this radius.")
    p.add_argument("--node-attach-max-dist", type=float, default=28.0)
    p.add_argument("--max-pair-dist", type=float, default=1e9)
    p.add_argument("--max-expansions", type=int, default=90000)
    p.add_argument("--min-arrow-confidence", type=float, default=0.32)
    p.add_argument("--allow-endpoint-reuse", action="store_true", default=False,
                   help="Broadly allow shared endpoint seeds. Useful for debugging; may overgenerate.")
    p.add_argument("--disable-branch-aware-endpoint-reuse", dest="branch_aware_endpoint_reuse",
                   action="store_false", default=True,
                   help="Disable safer fan-out/fan-in reuse of shared endpoints.")
    p.add_argument("--branch-min-opposite-endpoint-dist", type=float, default=16.0)
    p.add_argument("--branch-max-path-iou", type=float, default=0.62)
    p.add_argument("--allow-same-node-pair-branch", dest="branch_require_different_node_pair",
                   action="store_false", default=True,
                   help="Allow multiple branches between the same node pair. Usually keep this off.")
    p.add_argument("--keep-unattached-arrows", action="store_true", default=True)
    p.add_argument("--drop-unattached-arrows", dest="keep_unattached_arrows", action="store_false")
    p.add_argument("--use-image-ink-support", action="store_true", default=False,
                   help="Optional support from the resized input ink. Keep off for pure model-output decoding.")

    p.add_argument("--disable-port-role-repair", dest="enable_port_role_repair",
                   action="store_false", default=True,
                   help="Disable second-pass virtual endpoint repair for crowded node ports.")
    p.add_argument("--port-repair-score-scale", type=float, default=0.62)
    p.add_argument("--port-repair-confidence-scale", type=float, default=0.82)
    p.add_argument("--port-repair-min-confidence", type=float, default=0.46)
    p.add_argument("--port-repair-progress-cost-weight", type=float, default=0.20)
    p.add_argument("--port-repair-max-path-iou", type=float, default=0.52)
    p.add_argument("--port-repair-min-support-score", type=float, default=0.72)
    p.add_argument("--port-repair-min-shaft-score", type=float, default=0.72)
    p.add_argument("--port-repair-max-candidates", type=int, default=12)

    # V5 adaptive/stroke-graph decoder options
    p.add_argument("--disable-auto-adapt", dest="auto_adapt", action="store_false", default=True,
                   help="Disable per-image adaptive thresholds/distances and use raw CLI values.")
    p.add_argument("--disable-stroke-graph-candidates", dest="enable_stroke_graph_candidates",
                   action="store_false", default=True,
                   help="Disable V5 stroke-topology fallback candidates.")
    p.add_argument("--stroke-graph-min-component-area", type=int, default=18)
    p.add_argument("--stroke-graph-min-path-len", type=int, default=12)
    p.add_argument("--stroke-graph-min-confidence", type=float, default=0.34)
    p.add_argument("--stroke-graph-max-path-iou", type=float, default=0.78)
    p.add_argument("--stroke-graph-max-components", type=int, default=96)
    p.add_argument("--stroke-graph-max-candidates", type=int, default=80)
    p.add_argument("--disable-stroke-graph-self-loops", dest="stroke_graph_skip_self_loops",
                   action="store_true", default=False,
                   help="Disable V6 stroke-graph self-loop candidates.")

    # V6 ink-bridge options
    p.add_argument("--disable-ink-bridge-candidates", dest="enable_ink_bridge_candidates",
                   action="store_false", default=True,
                   help="Disable V6 raw-ink corridor fallback candidates.")
    p.add_argument("--ink-bridge-min-confidence", type=float, default=0.40)
    p.add_argument("--ink-bridge-confidence-scale", type=float, default=0.88)
    p.add_argument("--ink-bridge-min-support-score", type=float, default=0.42)
    p.add_argument("--ink-bridge-min-width", type=int, default=10)
    p.add_argument("--ink-bridge-max-width", type=int, default=32)
    p.add_argument("--ink-bridge-width-factor", type=float, default=0.10)
    p.add_argument("--ink-bridge-threshold", type=float, default=0.18)
    p.add_argument("--ink-bridge-max-candidates", type=int, default=80)
    p.add_argument("--ink-bridge-progress-cost-weight", type=float, default=0.28)
    p.add_argument("--ink-bridge-support-cost-weight", type=float, default=1.35)
    p.add_argument("--disable-ink-bridge-self-loops", dest="ink_bridge_allow_self_loops",
                   action="store_false", default=True,
                   help="Disable self-loop generation through the ink-bridge fallback.")

    # Saving
    p.add_argument("--save-pred-maps", action="store_true", default=True)
    p.add_argument("--no-save-pred-maps", dest="save_pred_maps", action="store_false")
    p.add_argument("--save-stage-maps", action="store_true", default=True)
    p.add_argument("--no-save-stage-maps", dest="save_stage_maps", action="store_false")
    p.add_argument("--save-original-space-json", action="store_true", default=True)
    p.add_argument("--no-save-original-space-json", dest="save_original_space_json", action="store_false")
    p.add_argument("--verbose", action="store_true", default=False)
    return p.parse_args(argv)


## 2. Plain ResNet34 node-region model

This is the only node-segmentation architecture used by the released pipeline.
The model proposes node regions, while the graph-evidence branch remains
responsible for connector topology. Node geometry is incorporated only after
Pass II has fixed the graph interpretation.


In [ ]:
# ============================================================
# PLAIN RESNET34 U-NET NODE SEGMENTER
# ============================================================

try:
    import torchvision.models as tvm
except Exception as exc:
    tvm = None
    print("torchvision import failed:", exc)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def node_conv_bn_relu(in_ch, out_ch, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class NodeUpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv1 = node_conv_bn_relu(out_ch + skip_ch, out_ch)
        self.conv2 = node_conv_bn_relu(out_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.size(-1) != skip.size(-1) or x.size(-2) != skip.size(-2):
            diff_y = skip.size(-2) - x.size(-2)
            diff_x = skip.size(-1) - x.size(-1)
            x = F.pad(
                x,
                [
                    diff_x // 2,
                    diff_x - diff_x // 2,
                    diff_y // 2,
                    diff_y - diff_y // 2,
                ],
            )
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class ResNet34NodeUNet(nn.Module):
    """Plain ResNet34 encoder with a U-Net-style node-region decoder."""

    def __init__(self, out_ch=1, pretrained=False):
        super().__init__()
        if tvm is None:
            raise ImportError("torchvision is required for ResNet34NodeUNet.")

        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        r34 = tvm.resnet34(weights=weights)

        self.conv1 = r34.conv1
        self.bn1 = r34.bn1
        self.relu = r34.relu
        self.maxpool = r34.maxpool

        self.layer1 = r34.layer1
        self.layer2 = r34.layer2
        self.layer3 = r34.layer3
        self.layer4 = r34.layer4

        self.bot = node_conv_bn_relu(512, 512)
        self.up4 = NodeUpBlock(512, 256, 256)
        self.up3 = NodeUpBlock(256, 128, 128)
        self.up2 = NodeUpBlock(128, 64, 64)
        self.up1 = NodeUpBlock(64, 64, 64)
        self.head = nn.Conv2d(64, out_ch, kernel_size=1)

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        bottleneck = self.bot(x4)
        d4 = self.up4(bottleneck, x3)
        d3 = self.up3(d4, x2)
        d2 = self.up2(d3, x1)
        d1 = self.up1(d2, x0)

        logits = self.head(d1)
        return F.interpolate(
            logits,
            size=(x.size(2), x.size(3)),
            mode="bilinear",
            align_corners=False,
        )


def _extract_node_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("model", "model_state_dict", "state_dict"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
        return checkpoint
    raise TypeError("Unsupported node checkpoint format.")


def _strip_data_parallel_prefix(state):
    return {
        key[7:] if str(key).startswith("module.") else key: value
        for key, value in state.items()
    }


def load_node_segmentation_model(checkpoint_path, device):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"NODE_MODEL_CHECKPOINT was not found: {checkpoint_path}"
        )

    checkpoint = torch.load(str(checkpoint_path), map_location="cpu")
    state = _strip_data_parallel_prefix(_extract_node_state_dict(checkpoint))


    model = ResNet34NodeUNet(out_ch=1, pretrained=False)
    model.load_state_dict(state, strict=True)
    model = model.to(device).eval()

    return model, {
        "checkpoint": str(checkpoint_path),
        "architecture": "plain_resnet34_unet",
        "strict": True,
        "state_keys": len(state),
    }


def load_node_model_input(image_path, img_size=320):
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"Could not read image for node model: {image_path}")

    orig_h, orig_w = gray.shape[:2]
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    rgb = cv2.resize(rgb, (int(img_size), int(img_size)), interpolation=cv2.INTER_LINEAR)

    array = rgb.astype(np.float32) / 255.0
    array = (
        array - IMAGENET_MEAN[None, None, :]
    ) / IMAGENET_STD[None, None, :]
    tensor = torch.from_numpy(
        np.transpose(array, (2, 0, 1))
    ).unsqueeze(0).float()

    return gray, tensor, orig_w, orig_h


@torch.no_grad()
def predict_node_probability(model, image_path, device, img_size=320):
    gray, tensor, orig_w, orig_h = load_node_model_input(
        image_path, img_size=img_size
    )
    logits = model(tensor.to(device))
    probability = torch.sigmoid(logits)[0, 0].detach().cpu().numpy().astype(np.float32)
    probability = cv2.resize(
        probability,
        (orig_w, orig_h),
        interpolation=cv2.INTER_LINEAR,
    )
    return gray, probability


def extract_node_boxes(probability, threshold=0.46, min_area=35, close_kernel=3):
    binary = (probability >= float(threshold)).astype(np.uint8)

    if close_kernel and int(close_kernel) > 1:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (int(close_kernel), int(close_kernel)),
        )
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    nodes = []
    for component_id in range(1, count):
        x, y, w, h, area = stats[component_id].tolist()
        if int(area) < int(min_area):
            continue

        component = labels == component_id
        nodes.append({
            "id": f"N{len(nodes)}",
            "bbox": [int(x), int(y), int(w), int(h)],
            "area": int(area),
            "score": float(probability[component].mean()),
        })

    return nodes, binary

## 3. Evaluation protocol

The released evaluator keeps detection, geometry, and topology metrics
separate:

- **Node F1:** tolerant one-to-one node detection matching.
- **Mask IoU:** mean overlap of one-to-one matched node regions only.
- **Soft-clDice:** tolerance-aware centerline agreement on matched directed
  arrow paths only.
- **Endpoint L2:** endpoint error on matched directed edges.
- **Link F1:** directed graph-link detection.
- **nGED:** structural edit cost divided by
  `max(|V_pred|, |V_gt|) + max(|E_pred|, |E_gt|)`.
- **Loop-back recall** and **decision-branch recall:** structural subset recall.

Node F1 and Link F1 account for unmatched objects. Mask IoU and Soft-clDice
therefore measure geometry without duplicating the detection penalty.


In [ ]:
# ============================================================
# PAPER EVALUATION METRICS
# ============================================================

from collections import Counter, defaultdict
try:
    from skimage.morphology import skeletonize as _sk_skeletonize
except Exception:
    _sk_skeletonize = None


def metric_tolerance_px(width, height, ratio=METRIC_TOLERANCE_RATIO):
    return int(max(2, round(float(ratio) * max(int(width), int(height)))))


def bbox_xywh_to_xyxy(b):
    x, y, w, h = [float(v) for v in b]
    return x, y, x + w, y + h


def bbox_iou_xywh(a, b):
    ax1, ay1, ax2, ay2 = bbox_xywh_to_xyxy(a)
    bx1, by1, bx2, by2 = bbox_xywh_to_xyxy(b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    aa = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    bb = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return float(inter / max(aa + bb - inter, 1e-9))


def bbox_center_xy(node):
    x, y, w, h = [float(v) for v in node.get("bbox", [0, 0, 0, 0])]
    return (x + w / 2.0, y + h / 2.0)


def tolerant_match_nodes(pred_nodes, gt_nodes, width, height, iou_accept=NODE_IOU_ACCEPT):
    tol = metric_tolerance_px(width, height)
    candidates = []
    for pi, pn in enumerate(pred_nodes):
        pc = bbox_center_xy(pn)
        for gi, gn in enumerate(gt_nodes):
            gc = bbox_center_xy(gn)
            iou = bbox_iou_xywh(pn.get("bbox", [0, 0, 0, 0]), gn.get("bbox", [0, 0, 0, 0]))
            cd = l2(pc, gc)
            gt_diag = math.hypot(float(gn.get("bbox", [0, 0, 1, 1])[2]), float(gn.get("bbox", [0, 0, 1, 1])[3]))
            center_tol = max(float(tol), 0.18 * gt_diag)
            if iou >= iou_accept or cd <= center_tol:
                # lower score is better
                score = (cd / max(center_tol, 1e-6)) + (1.0 - iou)
                candidates.append((score, -iou, cd, pi, gi, iou))
    candidates.sort()
    used_p, used_g = set(), set()
    matches = []
    pred_to_gt = {}
    for score, neg_iou, cd, pi, gi, iou in candidates:
        if pi in used_p or gi in used_g:
            continue
        used_p.add(pi); used_g.add(gi)
        pid = str(pred_nodes[pi].get("id", f"P{pi}"))
        gid = str(gt_nodes[gi].get("id", f"G{gi}"))
        pred_to_gt[pid] = gid
        matches.append((pid, gid, float(iou), float(cd)))
    return pred_to_gt, matches


def edge_pair(edge):
    return (edge.get("tail_node_id"), edge.get("head_node_id"))


def mapped_edge_pair(edge, pred_to_gt):
    return (pred_to_gt.get(str(edge.get("tail_node_id"))), pred_to_gt.get(str(edge.get("head_node_id"))))


def pair_counter(pairs):
    return Counter(tuple(p) for p in pairs)


def counter_intersection_count(a, b):
    return int(sum(min(v, b.get(k, 0)) for k, v in a.items()))


def path_points(edge):
    pts = edge.get("path", []) or []
    out = []
    for p in pts:
        if isinstance(p, dict) and "x" in p and "y" in p:
            out.append((float(p["x"]), float(p["y"])))
        elif isinstance(p, (list, tuple)) and len(p) >= 2:
            out.append((float(p[0]), float(p[1])))
    return out


def point_from_edge(edge, plain, hint):
    p = edge.get(plain)
    if isinstance(p, dict) and "x" in p and "y" in p:
        return (float(p["x"]), float(p["y"]))
    p = edge.get(hint)
    if isinstance(p, dict) and "x" in p and "y" in p:
        return (float(p["x"]), float(p["y"]))
    return None


def rasterize_writer_graph(writer, width, height, path_thickness=3, include_nodes=True, include_edges=True):
    mask = np.zeros((int(height), int(width)), dtype=np.uint8)
    if include_nodes:
        for node in writer.get("nodes", []) or []:
            x, y, w, h = [int(round(float(v))) for v in node.get("bbox", [0, 0, 0, 0])]
            x0, y0 = max(0, x), max(0, y)
            x1, y1 = min(int(width) - 1, x + w), min(int(height) - 1, y + h)
            if x1 >= x0 and y1 >= y0:
                cv2.rectangle(mask, (x0, y0), (x1, y1), 1, -1)
    if include_edges:
        for edge in writer.get("arrows", []) or []:
            pts = path_points(edge)
            if len(pts) >= 2:
                arr = np.array([[int(round(x)), int(round(y))] for x, y in pts], dtype=np.int32)
                cv2.polylines(mask, [arr], False, 1, int(path_thickness), lineType=cv2.LINE_AA)
            else:
                tail = point_from_edge(edge, "tail", "tail_hint")
                head = point_from_edge(edge, "head", "head_hint")
                if tail is not None and head is not None:
                    arr = np.array([[int(round(tail[0])), int(round(tail[1]))],
                                    [int(round(head[0])), int(round(head[1]))]], dtype=np.int32)
                    cv2.polylines(mask, [arr], False, 1, int(path_thickness), lineType=cv2.LINE_AA)
    return (mask > 0).astype(np.uint8)


def tolerant_mask_iou(pred_mask, gt_mask, tol_px):
    if tol_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol_px + 1, 2 * tol_px + 1))
        # symmetric relaxed IoU: allow small offsets without exploding false positives too badly
        p = cv2.dilate(pred_mask.astype(np.uint8), k, iterations=1)
        g = cv2.dilate(gt_mask.astype(np.uint8), k, iterations=1)
    else:
        p, g = pred_mask.astype(np.uint8), gt_mask.astype(np.uint8)
    inter = int(np.logical_and(p > 0, g > 0).sum())
    union = int(np.logical_or(p > 0, g > 0).sum())
    return _safe_div(inter, union)


def cldice_score(pred_mask, gt_mask, tol_px):
    pred = (pred_mask > 0).astype(np.uint8)
    gt = (gt_mask > 0).astype(np.uint8)
    if _sk_skeletonize is None:
        return 0.0
    sk_p = _sk_skeletonize(pred > 0).astype(np.uint8)
    sk_g = _sk_skeletonize(gt > 0).astype(np.uint8)
    if tol_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol_px + 1, 2 * tol_px + 1))
        gt_relaxed = cv2.dilate(gt, k, iterations=1)
        pred_relaxed = cv2.dilate(pred, k, iterations=1)
    else:
        gt_relaxed, pred_relaxed = gt, pred
    tprec = _safe_div(int(np.logical_and(sk_p > 0, gt_relaxed > 0).sum()), int((sk_p > 0).sum()))
    tsens = _safe_div(int(np.logical_and(sk_g > 0, pred_relaxed > 0).sum()), int((sk_g > 0).sum()))
    return _safe_div(2.0 * tprec * tsens, tprec + tsens)


def pair_matched_edges_for_geometry(pred_edges, gt_edges, pred_pairs_mapped, gt_pairs):
    gt_by_pair = defaultdict(list)
    for gi, pair in enumerate(gt_pairs):
        gt_by_pair[pair].append(gi)
    matched = []
    used_gt = set()
    for pred_edge, pair in zip(pred_edges, pred_pairs_mapped):
        choices = [i for i in gt_by_pair.get(pair, []) if i not in used_gt]
        if not choices:
            continue
        pt = point_from_edge(pred_edge, "tail", "tail_hint")
        ph = point_from_edge(pred_edge, "head", "head_hint")
        best_i, best_d = None, float("inf")
        for gi in choices:
            gt_edge = gt_edges[gi]
            gt = point_from_edge(gt_edge, "tail", "tail_hint")
            gh = point_from_edge(gt_edge, "head", "head_hint")
            d, c = 0.0, 0
            if pt is not None and gt is not None:
                d += l2(pt, gt); c += 1
            if ph is not None and gh is not None:
                d += l2(ph, gh); c += 1
            d = d / max(1, c)
            if d < best_d:
                best_d, best_i = d, gi
        if best_i is not None:
            used_gt.add(best_i)
            matched.append((pred_edge, gt_edges[best_i], best_d))
    return matched


def evaluate_graph_prediction_tolerant(pred_writer, gt_writer):
    gt_size = gt_writer.get("size", {}) or {}
    gt_w = int(gt_size.get("w", pred_writer.get("size", {}).get("w", 1)))
    gt_h = int(gt_size.get("h", pred_writer.get("size", {}).get("h", 1)))
    tol = metric_tolerance_px(gt_w, gt_h)
    pred_writer = _rescale_writer_to_size(pred_writer, gt_w, gt_h)

    pred_nodes = pred_writer.get("nodes", []) or []
    gt_nodes = gt_writer.get("nodes", []) or []
    pred_edges = pred_writer.get("arrows", []) or []
    gt_edges = gt_writer.get("arrows", []) or []

    pred_to_gt, node_matches = tolerant_match_nodes(pred_nodes, gt_nodes, gt_w, gt_h)
    node_tp = len(node_matches)
    node_fp = len(pred_nodes) - node_tp
    node_fn = len(gt_nodes) - node_tp
    node_precision, node_recall, node_f1 = _prf(node_tp, node_fp, node_fn)

    gt_pairs = [edge_pair(e) for e in gt_edges]
    pred_pairs_mapped = [mapped_edge_pair(e, pred_to_gt) for e in pred_edges]
    gt_counter = pair_counter(gt_pairs)
    pred_counter = pair_counter(pred_pairs_mapped)

    edge_tp = counter_intersection_count(pred_counter, gt_counter)
    edge_fp = len(pred_edges) - edge_tp
    edge_fn = len(gt_edges) - edge_tp
    edge_precision, edge_recall, edge_f1 = _prf(edge_tp, edge_fp, edge_fn)

    # Link_F1 is a paper-friendly alias for directed edge F1.
    link_precision, link_recall, link_f1 = edge_precision, edge_recall, edge_f1

    # endpoint error on matched directed edges only
    endpoint_errors = []
    path_ious = []
    path_f1s = []
    path_cldices = []
    matched_geom = pair_matched_edges_for_geometry(pred_edges, gt_edges, pred_pairs_mapped, gt_pairs)
    for pe, ge, _ in matched_geom:
        for plain, hint in [("tail", "tail_hint"), ("head", "head_hint")]:
            pp = point_from_edge(pe, plain, hint)
            gp = point_from_edge(ge, plain, hint)
            if pp is not None and gp is not None:
                endpoint_errors.append(l2(pp, gp))
        pred_path = path_points(pe)
        gt_path = path_points(ge)

        # Use endpoint-to-endpoint fallback lines only when an edge has no routed path.
        if len(pred_path) < 2:
            pred_tail = point_from_edge(pe, "tail", "tail_hint")
            pred_head = point_from_edge(pe, "head", "head_hint")
            pred_path = [pred_tail, pred_head] if pred_tail is not None and pred_head is not None else []
        if len(gt_path) < 2:
            gt_tail = point_from_edge(ge, "tail", "tail_hint")
            gt_head = point_from_edge(ge, "head", "head_hint")
            gt_path = [gt_tail, gt_head] if gt_tail is not None and gt_head is not None else []

        if len(pred_path) >= 2 and len(gt_path) >= 2:
            # Thick relaxed masks retain the historical path-overlap diagnostics.
            pm = np.zeros((gt_h, gt_w), dtype=np.uint8)
            gm = np.zeros((gt_h, gt_w), dtype=np.uint8)
            cv2.polylines(
                pm,
                [np.array([[int(round(x)), int(round(y))] for x, y in pred_path], dtype=np.int32)],
                False,
                1,
                max(PATH_BASE_TOLERANCE_PX, tol),
                lineType=cv2.LINE_AA,
            )
            cv2.polylines(
                gm,
                [np.array([[int(round(x)), int(round(y))] for x, y in gt_path], dtype=np.int32)],
                False,
                1,
                max(PATH_BASE_TOLERANCE_PX, tol),
                lineType=cv2.LINE_AA,
            )
            inter = int(np.logical_and(pm > 0, gm > 0).sum())
            pcount = int((pm > 0).sum())
            gcount = int((gm > 0).sum())
            prec = _safe_div(inter, pcount)
            rec = _safe_div(inter, gcount)
            path_f1s.append(_safe_div(2 * prec * rec, prec + rec))
            path_ious.append(
                _safe_div(inter, int(np.logical_or(pm > 0, gm > 0).sum()))
            )

            # Soft-clDice is isolated to matched arrow centerlines only.
            pred_centerline = np.zeros((gt_h, gt_w), dtype=np.uint8)
            gt_centerline = np.zeros((gt_h, gt_w), dtype=np.uint8)
            cv2.polylines(
                pred_centerline,
                [np.array([[int(round(x)), int(round(y))] for x, y in pred_path], dtype=np.int32)],
                False,
                1,
                max(1, PATH_BASE_TOLERANCE_PX),
                lineType=cv2.LINE_AA,
            )
            cv2.polylines(
                gt_centerline,
                [np.array([[int(round(x)), int(round(y))] for x, y in gt_path], dtype=np.int32)],
                False,
                1,
                max(1, PATH_BASE_TOLERANCE_PX),
                lineType=cv2.LINE_AA,
            )
            path_cldices.append(
                cldice_score(pred_centerline, gt_centerline, tol_px=tol)
            )

    # Mask IoU is isolated to one-to-one matched node regions only.
    node_ious = [m[2] for m in node_matches]
    node_center_errors = [m[3] for m in node_matches]
    mask_iou = float(np.mean(node_ious)) if node_ious else 0.0

    # Detection errors remain represented by Link F1; this geometry score is
    # averaged only across matched directed connectors.
    cldice = float(np.mean(path_cldices)) if path_cldices else 0.0

    gt_loop_counter = Counter(k for k in gt_pairs if k[0] is not None and k[0] == k[1])
    pred_loop_counter = Counter(k for k in pred_pairs_mapped if k[0] is not None and k[0] == k[1])
    matched_loop = counter_intersection_count(pred_loop_counter, gt_loop_counter)

    out_degree = defaultdict(int)
    for t, h in gt_pairs:
        if t is not None:
            out_degree[t] += 1
    branch_sources = {nid for nid, deg in out_degree.items() if deg >= 2}
    branch_counter = Counter(p for p in gt_pairs if p[0] in branch_sources)
    branch_matched = counter_intersection_count(pred_counter, branch_counter)

    # nGED normalization used in this project:
    # GED / (max(|V_pred|, |V_gt|) + max(|E_pred|, |E_gt|)).
    # This normalizes vertices and edges independently so dense graphs do not
    # eclipse sparse graphs through a denominator based only on one side.
    edit_cost = node_fp + node_fn + edge_fp + edge_fn
    normalizer = max(1, max(len(pred_nodes), len(gt_nodes)) + max(len(pred_edges), len(gt_edges)))
    nged = _safe_div(edit_cost, normalizer)

    return {
        "gt_nodes": len(gt_nodes),
        "pred_nodes": len(pred_nodes),
        "node_tp": node_tp,
        "node_fp": node_fp,
        "node_fn": node_fn,
        "node_precision": node_precision,
        "node_recall": node_recall,
        "node_f1": node_f1,
        "node_mean_iou": float(np.mean(node_ious)) if node_ious else 0.0,
        "node_center_mae_px": float(np.mean(node_center_errors)) if node_center_errors else 0.0,

        "gt_edges": len(gt_edges),
        "pred_edges": len(pred_edges),
        "edge_tp": edge_tp,
        "edge_fp": edge_fp,
        "edge_fn": edge_fn,
        "edge_precision": edge_precision,
        "edge_recall": edge_recall,
        "edge_f1": edge_f1,

        "link_precision": link_precision,
        "link_recall": link_recall,
        "link_f1": link_f1,

        "mask_iou": mask_iou,
        "node_mask_iou": mask_iou,
        "cldice": cldice,
        "soft_cldice": cldice,
        "arrow_soft_cldice": cldice,
        "endpoint_l2": float(np.mean(endpoint_errors)) if endpoint_errors else 0.0,

        "path_mean_iou": float(np.mean(path_ious)) if path_ious else 0.0,
        "path_mean_f1": float(np.mean(path_f1s)) if path_f1s else 0.0,

        "gt_self_loops": int(sum(gt_loop_counter.values())),
        "matched_self_loops": int(matched_loop),
        "loop_back_recall": _safe_div(matched_loop, sum(gt_loop_counter.values())),

        "gt_branch_edges": int(sum(branch_counter.values())),
        "matched_branch_edges": int(branch_matched),
        "decision_branch_recall": _safe_div(branch_matched, sum(branch_counter.values())),

        "nged": nged,
        "nGED": nged,
        "approx_nged": nged,
        "structural_edit_cost": edit_cost,
        "metric_tolerance_px": tol,
    }


def aggregate_metric_rows_tolerant(rows, group_keys=()):
    groups = defaultdict(list)
    for r in rows:
        groups[tuple(r.get(k, "") for k in group_keys)].append(r)

    out = []
    for key, items in sorted(groups.items(), key=lambda kv: kv[0]):
        s = lambda name: sum(float(x.get(name, 0.0)) for x in items)
        node_p, node_r, node_f = _prf(int(s("node_tp")), int(s("node_fp")), int(s("node_fn")))
        edge_p, edge_r, edge_f = _prf(int(s("edge_tp")), int(s("edge_fp")), int(s("edge_fn")))

        row = {"num_images": len(items)}
        for k, v in zip(group_keys, key):
            row[k] = v

        row.update({
            "gt_nodes": int(s("gt_nodes")),
            "pred_nodes": int(s("pred_nodes")),
            "node_precision": node_p,
            "node_recall": node_r,
            "node_f1": node_f,
            "mean_node_f1": node_f,

            "mean_mask_iou": float(np.mean([float(x.get("mask_iou", 0.0)) for x in items])),
            "mean_cldice": float(np.mean([float(x.get("cldice", 0.0)) for x in items])),
            "mean_soft_cldice": float(np.mean([float(x.get("soft_cldice", x.get("cldice", 0.0))) for x in items])),

            "gt_edges": int(s("gt_edges")),
            "pred_edges": int(s("pred_edges")),
            "edge_precision": edge_p,
            "edge_recall": edge_r,
            "edge_f1": edge_f,
            "link_f1": edge_f,
            "mean_link_f1": edge_f,

            "mean_nged": float(np.mean([float(x.get("nged", 0.0)) for x in items])),
            "mean_nGED": float(np.mean([float(x.get("nged", 0.0)) for x in items])),
            "mean_endpoint_l2": float(np.mean([float(x.get("endpoint_l2", 0.0)) for x in items])),

            "loop_back_recall": _safe_div(s("matched_self_loops"), s("gt_self_loops")),
            "decision_branch_recall": _safe_div(s("matched_branch_edges"), s("gt_branch_edges")),
            "path_mean_iou": float(np.mean([float(x.get("path_mean_iou", 0.0)) for x in items])),
            "path_mean_f1": float(np.mean([float(x.get("path_mean_f1", 0.0)) for x in items])),
            "avg_metric_tolerance_px": float(np.mean([float(x.get("metric_tolerance_px", 0.0)) for x in items])),
        })
        out.append(row)
    return out

## 4. Checkpoint, visualization, and node-refinement helpers

These utilities load both checkpoints strictly, preserve graph-model node IDs,
and replace node-box geometry only when ResNet34 proposals can be matched
reliably.


In [ ]:
def save_json_local(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_json(data), f, indent=2, ensure_ascii=False)


def write_csv_rows_local(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    # stable union of keys
    keys = []
    seen = set()
    for r in rows:
        for k in r.keys():
            if k not in seen:
                keys.append(k); seen.add(k)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)


def get_example_ids(samples, max_n=3, policy="first"):
    samples = list(samples)
    if max_n <= 0:
        return set()
    if policy == "random":
        rng = random.Random(RANDOM_SEED)
        selected = rng.sample(samples, min(max_n, len(samples)))
    else:
        selected = samples[:max_n]
    return {(s.family, s.split, s.sample_id) for s in selected}


def draw_overlay_from_writer(image_path, writer, out_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        # fallback to declared writer size
        size = writer.get("size", {"w": 1024, "h": 1024})
        img = np.zeros((int(size.get("h", 1024)), int(size.get("w", 1024)), 3), dtype=np.uint8)
    size = writer.get("size", {})
    w = int(size.get("w", img.shape[1]))
    h = int(size.get("h", img.shape[0]))
    if (img.shape[1], img.shape[0]) != (w, h):
        img = cv2.resize(img, (w, h), interpolation=cv2.INTER_LINEAR)

    for node in writer.get("nodes", []) or []:
        x, y, bw, bh = [int(round(float(v))) for v in node.get("bbox", [0, 0, 0, 0])]
        cv2.rectangle(img, (x, y), (x + bw, y + bh), (0, 255, 255), 2)
        cv2.putText(img, str(node.get("id", "")), (x, max(0, y - 3)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)

    for edge in writer.get("arrows", []) or []:
        pts = path_points(edge)
        if len(pts) >= 2:
            arr = np.array([[int(round(x)), int(round(y))] for x, y in pts], dtype=np.int32)
            cv2.polylines(img, [arr], False, (255, 255, 0), 2, lineType=cv2.LINE_AA)
        tail = point_from_edge(edge, "tail", "tail_hint")
        head = point_from_edge(edge, "head", "head_hint")
        if tail is not None:
            cv2.drawMarker(img, (int(round(tail[0])), int(round(tail[1]))), (255, 0, 255),
                           cv2.MARKER_CROSS, 12, 2)
        if head is not None:
            cv2.drawMarker(img, (int(round(head[0])), int(round(head[1]))), (0, 255, 0),
                           cv2.MARKER_CROSS, 12, 2)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), img)


def make_decoder_args(device_str=None):
    class Args:
        pass
    args = Args()
    args.image_mode = IMAGE_MODE
    args.normalize_image = NORMALIZE_IMAGE
    args.use_otsu = bool(APPLY_OTSU_THRESHOLD)
    args.target_w = int(TARGET_W)
    args.target_h = int(TARGET_H)
    args.base_ch = int(GRAPH_BASE_CHANNELS)
    args.norm_type = NORM_TYPE
    args.allow_partial_checkpoint = False
    args.checkpoint = GRAPH_MODEL_CHECKPOINT
    args.device = device_str
    args.save_original_space_json = True
    args.save_stage_maps = False

    # decoder defaults copied from parse_args defaults
    default_args = parse_args([])
    for k, v in vars(default_args).items():
        if not hasattr(args, k):
            setattr(args, k, v)
    args.image_mode = IMAGE_MODE
    args.normalize_image = NORMALIZE_IMAGE
    args.use_otsu = bool(APPLY_OTSU_THRESHOLD)
    args.target_w = int(TARGET_W)
    args.target_h = int(TARGET_H)
    args.base_ch = int(GRAPH_BASE_CHANNELS)
    args.norm_type = NORM_TYPE
    args.checkpoint = GRAPH_MODEL_CHECKPOINT
    args.device = device_str
    return args


def nearest_node_id_by_boundary(point_xy, nodes):
    if not nodes:
        return None
    px, py = point_xy
    best_id, best_d = None, float("inf")
    for n in nodes:
        x, y, w, h = [float(v) for v in n.get("bbox", [0, 0, 0, 0])]
        left, top, right, bottom = x, y, x + w, y + h
        dx = max(left - px, 0.0, px - right)
        dy = max(top - py, 0.0, py - bottom)
        if dx > 0 or dy > 0:
            d = math.hypot(dx, dy)
        else:
            d = min(abs(px - left), abs(px - right), abs(py - top), abs(py - bottom))
        if d < best_d:
            best_d, best_id = d, n.get("id")
    return best_id


def load_assembler_checkpoint_safe(checkpoint_path, device, image_mode, base_ch=0, norm_type="auto", strict=True):
    """
    Same intent as load_checkpoint_model, but checkpoint is deserialized on CPU first
    and then model is moved to the selected device. This is safer for notebook
    parallel execution and avoids CUDA deserialization races.
    """
    t0 = time.time()
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] torch.load CPU start on {device}: {checkpoint_path}")
    ckpt = torch.load(str(checkpoint_path), map_location="cpu")
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] torch.load CPU done on {device} in {time.time() - t0:.2f}s")

    t1 = time.time()
    state = strip_module_prefix(extract_state_dict(ckpt))
    inferred = infer_arch_from_state(state)

    ckpt_base_ch = inferred.get("base_ch")
    ckpt_in_channels = inferred.get("in_channels")
    ckpt_norm_type = inferred.get("norm_type") or "group"

    if base_ch <= 0:
        if ckpt_base_ch is None:
            raise ValueError("Could not infer base_ch from checkpoint. Pass GRAPH_BASE_CHANNELS explicitly.")
        base_ch = ckpt_base_ch

    requested_in_channels = 1 if image_mode.lower() == "grayscale" else 3
    in_channels = ckpt_in_channels if ckpt_in_channels is not None else requested_in_channels
    if in_channels != requested_in_channels:
        raise ValueError(
            f"Checkpoint expects in_channels={in_channels}, but IMAGE_MODE={image_mode!r} gives "
            f"{requested_in_channels}. Use the same image mode used during training."
        )

    if norm_type == "auto":
        norm_type = ckpt_norm_type

    model = DiagramUNet(in_channels=in_channels, base_ch=base_ch, norm_type=norm_type)
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] built DiagramUNet on CPU in {time.time() - t1:.2f}s")

    load_info = {
        "checkpoint_path": str(checkpoint_path),
        "base_ch": base_ch,
        "in_channels": in_channels,
        "norm_type": norm_type,
        "strict": strict,
        "inferred": inferred,
        "checkpoint_top_level_keys": list(ckpt.keys()) if isinstance(ckpt, dict) else [],
        "loaded_via": "cpu_then_to_device",
    }

    t2 = time.time()
    if strict:
        model.load_state_dict(state, strict=True)
        load_info["missing_keys"] = []
        load_info["unexpected_keys"] = []
    else:
        missing, unexpected = model.load_state_dict(state, strict=False)
        load_info["missing_keys"] = list(missing)
        load_info["unexpected_keys"] = list(unexpected)
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] load_state_dict done on {device} in {time.time() - t2:.2f}s")

    t3 = time.time()
    model = model.to(device)
    if torch.cuda.is_available() and str(device).startswith("cuda"):
        torch.cuda.synchronize(device)
    model.eval()
    if PRINT_TIMING_DIAGNOSTICS:
        print(f"[asm-load] moved to {device} in {time.time() - t3:.2f}s; total={time.time() - t0:.2f}s")
    return model, load_info


def _writer_size(writer, fallback=(1024, 1024)):
    size = writer.get("size", {}) or {}
    return int(size.get("w", fallback[0])), int(size.get("h", fallback[1]))


def match_resnet_nodes_to_graph_nodes(graph_nodes, resnet_nodes, width, height):
    """
    Returns a one-to-one mapping from graph-model nodes to ResNet34 nodes.
    Matching uses relaxed center distance plus light IoU.
    """
    tol = max(2.0, float(NODE_REFINEMENT_CENTER_TOLERANCE_RATIO) * max(int(width), int(height)))
    candidates = []
    for vi, vn in enumerate(graph_nodes):
        vc = bbox_center_xy(vn)
        vdiag = math.hypot(float(vn.get("bbox", [0, 0, 1, 1])[2]), float(vn.get("bbox", [0, 0, 1, 1])[3]))
        center_tol = max(tol, 0.25 * vdiag)
        for ei, en in enumerate(resnet_nodes):
            ec = bbox_center_xy(en)
            iou = bbox_iou_xywh(vn.get("bbox", [0,0,0,0]), en.get("bbox", [0,0,0,0]))
            cd = l2(vc, ec)
            if iou >= float(NODE_REFINEMENT_IOU_ACCEPT) or cd <= center_tol:
                score = (cd / max(center_tol, 1e-6)) + (1.0 - iou)
                candidates.append((score, -iou, cd, vi, ei))
    candidates.sort()
    used_v, used_e = set(), set()
    mapping = {}
    for score, neg_iou, cd, vi, ei in candidates:
        if vi in used_v or ei in used_e:
            continue
        used_v.add(vi)
        used_e.add(ei)
        mapping[vi] = ei
    return mapping


def build_refined_node_writer(graph_writer, resnet_nodes):
    """
    Topology-preserving node-refinement writer:
    - Preserve graph-model node IDs.
    - Preserve arrows, endpoint attachments, and routed paths.
    - Replace node-box geometry with matched ResNet34 proposals when reliable.
    This improves node geometry without altering the selected graph topology.
    """
    writer = deepcopy(graph_writer)
    graph_nodes = graph_writer.get("nodes", []) or []
    w, h = _writer_size(graph_writer)

    mapping = match_resnet_nodes_to_graph_nodes(graph_nodes, resnet_nodes, w, h)
    refined_nodes = []

    for vi, vn in enumerate(graph_nodes):
        hn = deepcopy(vn)
        if vi in mapping:
            en = resnet_nodes[mapping[vi]]
            hn["bbox"] = [int(round(v)) for v in en.get("bbox", hn.get("bbox", [0,0,0,0]))]
            hn["refinement_source"] = "resnet34_matched"
            hn["node_seg_score"] = float(en.get("score", 0.0))
            hn["node_seg_area"] = int(en.get("area", 0))
        else:
            hn["refinement_source"] = "graph_model_unmatched"
        hn["id"] = vn.get("id", f"N{vi}")
        refined_nodes.append(hn)

    if not NODE_REFINEMENT_KEEP_UNMATCHED_GRAPH_NODES:
        refined_nodes = [n for n in refined_nodes if n.get("refinement_source") == "resnet34_matched"]

    writer["nodes"] = refined_nodes
    writer["arrows"] = deepcopy(graph_writer.get("arrows", []) or [])

    if NODE_REFINEMENT_REPAIR_EDGE_IDS:
        for arrow in writer.get("arrows", []) or []:
            tail = point_from_edge(arrow, "tail", "tail_hint")
            head = point_from_edge(arrow, "head", "head_hint")
            if tail is not None:
                arrow["tail_node_id"] = nearest_node_id_by_boundary(tail, refined_nodes)
            if head is not None:
                arrow["head_node_id"] = nearest_node_id_by_boundary(head, refined_nodes)

    writer.setdefault("metadata", {})
    writer["metadata"]["node_refinement"] = {
        "preserve_graph_topology": not NODE_REFINEMENT_REPAIR_EDGE_IDS,
        "matched_nodes": int(len(mapping)),
        "graph_nodes": int(len(graph_nodes)),
        "resnet_nodes": int(len(resnet_nodes)),
    }
    return writer


def samples_to_dicts(samples):
    return [{
        "sample_id": s.sample_id,
        "image_path": str(s.image_path),
        "json_path": str(s.json_path) if s.json_path else "",
        "family": s.family,
        "split": s.split,
    } for s in samples]


def split_evenly(items, n):
    return [items[i::n] for i in range(n)]


def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

## 5. Pass-II structural-finalization rules

All finalization thresholds are exposed below. Pass II acts on the graph already
selected by Pass I. It can complete clear attachments, reject weak fallback
edges, suppress physical duplicates, restore a small number of strongly
supported candidates, and refine node geometry after topology is fixed.


In [ ]:
FINALIZE_ATTACH_RELAX_MODEL_PX = 15.0


FINALIZE_ATTACH_MIN_MARGIN_MODEL_PX = 10.0


FINALIZE_ATTACH_BOTH_NULL_MIN_MARGIN = 24.0


FINALIZE_ATTACH_CONFIDENCE_MULTIPLIER = 1.22


FINALIZE_MAX_REPAIRED_ATTACHMENTS_PER_IMAGE = 8


FINALIZE_ENABLE_CANDIDATE_RESURRECTION = True


FINALIZE_MAX_RESURRECTED_PER_IMAGE = 2


FINALIZE_RESURRECT_MIN_CONFIDENCE = 0.56


FINALIZE_RESURRECT_MIN_SUPPORT = 0.72


FINALIZE_RESURRECT_MIN_SHAFT = 0.70


FINALIZE_RESURRECT_MAX_PATH_OVERLAP = 0.58


FINALIZE_DUPLICATE_STRONG_IOU = 0.82


FINALIZE_DUPLICATE_FALLBACK_IOU = 0.66


FINALIZE_DUPLICATE_ENDPOINT_DIST = 24.0


FINALIZE_PATH_MASK_THICKNESS = 5


FINALIZE_STRICT_FALLBACK_FAMILIES = {"fca"}


FINALIZE_INK_MIN_CONFIDENCE = 0.78


FINALIZE_INK_MIN_SUPPORT = 0.76


FINALIZE_INK_MIN_SHAFT = 0.72


FINALIZE_INK_MIN_DIRECTION = 0.62


FINALIZE_STROKE_MIN_CONFIDENCE = 0.60


FINALIZE_STROKE_MIN_SUPPORT = 0.70


FINALIZE_STROKE_MIN_SHAFT = 0.68


FINALIZE_STROKE_MIN_DIRECTION = 0.54


FINALIZE_NODE_MIN_MATCH_RATIO = 0.72


FINALIZE_NODE_MAX_COUNT_DELTA_RATIO = 0.24


FINALIZE_NODE_MIN_MATCHED = 2


FINALIZE_REATTACH_NULL_ON_REFINED_NODES = True


@dataclass
class AspectGeometry:
    """Aspect-correct geometry while the network still sees stretched 512x512 input.

    Coordinates passed to this object are in model space. Distances are measured in
    original-image geometry, then converted back to area-equivalent model pixels so
    the existing decoder thresholds retain roughly the same numerical meaning.
    """
    orig_w: int
    orig_h: int
    model_w: int = 512
    model_h: int = 512

    def __post_init__(self):
        self.sx = float(self.orig_w) / max(float(self.model_w), 1.0)
        self.sy = float(self.orig_h) / max(float(self.model_h), 1.0)
        self.ref = math.sqrt(max(self.sx * self.sy, 1e-12))
        self.aspect_ratio = max(float(self.orig_w) / max(self.orig_h, 1), float(self.orig_h) / max(self.orig_w, 1))

    def vector_to_original(self, v):
        return float(v[0]) * self.sx, float(v[1]) * self.sy

    def distance(self, p, q):
        dx = (float(p[0]) - float(q[0])) * self.sx
        dy = (float(p[1]) - float(q[1])) * self.sy
        return math.hypot(dx, dy) / self.ref

    def point_to_bbox_distance(self, point, bbox_xyxy):
        px, py = float(point[0]), float(point[1])
        x0, y0, x1, y1 = [float(v) for v in bbox_xyxy]
        dx_model = max(x0 - px, 0.0, px - x1)
        dy_model = max(y0 - py, 0.0, py - y1)
        return math.hypot(dx_model * self.sx, dy_model * self.sy) / self.ref

    def bbox_diag(self, bbox_xyxy):
        x0, y0, x1, y1 = [float(v) for v in bbox_xyxy]
        return math.hypot((x1 - x0) * self.sx, (y1 - y0) * self.sy) / self.ref

    def corrected_probe(self, ep, probe_dist):
        tx, ty = float(ep.tangent_xy[0]), float(ep.tangent_xy[1])
        ox, oy = tx * self.sx, ty * self.sy
        norm = math.hypot(ox, oy)
        if norm < 1e-8:
            ux, uy = 1.0, 0.0
        else:
            ux, uy = ox / norm, oy / norm
        physical_len = float(probe_dist) * self.ref
        dx_model = ux * physical_len / max(self.sx, 1e-8)
        dy_model = uy * physical_len / max(self.sy, 1e-8)
        px, py = ep.refined_xy
        if ep.kind == "tail":
            return float(px) - dx_model, float(py) - dy_model
        return float(px) + dx_model, float(py) + dy_model


def _arrow_source(arrow):
    meta = getattr(arrow, "meta", {}) or {}
    if meta.get("source"):
        return str(meta["source"])
    if meta.get("port_role_repair"):
        return "port_role_repair"
    return "endpoint_ast"


def _attached_count(arrow):
    return int(arrow.tail_node_id is not None) + int(arrow.head_node_id is not None)


def _arrow_quality(arrow):
    priority = {
        "endpoint_ast": 4.0,
        "port_role_repair": 3.4,
        "stroke_graph": 2.0,
        "ink_bridge": 1.0,
    }.get(_arrow_source(arrow), 1.5)
    return (
        _attached_count(arrow),
        priority,
        float(arrow.confidence),
        float(arrow.support_score),
        float(arrow.shaft_score),
        float(arrow.direction_score),
    )


def _node_by_id(nodes):
    return {n.node_id: n for n in nodes}


def aspect_attachment_rankings(ep, nodes, cfg, geom, top_k=3):
    probe = geom.corrected_probe(ep, cfg.tangent_probe_dist)
    ranked = []
    for n in nodes:
        d_point = geom.point_to_bbox_distance(ep.refined_xy, n.bbox_xyxy)
        d_probe = geom.point_to_bbox_distance(probe, n.bbox_xyxy)
        inside_bonus = -4.0 if point_inside_bbox(ep.refined_xy, n.bbox_xyxy) else 0.0
        score = 0.65 * d_point + 0.35 * d_probe + inside_bonus
        ranked.append({
            "node_id": n.node_id,
            "score": float(score),
            "endpoint_distance": float(d_point),
            "probe_distance": float(d_probe),
            "inside_bonus": float(inside_bonus),
            "bbox_xyxy": [int(v) for v in n.bbox_xyxy],
        })
    ranked.sort(key=lambda r: r["score"])
    margin = float(ranked[1]["score"] - ranked[0]["score"]) if len(ranked) > 1 else float("inf")
    return {
        "endpoint_id": ep.endpoint_id,
        "kind": ep.kind,
        "refined_xy": [float(ep.refined_xy[0]), float(ep.refined_xy[1])],
        "probe_xy": [float(probe[0]), float(probe[1])],
        "top_candidates": ranked[:max(1, int(top_k))],
        "margin_top1_top2": margin,
        "aspect_ratio": float(geom.aspect_ratio),
    }


def _allow_repaired_attachment(arrow, kind, node_id, nodes_by_id, geom):
    other_id = arrow.head_node_id if kind == "tail" else arrow.tail_node_id
    if other_id is None or node_id != other_id:
        return True
    # Avoid manufacturing a self-loop from a near-node attachment unless the path
    # clearly leaves the node region or the assembler already considered it a loop.
    if bool(arrow.self_loop):
        return True
    node = nodes_by_id.get(node_id)
    if node is None or len(arrow.path_xy) < 32:
        return False
    max_excursion = max(geom.point_to_bbox_distance(p, node.bbox_xyxy) for p in arrow.path_xy)
    return max_excursion >= max(8.0, 0.15 * geom.bbox_diag(node.bbox_xyxy))


def _repair_single_arrow_attachments(arrows, heads, tails, nodes, cfg, geom):
    heads_by_id = {e.endpoint_id: e for e in heads}
    tails_by_id = {e.endpoint_id: e for e in tails}
    nodes_by_id = _node_by_id(nodes)
    actions = []
    repaired = 0

    for arrow in sorted(arrows, key=lambda a: float(a.confidence), reverse=True):
        if repaired >= int(FINALIZE_MAX_REPAIRED_ATTACHMENTS_PER_IMAGE):
            break
        null_kinds = []
        if arrow.tail_node_id is None:
            null_kinds.append("tail")
        if arrow.head_node_id is None:
            null_kinds.append("head")
        if not null_kinds:
            continue

        pending = []
        for kind in null_kinds:
            ep = tails_by_id.get(arrow.tail_endpoint_id) if kind == "tail" else heads_by_id.get(arrow.head_endpoint_id)
            if ep is None:
                continue
            ranking = aspect_attachment_rankings(ep, nodes, cfg, geom, top_k=3)
            if not ranking["top_candidates"]:
                continue
            top = ranking["top_candidates"][0]
            max_score = float(cfg.node_attach_max_dist) + float(FINALIZE_ATTACH_RELAX_MODEL_PX)
            min_margin = float(FINALIZE_ATTACH_MIN_MARGIN_MODEL_PX)
            if len(null_kinds) == 2:
                min_margin = float(FINALIZE_ATTACH_BOTH_NULL_MIN_MARGIN)
            accepted = bool(top["score"] <= max_score and ranking["margin_top1_top2"] >= min_margin)
            if accepted and _allow_repaired_attachment(arrow, kind, top["node_id"], nodes_by_id, geom):
                pending.append((kind, top, ranking, ep))

        # If both endpoints were null, require both to be strongly resolved.
        if len(null_kinds) == 2 and len(pending) != 2:
            continue
        if len(null_kinds) == 1 and len(pending) != 1:
            continue

        before = {"tail_node_id": arrow.tail_node_id, "head_node_id": arrow.head_node_id, "confidence": float(arrow.confidence)}
        for kind, top, ranking, ep in pending:
            if kind == "tail":
                arrow.tail_node_id = top["node_id"]
                arrow.meta["tail_attach_score_post"] = float(top["score"])
            else:
                arrow.head_node_id = top["node_id"]
                arrow.meta["head_attach_score_post"] = float(top["score"])
            arrow.meta[f"{kind}_attachment_post_margin"] = float(ranking["margin_top1_top2"])
            arrow.meta[f"{kind}_attachment_aspect_corrected"] = True
            repaired += 1

        arrow.self_loop = bool(arrow.tail_node_id is not None and arrow.tail_node_id == arrow.head_node_id)
        if arrow.tail_node_id is not None and arrow.head_node_id is not None:
            arrow.confidence = float(min(1.0, arrow.confidence * FINALIZE_ATTACH_CONFIDENCE_MULTIPLIER))
        actions.append({
            "action": "complete_endpoint_attachment",
            "arrow_id": int(arrow.arrow_id),
            "source": _arrow_source(arrow),
            "before": before,
            "after": {"tail_node_id": arrow.tail_node_id, "head_node_id": arrow.head_node_id, "confidence": float(arrow.confidence)},
            "resolved": [
                {"kind": kind, "node_id": top["node_id"], "score": float(top["score"]), "margin": float(ranking["margin_top1_top2"])}
                for kind, top, ranking, _ in pending
            ],
        })
    return arrows, actions


def _arrow_mask(path_xy, shape_hw=(512, 512), thickness=5):
    h, w = int(shape_hw[0]), int(shape_hw[1])
    mask = np.zeros((h, w), np.uint8)
    if len(path_xy) >= 2:
        pts = np.asarray([[int(round(x)), int(round(y))] for x, y in path_xy], dtype=np.int32)
        cv2.polylines(mask, [pts], False, 1, int(thickness), cv2.LINE_8)
    return mask


def _mask_iou_arrays(ma, mb):
    inter = int(np.logical_and(ma > 0, mb > 0).sum())
    union = int(np.logical_or(ma > 0, mb > 0).sum())
    return float(inter / max(union, 1))


def _path_bounds(path_xy, pad=0):
    if not path_xy:
        return (0, 0, -1, -1)
    xs = [float(p[0]) for p in path_xy]
    ys = [float(p[1]) for p in path_xy]
    return (min(xs) - pad, min(ys) - pad, max(xs) + pad, max(ys) + pad)


def _bounds_overlap(a, b):
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def _arrow_path_iou_model(a, b, shape_hw=(512, 512), thickness=5, mask_a=None, mask_b=None):
    pad = max(2, int(thickness))
    if not _bounds_overlap(_path_bounds(a.path_xy, pad), _path_bounds(b.path_xy, pad)):
        return 0.0
    ma = mask_a if mask_a is not None else _arrow_mask(a.path_xy, shape_hw, thickness)
    mb = mask_b if mask_b is not None else _arrow_mask(b.path_xy, shape_hw, thickness)
    return _mask_iou_arrays(ma, mb)


def _arrow_endpoint_distance(a, b, geom):
    direct = geom.distance(a.tail_xy, b.tail_xy) + geom.distance(a.head_xy, b.head_xy)
    reverse = geom.distance(a.tail_xy, b.head_xy) + geom.distance(a.head_xy, b.tail_xy)
    return min(direct, reverse) * 0.5


def _is_physical_duplicate(a, b, geom, shape_hw=(512, 512), mask_a=None, mask_b=None):
    iou = _arrow_path_iou_model(a, b, shape_hw, FINALIZE_PATH_MASK_THICKNESS, mask_a=mask_a, mask_b=mask_b)
    same_pair = a.tail_node_id == b.tail_node_id and a.head_node_id == b.head_node_id
    if same_pair and iou >= 0.45:
        return True, iou, "same_node_pair_path_overlap"
    if iou >= float(FINALIZE_DUPLICATE_STRONG_IOU):
        return True, iou, "strong_physical_path_overlap"
    fallback_involved = _arrow_source(a) in {"stroke_graph", "ink_bridge"} or _arrow_source(b) in {"stroke_graph", "ink_bridge"}
    null_involved = _attached_count(a) < 2 or _attached_count(b) < 2
    if (fallback_involved or null_involved) and iou >= float(FINALIZE_DUPLICATE_FALLBACK_IOU):
        if _arrow_endpoint_distance(a, b, geom) <= float(FINALIZE_DUPLICATE_ENDPOINT_DIST):
            return True, iou, "fallback_or_null_duplicate"
    return False, iou, ""


def _fallback_graph_need(arrow, base_arrows):
    if arrow.tail_node_id is None or arrow.head_node_id is None:
        return False
    outdeg, indeg = Counter(), Counter()
    for a in base_arrows:
        if a.tail_node_id is not None:
            outdeg[a.tail_node_id] += 1
        if a.head_node_id is not None:
            indeg[a.head_node_id] += 1
    return outdeg[arrow.tail_node_id] == 0 or indeg[arrow.head_node_id] == 0


def _gate_fallback_arrows(arrows, family):
    if str(family).lower() not in FINALIZE_STRICT_FALLBACK_FAMILIES:
        return arrows, []
    base = [a for a in arrows if _arrow_source(a) not in {"stroke_graph", "ink_bridge"} and _attached_count(a) == 2]
    kept, actions = [], []
    for a in arrows:
        src = _arrow_source(a)
        keep = True
        reasons = []
        if src == "ink_bridge":
            keep = bool(
                _attached_count(a) == 2
                and float(a.confidence) >= FINALIZE_INK_MIN_CONFIDENCE
                and float(a.support_score) >= FINALIZE_INK_MIN_SUPPORT
                and float(a.shaft_score) >= FINALIZE_INK_MIN_SHAFT
                and float(a.direction_score) >= FINALIZE_INK_MIN_DIRECTION
                and _fallback_graph_need(a, base)
            )
            reasons = ["strict_fca_ink_bridge_gate"]
        elif src == "stroke_graph":
            keep = bool(
                _attached_count(a) == 2
                and float(a.confidence) >= FINALIZE_STROKE_MIN_CONFIDENCE
                and float(a.support_score) >= FINALIZE_STROKE_MIN_SUPPORT
                and float(a.shaft_score) >= FINALIZE_STROKE_MIN_SHAFT
                and float(a.direction_score) >= FINALIZE_STROKE_MIN_DIRECTION
                and _fallback_graph_need(a, base)
            )
            reasons = ["strict_fca_stroke_graph_gate"]
        if keep:
            kept.append(a)
        else:
            actions.append({
                "action": "remove_fallback_candidate",
                "arrow_id": int(a.arrow_id),
                "source": src,
                "reason": reasons[0],
                "tail_node_id": a.tail_node_id,
                "head_node_id": a.head_node_id,
                "confidence": float(a.confidence),
                "support_score": float(a.support_score),
                "shaft_score": float(a.shaft_score),
                "direction_score": float(a.direction_score),
            })
    return kept, actions


def _suppress_physical_duplicates(arrows, geom, shape_hw=(512, 512)):
    kept, actions = [], []
    mask_cache = {}
    def get_mask(a):
        key = id(a)
        if key not in mask_cache:
            mask_cache[key] = _arrow_mask(a.path_xy, shape_hw, FINALIZE_PATH_MASK_THICKNESS)
        return mask_cache[key]
    for cand in sorted(arrows, key=_arrow_quality, reverse=True):
        duplicate = None
        cand_mask = None
        for old in kept:
            if not _bounds_overlap(_path_bounds(cand.path_xy, FINALIZE_PATH_MASK_THICKNESS), _path_bounds(old.path_xy, FINALIZE_PATH_MASK_THICKNESS)):
                continue
            if cand_mask is None:
                cand_mask = get_mask(cand)
            is_dup, iou, reason = _is_physical_duplicate(cand, old, geom, shape_hw, mask_a=cand_mask, mask_b=get_mask(old))
            if is_dup:
                duplicate = (old, iou, reason)
                break
        if duplicate is None:
            kept.append(cand)
        else:
            old, iou, reason = duplicate
            actions.append({
                "action": "suppress_physical_duplicate",
                "removed_arrow_id": int(cand.arrow_id),
                "kept_arrow_id": int(old.arrow_id),
                "removed_source": _arrow_source(cand),
                "kept_source": _arrow_source(old),
                "path_iou": float(iou),
                "reason": reason,
                "removed_pair": [cand.tail_node_id, cand.head_node_id],
                "kept_pair": [old.tail_node_id, old.head_node_id],
            })
    return kept, actions


def _candidate_already_selected(candidate, selected):
    for a in selected:
        if candidate.tail_endpoint_id == a.tail_endpoint_id and candidate.head_endpoint_id == a.head_endpoint_id:
            return True
    return False


def _resurrect_existing_candidates(selected, all_candidates, fixed_heads, fixed_tails, geom, family):
    if not FINALIZE_ENABLE_CANDIDATE_RESURRECTION:
        return selected, []
    heads = {e.endpoint_id: e for e in fixed_heads}
    tails = {e.endpoint_id: e for e in fixed_tails}
    outdeg, indeg = Counter(), Counter()
    pair_set = set()
    for a in selected:
        if a.tail_node_id is not None:
            outdeg[a.tail_node_id] += 1
        if a.head_node_id is not None:
            indeg[a.head_node_id] += 1
        pair_set.add((a.tail_node_id, a.head_node_id))

    proposals = []
    for raw in all_candidates:
        if _candidate_already_selected(raw, selected):
            continue
        src = _arrow_source(raw)
        if src == "ink_bridge":
            continue
        cand = deepcopy(raw)
        te = tails.get(cand.tail_endpoint_id)
        he = heads.get(cand.head_endpoint_id)
        if te is not None:
            cand.tail_node_id = te.attached_node_id
        if he is not None:
            cand.head_node_id = he.attached_node_id
        cand.self_loop = bool(cand.tail_node_id is not None and cand.tail_node_id == cand.head_node_id)
        if _attached_count(cand) != 2 or cand.self_loop:
            continue
        if (cand.tail_node_id, cand.head_node_id) in pair_set:
            continue
        graph_need = outdeg[cand.tail_node_id] == 0 or indeg[cand.head_node_id] == 0 or bool(cand.meta.get("port_role_repair"))
        if not graph_need:
            continue
        if float(cand.confidence) < FINALIZE_RESURRECT_MIN_CONFIDENCE:
            continue
        if float(cand.support_score) < FINALIZE_RESURRECT_MIN_SUPPORT or float(cand.shaft_score) < FINALIZE_RESURRECT_MIN_SHAFT:
            continue
        cand_bounds = _path_bounds(cand.path_xy, FINALIZE_PATH_MASK_THICKNESS)
        cand_mask = None
        max_overlap = 0.0
        for old in selected:
            if not _bounds_overlap(cand_bounds, _path_bounds(old.path_xy, FINALIZE_PATH_MASK_THICKNESS)):
                continue
            if cand_mask is None:
                cand_mask = _arrow_mask(cand.path_xy, (TARGET_H, TARGET_W), FINALIZE_PATH_MASK_THICKNESS)
            old_mask = _arrow_mask(old.path_xy, (TARGET_H, TARGET_W), FINALIZE_PATH_MASK_THICKNESS)
            max_overlap = max(max_overlap, _mask_iou_arrays(cand_mask, old_mask))
            if max_overlap > FINALIZE_RESURRECT_MAX_PATH_OVERLAP:
                break
        if max_overlap > FINALIZE_RESURRECT_MAX_PATH_OVERLAP:
            continue
        proposals.append((
            float(cand.confidence) + 0.20 * float(cand.support_score) + 0.15 * float(cand.shaft_score),
            max_overlap,
            cand,
        ))

    actions = []
    for _, max_overlap, cand in sorted(proposals, key=lambda x: x[0], reverse=True)[:int(FINALIZE_MAX_RESURRECTED_PER_IMAGE)]:
        cand.meta["Pass-II finalization_resurrected"] = True
        selected.append(cand)
        pair_set.add((cand.tail_node_id, cand.head_node_id))
        actions.append({
            "action": "resurrect_existing_candidate",
            "arrow_id": int(cand.arrow_id),
            "source": _arrow_source(cand),
            "tail_node_id": cand.tail_node_id,
            "head_node_id": cand.head_node_id,
            "confidence": float(cand.confidence),
            "support_score": float(cand.support_score),
            "shaft_score": float(cand.shaft_score),
            "max_path_overlap": float(max_overlap),
        })
    return selected, actions


def _propagate_fixed_endpoint_nodes(arrows, heads, tails):
    heads_by_id = {e.endpoint_id: e for e in heads}
    tails_by_id = {e.endpoint_id: e for e in tails}
    for a in arrows:
        te = tails_by_id.get(a.tail_endpoint_id)
        he = heads_by_id.get(a.head_endpoint_id)
        if te is not None and a.tail_node_id is None:
            a.tail_node_id = te.attached_node_id
        if he is not None and a.head_node_id is None:
            a.head_node_id = he.attached_node_id
        a.self_loop = bool(a.tail_node_id is not None and a.tail_node_id == a.head_node_id)
    return arrows


def _repair_endpoint_objects(endpoints, nodes, cfg, geom):
    fixed = deepcopy(endpoints)
    audits = []
    for ep in fixed:
        if ep.attached_node_id is not None:
            continue
        ranking = aspect_attachment_rankings(ep, nodes, cfg, geom, top_k=3)
        if not ranking["top_candidates"]:
            continue
        top = ranking["top_candidates"][0]
        accepted = bool(
            top["score"] <= float(cfg.node_attach_max_dist) + FINALIZE_ATTACH_RELAX_MODEL_PX
            and ranking["margin_top1_top2"] >= FINALIZE_ATTACH_MIN_MARGIN_MODEL_PX
        )
        audits.append({
            "endpoint_id": ep.endpoint_id,
            "kind": ep.kind,
            "accepted": accepted,
            "top_node_id": top["node_id"],
            "top_score": float(top["score"]),
            "margin": float(ranking["margin_top1_top2"]),
        })
        if accepted:
            ep.attached_node_id = top["node_id"]
            ep.attach_score = float(top["score"])
    return fixed, audits


def _bbox_distance_original(point_xy, bbox_xywh):
    px, py = float(point_xy[0]), float(point_xy[1])
    x, y, w, h = [float(v) for v in bbox_xywh]
    x1, y1 = x + w, y + h
    dx = max(x - px, 0.0, px - x1)
    dy = max(y - py, 0.0, py - y1)
    if dx > 0.0 or dy > 0.0:
        return math.hypot(dx, dy)
    return min(abs(px - x), abs(px - x1), abs(py - y), abs(py - y1))


def _rank_writer_nodes(point_xy, nodes):
    rows = []
    for n in nodes:
        rows.append({"node_id": n.get("id"), "distance": float(_bbox_distance_original(point_xy, n.get("bbox", [0, 0, 0, 0])))})
    rows.sort(key=lambda r: r["distance"])
    margin = rows[1]["distance"] - rows[0]["distance"] if len(rows) > 1 else float("inf")
    return rows, float(margin)

## 6. Canonical two-pass assembler

This cell joins Pass I, Pass II, and the agreement-gated node refinement into
the single final method used by the paper.


In [ ]:
def refine_nodes_after_topology(structural_writer, resnet_nodes):
    """Reliability-gated node refinement after topology is fixed.

    This agreement-gated refinement is executed only after graph topology is fixed. It cannot
    perturb A*, candidate generation, endpoint reuse, or branch selection.
    """
    graph_nodes = structural_writer.get("nodes", []) or []
    width, height = _writer_size(structural_writer)
    mapping = match_resnet_nodes_to_graph_nodes(graph_nodes, resnet_nodes, width, height)
    denom = max(len(graph_nodes), len(resnet_nodes), 1)
    match_ratio = float(len(mapping) / denom)
    count_delta_ratio = float(
        abs(len(graph_nodes) - len(resnet_nodes)) / max(len(graph_nodes), 1)
    )
    reliable = bool(
        len(mapping) >= min(int(FINALIZE_NODE_MIN_MATCHED), max(1, len(graph_nodes)))
        and match_ratio >= float(FINALIZE_NODE_MIN_MATCH_RATIO)
        and count_delta_ratio <= float(FINALIZE_NODE_MAX_COUNT_DELTA_RATIO)
    )

    if ENABLE_NODE_REFINEMENT and reliable:
        writer = build_refined_node_writer(structural_writer, resnet_nodes)
        decision = "agreement_gated_node_refinement_after_topology"
    else:
        writer = deepcopy(structural_writer)
        decision = (
            "node_finalization_disabled"
            if not ENABLE_NODE_REFINEMENT
            else "retain_structural_topology_due_to_low_node_agreement"
        )

    reattach_actions = []
    if ENABLE_NODE_REFINEMENT and FINALIZE_REATTACH_NULL_ON_REFINED_NODES:
        nodes = writer.get("nodes", []) or []
        diag = math.hypot(width, height)
        max_dist = max(4.0, 0.018 * diag)
        min_margin = max(3.0, 0.004 * diag)
        for edge in writer.get("arrows", []) or []:
            for kind, id_key, hint_key in [
                ("tail", "tail_node_id", "tail_hint"),
                ("head", "head_node_id", "head_hint"),
            ]:
                if edge.get(id_key) is not None:
                    continue
                p = point_from_edge(edge, kind, hint_key)
                if p is None or not nodes:
                    continue
                ranked, margin = _rank_writer_nodes(p, nodes)
                if ranked and ranked[0]["distance"] <= max_dist and margin >= min_margin:
                    edge[id_key] = ranked[0]["node_id"]
                    reattach_actions.append({
                        "action": "attach_null_endpoint_after_node_refinement",
                        "edge_id": edge.get("id"),
                        "kind": kind,
                        "node_id": ranked[0]["node_id"],
                        "distance": float(ranked[0]["distance"]),
                        "margin": float(margin),
                    })

    debug = {
        "decision": decision,
        "reliable": bool(reliable),
        "matched_nodes": int(len(mapping)),
        "graph_nodes": int(len(graph_nodes)),
        "resnet_nodes": int(len(resnet_nodes)),
        "match_ratio": float(match_ratio),
        "count_delta_ratio": float(count_delta_ratio),
        "mapping": {str(k): int(v) for k, v in mapping.items()},
        "reattach_actions": reattach_actions,
    }
    return writer, debug


def finalize_two_pass_graph(
    first_pass_result, image_name, orig_size, family, resnet_nodes_orig=None,
):
    """Pass 2: structural finalization using the already-selected graph.

    The stage order intentionally mirrors the validated successful behaviour:
      1. repair endpoint objects;
      2. propagate repairs to selected arrows;
      3. complete clear selected-arrow attachments;
      4. gate selected fallbacks using the actual graph;
      5. suppress duplicates among selected paths;
      6. resurrect only strong already-generated candidates when graph need remains;
      7. suppress duplicates again;
      8. refine node geometry after topology is frozen.
    """
    orig_w, orig_h = int(orig_size[0]), int(orig_size[1])
    cfg_used = LADDecoderConfig(**first_pass_result.decoder_debug_json["config"])
    geom = AspectGeometry(
        orig_w=orig_w, orig_h=orig_h,
        model_w=int(TARGET_W), model_h=int(TARGET_H),
    )

    fixed_heads, head_audits = _repair_endpoint_objects(
        first_pass_result.heads, first_pass_result.nodes, cfg_used, geom
    )
    fixed_tails, tail_audits = _repair_endpoint_objects(
        first_pass_result.tails, first_pass_result.nodes, cfg_used, geom
    )

    arrows = deepcopy(first_pass_result.arrows)
    arrows = _propagate_fixed_endpoint_nodes(arrows, fixed_heads, fixed_tails)
    arrows, attach_actions = _repair_single_arrow_attachments(
        arrows, fixed_heads, fixed_tails,
        first_pass_result.nodes, cfg_used, geom,
    )
    arrows, fallback_actions = _gate_fallback_arrows(arrows, family)
    arrows, duplicate_actions_1 = _suppress_physical_duplicates(
        arrows, geom, shape_hw=(int(TARGET_H), int(TARGET_W))
    )
    arrows, resurrect_actions = _resurrect_existing_candidates(
        arrows, first_pass_result.candidate_arrows,
        fixed_heads, fixed_tails, geom, family,
    )
    arrows, duplicate_actions_2 = _suppress_physical_duplicates(
        arrows, geom, shape_hw=(int(TARGET_H), int(TARGET_W))
    )

    arrows = sorted(
        arrows, key=lambda a: (_arrow_quality(a), -int(a.arrow_id)), reverse=True
    )
    for idx, arrow in enumerate(arrows, start=1):
        arrow.arrow_id = int(idx)

    writer_model = build_writer_json(
        image_name, int(TARGET_W), int(TARGET_H),
        first_pass_result.nodes, arrows,
    )
    writer_model["coordinate_space"] = "model"
    structural_writer = rescale_writer_json(writer_model, orig_w, orig_h)

    final_writer, node_debug = refine_nodes_after_topology(
        structural_writer, resnet_nodes_orig or []
    )

    aspect_debug = {
        "orig_w": orig_w,
        "orig_h": orig_h,
        "model_w": int(TARGET_W),
        "model_h": int(TARGET_H),
        "scale_x": float(geom.sx),
        "scale_y": float(geom.sy),
        "area_equivalent_scale": float(geom.ref),
        "aspect_ratio": float(geom.aspect_ratio),
        "extreme_aspect": bool(geom.aspect_ratio > 4.0),
    }
    counts = {
        "raw_candidates": int(len(first_pass_result.candidate_arrows)),
        "first_pass_selected_arrows": int(len(first_pass_result.arrows)),
        "endpoint_object_repairs": int(
            sum(int(a["accepted"]) for a in head_audits + tail_audits)
        ),
        "selected_attachment_repairs": int(len(attach_actions)),
        "selected_fallback_removals": int(len(fallback_actions)),
        "selected_duplicate_removals": int(
            len(duplicate_actions_1) + len(duplicate_actions_2)
        ),
        "candidate_resurrections": int(len(resurrect_actions)),
        "structurally_finalized_arrows": int(len(arrows)),
        "final_writer_arrows": int(len(final_writer.get("arrows", []) or [])),
        "matched_node_refinements": int(node_debug["matched_nodes"]),
        "node_reattach_actions": int(len(node_debug["reattach_actions"])),
    }
    debug = {
        "version": "two_pass_evidence_aware_assembler_paper_v1",
        "family": family,
        "pass_1": {
            "decoder": "unchanged_learning_aligned_assembler",
            "candidate_count": int(len(first_pass_result.candidate_arrows)),
            "selected_count": int(len(first_pass_result.arrows)),
        },
        "pass_2": {
            "stage_order": [
                "endpoint_object_repair",
                "selected_arrow_attachment_completion",
                "selected_fallback_gating",
                "selected_physical_duplicate_suppression",
                "limited_candidate_resurrection",
                "final_duplicate_suppression",
                "node_refinement_after_topology",
            ],
            "aspect_geometry": aspect_debug,
            "endpoint_object_audits": {
                "heads": head_audits,
                "tails": tail_audits,
            },
            "actions": {
                "attachment": attach_actions,
                "fallback_gate": fallback_actions,
                "duplicate_suppression_first": duplicate_actions_1,
                "candidate_resurrection": resurrect_actions,
                "duplicate_suppression_final": duplicate_actions_2,
            },
            "node_finalization": node_debug,
        },
        "candidate_counts": {
            "raw_total": int(len(first_pass_result.candidate_arrows)),
            "first_pass_selected": int(len(first_pass_result.arrows)),
            "selected": int(len(final_writer.get("arrows", []) or [])),
        },
        "counts": counts,
        "final_structural_arrows_model": [safe_json(asdict(a)) for a in arrows],
        "final_writer_arrows_original": safe_json(final_writer.get("arrows", []) or []),
        "config": safe_json(asdict(cfg_used)),
    }

    final_writer.setdefault("metadata", {})["assembler"] = {
        "name": "two_pass_evidence_aware_structural_assembler",
        "version": "paper-v1",
        "counts": counts,
        "aspect_geometry": aspect_debug,
        "node_finalization": {
            "decision": node_debug["decision"],
            "reliable": node_debug["reliable"],
            "matched_nodes": node_debug["matched_nodes"],
        },
    }

    return DecodeResult(
        nodes=first_pass_result.nodes,
        heads=fixed_heads,
        tails=fixed_tails,
        candidate_arrows=first_pass_result.candidate_arrows,
        arrows=arrows,
        writer_style_json=writer_model,
        writer_style_original_json=final_writer,
        decoder_debug_json=debug,
        debug_maps=first_pass_result.debug_maps,
        nx_graph=build_networkx_graph(first_pass_result.nodes, arrows),
    )


def decode_two_pass_graph(
    pred, decoder_img, image_name, cfg, orig_size, family,
    resnet_nodes_orig=None, first_pass_result=None,
):
    """Canonical two-pass decode.

    `first_pass_result` should be supplied by the worker so the unchanged assembler
    is run only once. The fallback call is retained for standalone use.
    """
    if first_pass_result is None:
        first_pass_result = decode_graph_learning_aligned(
            pred, decoder_img, image_name, cfg, orig_size=orig_size
        )
    return finalize_two_pass_graph(
        first_pass_result=first_pass_result,
        image_name=image_name,
        orig_size=orig_size,
        family=family,
        resnet_nodes_orig=resnet_nodes_orig,
    )

## 7. Inference and result export

The runner supports one GPU, CPU execution, or notebook-safe threaded sharding
across multiple GPUs. Every final graph can be saved, while detailed debug files
are limited to a small number of examples.


In [ ]:
# ============================================================
# CANONICAL INFERENCE, OUTPUT, AND METRIC RUNNER
# ============================================================

import hashlib


def require_filled_path(value, variable_name, must_exist=True):
    value = str(value).strip()
    if not value:
        raise ValueError(
            f"{variable_name} is blank. Fill it in the USER SETTINGS cell."
        )
    path = Path(value).expanduser().resolve()
    if must_exist and not path.exists():
        raise FileNotFoundError(f"{variable_name} does not exist: {path}")
    return path


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def validate_run_configuration():
    dataset_root = require_filled_path(DATASET_ROOT, "DATASET_ROOT")
    graph_checkpoint = require_filled_path(
        GRAPH_MODEL_CHECKPOINT, "GRAPH_MODEL_CHECKPOINT"
    )
    node_checkpoint = require_filled_path(
        NODE_MODEL_CHECKPOINT, "NODE_MODEL_CHECKPOINT"
    )
    output_root = require_filled_path(
        OUTPUT_ROOT, "OUTPUT_ROOT", must_exist=False
    )

    if output_root in {dataset_root, graph_checkpoint, node_checkpoint}:
        raise ValueError("OUTPUT_ROOT must be a separate output directory.")

    if _sk_skeletonize is None:
        raise ImportError(
            "scikit-image is required to calculate arrow-only Soft-clDice."
        )

    if tvm is None:
        raise ImportError(
            "torchvision is required to load the plain ResNet34 node model."
        )

    if int(GRAPH_BASE_CHANNELS) != 16:
        raise ValueError(
            "The paper graph-evidence checkpoint uses GRAPH_BASE_CHANNELS = 16."
        )

    if CLEAR_OUTPUT_BEFORE_RUN and output_root.exists():
        shutil.rmtree(output_root)

    output_root.mkdir(parents=True, exist_ok=True)
    cv2.setNumThreads(max(1, int(CPU_WORKERS_FOR_IO)))
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    return dataset_root, graph_checkpoint, node_checkpoint, output_root


def discover_paper_samples():
    samples = discover_converted_dataset_samples(
        dataset_root=Path(DATASET_ROOT),
        families=list(DATASET_FAMILIES),
        splits=list(INFERENCE_SPLITS),
        require_gt=True,
    )

    if QUICK_TEST:
        rng = random.Random(RANDOM_SEED)
        samples = rng.sample(samples, min(int(QUICK_TEST_N), len(samples)))

    if not samples:
        raise RuntimeError("No image/annotation pairs were discovered.")

    print(f"Discovered {len(samples)} inference samples.")
    return samples


def process_sample_list_worker(
    worker_id,
    sample_dicts,
    device_str,
    example_keys,
    output_root,
):
    output_root = Path(output_root)
    device = torch.device(
        device_str if device_str else (
            "cuda" if torch.cuda.is_available() else "cpu"
        )
    )

    decoder_args = make_decoder_args(device_str=device_str)

    print(f"[worker {worker_id}] loading graph-evidence model on {device}")
    graph_model, graph_load_info = load_assembler_checkpoint_safe(
        checkpoint_path=Path(GRAPH_MODEL_CHECKPOINT),
        device=device,
        image_mode=IMAGE_MODE,
        base_ch=int(GRAPH_BASE_CHANNELS),
        norm_type=NORM_TYPE,
        strict=True,
    )

    inferred_base_channels = int(
        graph_load_info.get("base_ch", GRAPH_BASE_CHANNELS)
    )
    if inferred_base_channels != int(GRAPH_BASE_CHANNELS):
        raise ValueError(
            "Graph checkpoint architecture mismatch: "
            f"checkpoint base channels={inferred_base_channels}, "
            f"configured={GRAPH_BASE_CHANNELS}."
        )

    decoder_config = build_decoder_cfg(decoder_args)

    print(f"[worker {worker_id}] loading plain ResNet34 node model on {device}")
    node_model, node_load_info = load_node_segmentation_model(
        NODE_MODEL_CHECKPOINT, device
    )

    metric_rows = []
    failures = []

    for index, sample_dict in enumerate(sample_dicts, start=1):
        sample = InferenceSample(
            sample_id=sample_dict["sample_id"],
            image_path=Path(sample_dict["image_path"]),
            json_path=Path(sample_dict["json_path"]),
            family=sample_dict.get("family", ""),
            split=sample_dict.get("split", ""),
        )

        try:
            start_time = time.time()

            if PRINT_TIMING_DIAGNOSTICS and (
                index <= 3 or index % int(PROGRESS_EVERY_N) == 0
            ):
                print(
                    f"[worker {worker_id}] {index}/{len(sample_dicts)} "
                    f"{sample.family}/{sample.split}/{sample.image_path.name}"
                )

            original_bgr, decoder_image, image_tensor = load_image_for_model(
                image_path=sample.image_path,
                target_w=int(TARGET_W),
                target_h=int(TARGET_H),
                image_mode=IMAGE_MODE,
                normalize_image=NORMALIZE_IMAGE,
                use_otsu=bool(APPLY_OTSU_THRESHOLD),
            )

            with torch.no_grad():
                graph_outputs = graph_model(image_tensor.to(device))
            evidence_maps = outputs_to_pred_maps(graph_outputs)

            _, node_probability = predict_node_probability(
                node_model,
                sample.image_path,
                device,
                img_size=int(NODE_SEG_INPUT_SIZE),
            )
            resnet_nodes, node_binary = extract_node_boxes(
                node_probability,
                threshold=float(NODE_SEG_THRESHOLD),
                min_area=int(NODE_SEG_MIN_AREA),
                close_kernel=int(NODE_SEG_CLOSE_KERNEL),
            )

            original_height, original_width = original_bgr.shape[:2]

            # Pass I: physical graph hypothesis construction.
            first_pass = decode_graph_learning_aligned(
                pred=evidence_maps,
                decoder_img=decoder_image,
                image_name=sample.image_path.name,
                cfg=decoder_config,
                orig_size=(original_width, original_height),
            )

            # Pass II: graph-context structural finalization followed by
            # agreement-gated node-region refinement.
            final_result = decode_two_pass_graph(
                pred=evidence_maps,
                decoder_img=decoder_image,
                image_name=sample.image_path.name,
                cfg=decoder_config,
                orig_size=(original_width, original_height),
                family=sample.family,
                resnet_nodes_orig=resnet_nodes,
                first_pass_result=first_pass,
            )
            prediction = (
                final_result.writer_style_original_json
                or final_result.writer_style_json
            )

            with open(sample.json_path, "r", encoding="utf-8") as handle:
                ground_truth = json.load(handle)

            row = evaluate_graph_prediction_tolerant(
                prediction, ground_truth
            )
            final_counts = (
                final_result.decoder_debug_json.get("counts", {}) or {}
            )
            row.update({
                "sample_id": sample.sample_id,
                "image_name": sample.image_path.name,
                "family": sample.family,
                "split": sample.split,
                "image_path": str(sample.image_path),
                "json_path": str(sample.json_path),
                "worker_id": worker_id,
                "first_pass_candidates": int(
                    len(first_pass.candidate_arrows)
                ),
                "first_pass_selected_edges": int(
                    len(first_pass.arrows)
                ),
                "final_selected_edges": int(
                    len(prediction.get("arrows", []) or [])
                ),
                "endpoint_repairs": int(
                    final_counts.get("endpoint_object_repairs", 0)
                    + final_counts.get("selected_attachment_repairs", 0)
                ),
                "fallback_removals": int(
                    final_counts.get("selected_fallback_removals", 0)
                ),
                "duplicate_removals": int(
                    final_counts.get("selected_duplicate_removals", 0)
                ),
                "candidate_resurrections": int(
                    final_counts.get("candidate_resurrections", 0)
                ),
                "matched_node_refinements": int(
                    final_counts.get("matched_node_refinements", 0)
                ),
                "elapsed_seconds": float(time.time() - start_time),
            })
            metric_rows.append(row)

            if SAVE_ALL_PREDICTION_JSON:
                prediction_path = (
                    output_root
                    / "predictions"
                    / sample.family
                    / sample.split
                    / "json"
                    / f"{sample.sample_id}.json"
                )
                save_json_local(prediction_path, prediction)

            sample_key = (
                sample.family,
                sample.split,
                sample.sample_id,
            )
            if SAVE_DEBUG_EXAMPLES and sample_key in example_keys:
                example_dir = (
                    output_root
                    / "examples"
                    / f"{sample.family}_{sample.split}_{sample.sample_id}"
                )
                example_dir.mkdir(parents=True, exist_ok=True)

                save_json_local(
                    example_dir / "prediction.json", prediction
                )
                save_json_local(
                    example_dir / "ground_truth.json", ground_truth
                )
                save_json_local(
                    example_dir / "metrics.json", row
                )

                if SAVE_DEBUG_ASSEMBLER_JSON:
                    save_json_local(
                        example_dir / "two_pass_assembler_debug.json",
                        final_result.decoder_debug_json,
                    )

                if SAVE_DEBUG_OVERLAYS:
                    draw_overlay_from_writer(
                        sample.image_path,
                        prediction,
                        example_dir / "prediction_overlay.png",
                    )
                    draw_overlay_from_writer(
                        sample.image_path,
                        ground_truth,
                        example_dir / "ground_truth_overlay.png",
                    )

            if PRINT_TIMING_DIAGNOSTICS and (
                index <= 3 or index % int(PROGRESS_EVERY_N) == 0
            ):
                print(
                    f"[worker {worker_id}] completed in "
                    f"{time.time() - start_time:.2f}s"
                )

        except Exception as exc:
            failures.append({
                "sample_id": sample_dict.get("sample_id", ""),
                "family": sample_dict.get("family", ""),
                "split": sample_dict.get("split", ""),
                "image_path": sample_dict.get("image_path", ""),
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(limit=60),
                "worker_id": worker_id,
            })

    shard_dir = output_root / "shards"
    shard_dir.mkdir(parents=True, exist_ok=True)

    with open(
        shard_dir / f"worker_{worker_id}_metrics.jsonl",
        "w",
        encoding="utf-8",
    ) as handle:
        for row in metric_rows:
            handle.write(
                json.dumps(safe_json(row), ensure_ascii=False) + "\n"
            )

    with open(
        shard_dir / f"worker_{worker_id}_failures.jsonl",
        "w",
        encoding="utf-8",
    ) as handle:
        for row in failures:
            handle.write(
                json.dumps(safe_json(row), ensure_ascii=False) + "\n"
            )

    save_json_local(
        shard_dir / f"worker_{worker_id}_load_info.json",
        {
            "device": str(device),
            "num_samples": len(sample_dicts),
            "num_successful": len(metric_rows),
            "num_failures": len(failures),
            "graph_model": graph_load_info,
            "node_model": node_load_info,
        },
    )

    return True


def run_parallel_or_single(samples):
    output_root = Path(OUTPUT_ROOT)
    output_root.mkdir(parents=True, exist_ok=True)

    example_keys = get_example_ids(
        samples,
        max_n=int(MAX_DEBUG_EXAMPLES),
        policy=DEBUG_EXAMPLE_POLICY,
    )
    sample_dicts = samples_to_dicts(samples)

    cuda_count = (
        torch.cuda.device_count() if torch.cuda.is_available() else 0
    )
    worker_count = (
        min(int(MAX_GPU_WORKERS), max(1, cuda_count))
        if PARALLEL_INFERENCE
        else 1
    )

    if (
        not PARALLEL_INFERENCE
        or str(PARALLEL_BACKEND).lower() == "single"
        or worker_count <= 1
    ):
        device_str = "cuda:0" if cuda_count else "cpu"
        process_sample_list_worker(
            0,
            sample_dicts,
            device_str,
            example_keys,
            output_root,
        )
        return

    if str(PARALLEL_BACKEND).lower() != "threads":
        raise ValueError(
            "PARALLEL_BACKEND must be 'threads' or 'single'. "
            "Forked CUDA multiprocessing is intentionally not used."
        )

    from concurrent.futures import ThreadPoolExecutor, as_completed

    shards = split_evenly(sample_dicts, worker_count)
    futures = []

    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        for worker_id, shard in enumerate(shards):
            if not shard:
                continue
            device_str = f"cuda:{worker_id % cuda_count}"
            futures.append(
                executor.submit(
                    process_sample_list_worker,
                    worker_id,
                    shard,
                    device_str,
                    example_keys,
                    output_root,
                )
            )

        for future in as_completed(futures):
            future.result()


def collect_worker_rows(pattern):
    rows = []
    shard_dir = Path(OUTPUT_ROOT) / "shards"
    for path in sorted(shard_dir.glob(pattern)):
        rows.extend(read_jsonl(path))
    return rows


def percent(value):
    return 100.0 * float(value)


def build_table4_row(overall):
    return {
        "Method": "Proposed framework",
        "Node F1": round(percent(overall.get("mean_node_f1", 0.0)), 2),
        "Mask IoU": round(percent(overall.get("mean_mask_iou", 0.0)), 2),
        "Soft-clDice": round(
            percent(overall.get("mean_soft_cldice", 0.0)), 2
        ),
        "End. L2": round(float(overall.get("mean_endpoint_l2", 0.0)), 2),
        "Link F1": round(percent(overall.get("mean_link_f1", 0.0)), 2),
        "nGED": round(float(overall.get("mean_nged", 0.0)), 3),
        "Loop Rec.": round(
            percent(overall.get("loop_back_recall", 0.0)), 2
        ),
        "Branch Rec.": round(
            percent(overall.get("decision_branch_recall", 0.0)), 2
        ),
    }


def build_table5_rows(by_family):
    rows = []
    for family_row in by_family:
        rows.append({
            "Family": str(family_row.get("family", "")).upper(),
            "Images": int(family_row.get("num_images", 0)),
            "Link F1": round(
                percent(family_row.get("mean_link_f1", 0.0)), 2
            ),
            "nGED": round(float(family_row.get("mean_nged", 0.0)), 3),
            "End. L2": round(
                float(family_row.get("mean_endpoint_l2", 0.0)), 2
            ),
        })
    return rows


def finalize_paper_metrics():
    output_root = Path(OUTPUT_ROOT)
    metrics_dir = output_root / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    rows = collect_worker_rows("worker_*_metrics.jsonl")
    failures = collect_worker_rows("worker_*_failures.jsonl")

    if not rows:
        raise RuntimeError(
            "No successful metric rows were produced. Inspect failures.csv."
        )

    write_csv_rows_local(metrics_dir / "metrics_per_image.csv", rows)
    write_csv_rows_local(output_root / "failures.csv", failures)

    overall = aggregate_metric_rows_tolerant(
        rows, group_keys=[]
    )[0]
    by_family = aggregate_metric_rows_tolerant(
        rows, group_keys=["family"]
    )
    by_split = aggregate_metric_rows_tolerant(
        rows, group_keys=["split"]
    )
    by_family_split = aggregate_metric_rows_tolerant(
        rows, group_keys=["family", "split"]
    )

    table4 = [build_table4_row(overall)]
    table5 = build_table5_rows(by_family)

    write_csv_rows_local(metrics_dir / "metrics_by_family.csv", by_family)
    write_csv_rows_local(metrics_dir / "metrics_by_split.csv", by_split)
    write_csv_rows_local(
        metrics_dir / "metrics_by_family_split.csv",
        by_family_split,
    )
    write_csv_rows_local(metrics_dir / "table4_proposed_row.csv", table4)
    write_csv_rows_local(metrics_dir / "table5_family_results.csv", table5)

    summary = {
        "overall": overall,
        "by_family": by_family,
        "by_split": by_split,
        "by_family_split": by_family_split,
        "paper_table_4_proposed_row": table4[0],
        "paper_table_5_rows": table5,
        "successful_images": len(rows),
        "failed_images": len(failures),
        "metric_definitions": {
            "node_f1": (
                "One-to-one tolerant node matching using IoU or "
                "center-distance acceptance."
            ),
            "mask_iou": (
                "Strict binary IoU between predicted and ground-truth "
                "node-region rasters only."
            ),
            "soft_cldice": (
                "Tolerance-aware centerline Dice between predicted and "
                "ground-truth arrow-path rasters only."
            ),
            "endpoint_l2": (
                "Mean endpoint Euclidean error on matched directed edges."
            ),
            "link_f1": "Directed edge F1 after node matching.",
            "nged": (
                "Structural edit cost divided by "
                "max(|V_pred|,|V_gt|)+max(|E_pred|,|E_gt|)."
            ),
        },
    }
    save_json_local(metrics_dir / "metrics_summary.json", summary)

    checkpoint_manifest = {
        "graph_model": {
            "path": str(Path(GRAPH_MODEL_CHECKPOINT).resolve()),
            "sha256": sha256_file(GRAPH_MODEL_CHECKPOINT),
        },
        "node_model": {
            "path": str(Path(NODE_MODEL_CHECKPOINT).resolve()),
            "sha256": sha256_file(NODE_MODEL_CHECKPOINT),
        },
        "dataset_root": str(Path(DATASET_ROOT).resolve()),
        "families": list(DATASET_FAMILIES),
        "splits": list(INFERENCE_SPLITS),
        "successful_images": len(rows),
        "failed_images": len(failures),
    }
    save_json_local(
        output_root / "reproduction_manifest.json",
        checkpoint_manifest,
    )

    print("\n" + "=" * 72)
    print("TABLE 4 — PROPOSED FRAMEWORK ROW")
    print("=" * 72)
    print(json.dumps(table4[0], indent=2))

    print("\n" + "=" * 72)
    print("TABLE 5 — FAMILY-WISE RESULTS")
    print("=" * 72)
    for row in table5:
        print(row)

    print(f"\nSaved all outputs to: {output_root}")
    if failures:
        print(
            f"Warning: {len(failures)} samples failed. "
            "Inspect failures.csv before using the table values."
        )

    return summary

## 8. Run the complete held-out evaluation

For the paper run, leave `QUICK_TEST = False`. The printed values and generated
CSV files are the numbers to transfer into Tables 4 and 5.


In [ ]:
# ============================================================
# RUN THE COMPLETE PAPER PIPELINE
# ============================================================

dataset_root, graph_checkpoint, node_checkpoint, output_root = (
    validate_run_configuration()
)

samples = discover_paper_samples()

start_time = time.time()
run_parallel_or_single(samples)
PAPER_RESULTS = finalize_paper_metrics()

print(
    f"\nCompleted {len(samples)} samples in "
    f"{(time.time() - start_time) / 60.0:.2f} minutes."
)

## Output layout

```text
<output_root>/
├── predictions/
│   ├── fa/test/json/
│   ├── fca/test/json/
│   └── fcb/test/json/
├── examples/
├── metrics/
│   ├── metrics_per_image.csv
│   ├── metrics_summary.json
│   ├── metrics_by_family.csv
│   ├── table4_proposed_row.csv
│   └── table5_family_results.csv
├── shards/
├── failures.csv
└── reproduction_manifest.json
```

Do not use the paper table values when `failures.csv` contains unresolved
samples.
